# BLE Indoor Localization — Reproducibility Notebook

This notebook contains the experiment workflow used for the manuscript: observation-level evaluation, held-out-location evaluation, chronological evaluation, and controlled RSSI perturbation robustness. It is provided as research code; the original dataset is not included. See the repository README for setup and data instructions.

**Reproducibility note:** The notebook is designed for Google Colab and expects the dataset and Stage 4 result archive to be supplied by the user. It does not embed private data, trained weights, or credentials.

---

Phase-2 for Paper

In [ ]:
# ============================================================
# 1. REPRODUCIBILITY AND IMPORTS
# ============================================================
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Seed:", SEED)

In [ ]:
# ============================================================
# 2. LOAD DATA
# ============================================================
LABELED_PATH = "/content/iBeacon_RSSI_Labeled.csv"
UNLABELED_PATH = "/content/iBeacon_RSSI_Unlabeled.csv"

df = pd.read_csv(LABELED_PATH)
df_unlabeled = pd.read_csv(UNLABELED_PATH)

print("Labeled shape:", df.shape)
print("Unlabeled shape:", df_unlabeled.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
# ============================================================
# 3. DATA QUALITY AND RSSI SPARSITY
# ============================================================
beacon_cols = [f"b30{i:02d}" for i in range(1, 14)]

print("Unique locations:", df["location"].nunique())
print("Duplicate rows:", df.duplicated().sum())
print("Missing values:")
display(df.isnull().sum().to_frame("Missing"))

missing_beacon_counts = (df[beacon_cols] == -200).sum()
missing_summary = pd.DataFrame({
    "Count_-200": missing_beacon_counts,
    "Percentage": 100 * missing_beacon_counts / len(df)
})
print("RSSI non-detection summary:")
display(missing_summary.round(2))

df["num_detected_beacons"] = (df[beacon_cols] != -200).sum(axis=1)
print("Detected beacons per observation:")
display(df["num_detected_beacons"].describe().to_frame().T.round(3))

plt.figure(figsize=(7,5))
df["num_detected_beacons"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Number of detected beacons")
plt.ylabel("Number of observations")
plt.title("Beacon Visibility per RSSI Observation")
plt.tight_layout()
plt.savefig("/content/beacon_visibility.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 4. MAP LOCATION LABELS TO 2-D GRID COORDINATES
# ============================================================
def location_to_grid(label):
    label = str(label).strip().upper()
    col_letter = label[0]
    row_number = int(label[1:])
    x = ord(col_letter) - ord("A")
    y = row_number - 1
    return x, y

df[["x", "y"]] = df["location"].apply(lambda loc: pd.Series(location_to_grid(loc)))

coords = df[["location", "x", "y"]].drop_duplicates().sort_values(["y", "x"])
print("Unique physical locations:", len(coords))
display(coords.head(20))

plt.figure(figsize=(12,8))
plt.scatter(coords["x"], coords["y"])
for _, row in coords.iterrows():
    plt.text(row["x"] + 0.1, row["y"] + 0.1, row["location"], fontsize=8)
plt.xlabel("Grid X")
plt.ylabel("Grid Y")
plt.title("Ground-Truth Localization Reference Points")
plt.grid(True)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("/content/ground_truth_locations.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 5. LOCATION-AWARE OBSERVATION SPLIT
# Each physical location contributes observations to train/val/test.
# This is a SEEN-LOCATION protocol, not unseen-location generalization.
# ============================================================
rng = np.random.default_rng(SEED)
train_indices, val_indices, test_indices = [], [], []

for location, group in df.groupby("location"):
    indices = group.index.to_numpy().copy()
    rng.shuffle(indices)
    n = len(indices)

    if n == 2:
        n_train, n_val, n_test = 1, 0, 1
    elif n == 3:
        n_train, n_val, n_test = 1, 1, 1
    elif n == 4:
        n_train, n_val, n_test = 2, 1, 1
    else:
        n_val = max(1, round(n * 0.15))
        n_test = max(1, round(n * 0.15))
        n_train = n - n_val - n_test

    train_indices.extend(indices[:n_train])
    val_indices.extend(indices[n_train:n_train+n_val])
    test_indices.extend(indices[n_train+n_val:n_train+n_val+n_test])

rng.shuffle(train_indices); rng.shuffle(val_indices); rng.shuffle(test_indices)

X = df[beacon_cols].astype(np.float32)
y = df[["x", "y"]].astype(np.float32)

X_train = X.loc[train_indices].copy(); y_train = y.loc[train_indices].copy()
X_val = X.loc[val_indices].copy(); y_val = y.loc[val_indices].copy()
X_test = X.loc[test_indices].copy(); y_test = y.loc[test_indices].copy()

loc_train = df.loc[train_indices, "location"].copy()
loc_val = df.loc[val_indices, "location"].copy()
loc_test = df.loc[test_indices, "location"].copy()

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
print("Train locations:", loc_train.nunique())
print("Validation locations:", loc_val.nunique())
print("Test locations:", loc_test.nunique())

In [ ]:
# ============================================================
# 6. SPLIT INTEGRITY CHECK
# ============================================================
assert len(X_train) == 970
assert len(X_val) == 224
assert len(X_test) == 226
assert not (set(train_indices) & set(val_indices))
assert not (set(train_indices) & set(test_indices))
assert not (set(val_indices) & set(test_indices))
assert len(train_indices) + len(val_indices) + len(test_indices) == len(df)

split_df = pd.DataFrame({
    "original_index": train_indices + val_indices + test_indices,
    "split": (["train"] * len(train_indices) +
               ["validation"] * len(val_indices) +
               ["test"] * len(test_indices))
})
split_df["location"] = split_df["original_index"].map(df["location"])
split_df["x"] = split_df["original_index"].map(df["x"])
split_df["y"] = split_df["original_index"].map(df["y"])
split_df.to_csv("/content/location_aware_split_seed42.csv", index=False)

print("✓ Split integrity verified")
display(split_df["split"].value_counts().to_frame("Count"))

In [ ]:
# ============================================================
# 7. TRAIN-ONLY RSSI SCALING
# ============================================================
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Train range:", X_train_scaled.min(), X_train_scaled.max())
print("Validation range:", X_val_scaled.min(), X_val_scaled.max())
print("Test range:", X_test_scaled.min(), X_test_scaled.max())

In [ ]:
# ============================================================
# 8. COMMON EVALUATION METRICS
# ============================================================
def localization_metrics(y_true, y_pred, locations=None):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    if y_true.shape != y_pred.shape:
        raise ValueError(f"Shape mismatch: {y_true.shape} vs {y_pred.shape}")

    dx = y_pred[:, 0] - y_true[:, 0]
    dy = y_pred[:, 1] - y_true[:, 1]
    errors = np.sqrt(dx**2 + dy**2)

    metrics = {
        "Mean Localization Error": np.mean(errors),
        "Median Localization Error": np.median(errors),
        "Std Localization Error": np.std(errors),
        "75th Percentile": np.percentile(errors, 75),
        "90th Percentile": np.percentile(errors, 90),
        "Maximum Error": np.max(errors),
        "Coordinate MAE": np.mean(np.abs(y_pred - y_true)),
        "Coordinate RMSE": np.sqrt(np.mean((y_pred - y_true)**2)),
    }
    if locations is not None:
        temp = pd.DataFrame({"location": np.asarray(locations), "error": errors})
        metrics["Macro Location MLE"] = temp.groupby("location")["error"].mean().mean()
    return metrics, errors

def mean_localization_error(y_true, y_pred):
    return localization_metrics(y_true, y_pred)[0]["Mean Localization Error"]

In [ ]:
# ============================================================
# 9. CONVENTIONAL BASELINES + SVR
# ============================================================
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR

# Existing selected configurations from the completed validation tuning
knn_final = KNeighborsRegressor(n_neighbors=15, weights="distance")
dt_final = DecisionTreeRegressor(max_depth=15, min_samples_leaf=1, random_state=SEED)
rf_final = RandomForestRegressor(n_estimators=500, max_depth=20, min_samples_leaf=1, random_state=SEED, n_jobs=-1)

models = {"kNN": knn_final, "Decision Tree": dt_final, "Random Forest": rf_final}

preds = {}
error_sets = {}
metric_rows = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train.to_numpy())
    pred = model.predict(X_test_scaled)
    metrics, errors = localization_metrics(y_test, pred, loc_test)
    preds[name] = pred; error_sets[name] = errors
    metric_rows.append({"Model": name, **metrics})

# SVR: compact nonlinear regression baseline; tune on validation set only.
svr_grid = [
    {"C": 1.0, "gamma": "scale", "epsilon": 0.1},
    {"C": 10.0, "gamma": "scale", "epsilon": 0.1},
    {"C": 100.0, "gamma": "scale", "epsilon": 0.1},
    {"C": 10.0, "gamma": 0.01, "epsilon": 0.1},
    {"C": 10.0, "gamma": 0.1, "epsilon": 0.1},
    {"C": 10.0, "gamma": "scale", "epsilon": 0.01},
]
svr_tuning_rows = []
for p in svr_grid:
    m = MultiOutputRegressor(SVR(C=p["C"], gamma=p["gamma"], epsilon=p["epsilon"]))
    m.fit(X_train_scaled, y_train.to_numpy())
    pv = m.predict(X_val_scaled)
    svr_tuning_rows.append({**p, "Validation_MLE": mean_localization_error(y_val, pv)})
svr_tuning = pd.DataFrame(svr_tuning_rows).sort_values("Validation_MLE").reset_index(drop=True)
display(svr_tuning)

best_svr_params = svr_tuning.iloc[0].to_dict()
svr_final = MultiOutputRegressor(SVR(C=float(best_svr_params["C"]), gamma=best_svr_params["gamma"], epsilon=float(best_svr_params["epsilon"])))
svr_final.fit(X_train_scaled, y_train.to_numpy())
pred_svr = svr_final.predict(X_test_scaled)
metrics_svr, errors_svr = localization_metrics(y_test, pred_svr, loc_test)
preds["SVR"] = pred_svr; error_sets["SVR"] = errors_svr
metric_rows.append({"Model": "SVR", **metrics_svr})

conventional_results = pd.DataFrame(metric_rows).sort_values("Mean Localization Error")
display(conventional_results.round(4))
conventional_results.to_csv("/content/final_conventional_with_svr.csv", index=False)
svr_tuning.to_csv("/content/svr_validation_tuning.csv", index=False)

In [ ]:
# ============================================================
# 10. NEURAL MODELS: MLP, CNN-GAP, CNN-FLAT
# ============================================================
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, Conv1D, BatchNormalization, GlobalAveragePooling1D, Flatten
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

def reset_seeds():
    random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

def train_neural_model(model, train_X, val_X, model_name):
    reset_seeds()
    model.compile(optimizer=Adam(learning_rate=0.001), loss="mse", metrics=["mae"])
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10, min_lr=1e-6)
    ]
    history = model.fit(train_X, y_train.to_numpy(), validation_data=(val_X, y_val.to_numpy()),
                        epochs=300, batch_size=32, callbacks=callbacks, verbose=0)
    print(f"{model_name}: {len(history.history['loss'])} epochs; best val loss={min(history.history['val_loss']):.4f}")
    return history

X_train_cnn = X_train_scaled[..., np.newaxis]
X_val_cnn = X_val_scaled[..., np.newaxis]
X_test_cnn = X_test_scaled[..., np.newaxis]

In [ ]:
# MLP
reset_seeds()
mlp = Sequential([Input(shape=(13,)), Dense(128, activation="relu"), BatchNormalization(), Dropout(0.30),
                 Dense(64, activation="relu"), Dropout(0.20), Dense(32, activation="relu"), Dense(2, activation="linear")])
history_mlp = train_neural_model(mlp, X_train_scaled, X_val_scaled, "MLP")
pred_mlp = mlp.predict(X_test_scaled, verbose=0)
metrics_mlp, errors_mlp = localization_metrics(y_test, pred_mlp, loc_test)
preds["MLP"] = pred_mlp; error_sets["MLP"] = errors_mlp
print(pd.Series(metrics_mlp).round(4))

In [ ]:
# CNN-GAP
reset_seeds()
cnn_gap = Sequential([Input(shape=(13,1)), Conv1D(32,3,padding="same",activation="relu"), BatchNormalization(),
                      Conv1D(64,3,padding="same",activation="relu"), BatchNormalization(),
                      GlobalAveragePooling1D(), Dense(64,activation="relu"), Dropout(0.30), Dense(2,activation="linear")])
history_gap = train_neural_model(cnn_gap, X_train_cnn, X_val_cnn, "CNN-GAP")
pred_gap = cnn_gap.predict(X_test_cnn, verbose=0)
metrics_gap, errors_gap = localization_metrics(y_test, pred_gap, loc_test)
preds["1D-CNN-GAP"] = pred_gap; error_sets["1D-CNN-GAP"] = errors_gap
print(pd.Series(metrics_gap).round(4))

In [ ]:
# CNN-FLAT
reset_seeds()
cnn_flat = Sequential([Input(shape=(13,1)), Conv1D(32,3,padding="same",activation="relu"), BatchNormalization(),
                       Conv1D(64,3,padding="same",activation="relu"), BatchNormalization(), Flatten(),
                       Dense(128,activation="relu"), Dropout(0.30), Dense(64,activation="relu"), Dropout(0.20), Dense(2,activation="linear")])
history_flat = train_neural_model(cnn_flat, X_train_cnn, X_val_cnn, "CNN-FLAT")
pred_flat = cnn_flat.predict(X_test_cnn, verbose=0)
metrics_flat, errors_flat = localization_metrics(y_test, pred_flat, loc_test)
preds["1D-CNN-FLAT"] = pred_flat; error_sets["1D-CNN-FLAT"] = errors_flat
print(pd.Series(metrics_flat).round(4))

In [ ]:
# ============================================================
# 11. MASTER RESULTS TABLE
# ============================================================
neural_results = pd.DataFrame([
    {"Model":"MLP", **metrics_mlp},
    {"Model":"1D-CNN-GAP", **metrics_gap},
    {"Model":"1D-CNN-FLAT", **metrics_flat}
])

all_results_stage1 = pd.concat([conventional_results, neural_results], ignore_index=True)
all_results_stage1 = all_results_stage1.sort_values("Mean Localization Error").reset_index(drop=True)
display(all_results_stage1.round(4))
all_results_stage1.to_csv("/content/all_model_results_stage1.csv", index=False)

print("Best model:", all_results_stage1.iloc[0]["Model"])
print("Best MLE: %.4f" % all_results_stage1.iloc[0]["Mean Localization Error"])

In [ ]:
# ============================================================
# 12. 95% BOOTSTRAP CI FOR MEAN LOCALIZATION ERROR
# ============================================================
def bootstrap_mle_ci(errors, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    errors = np.asarray(errors, dtype=float)
    samples = rng.choice(errors, size=(n_boot, len(errors)), replace=True)
    means = samples.mean(axis=1)
    return np.mean(errors), np.percentile(means, 2.5), np.percentile(means, 97.5)

ci_rows = []
for name, errors in error_sets.items():
    mle, lo, hi = bootstrap_mle_ci(errors)
    ci_rows.append({"Model": name, "MLE": mle, "95% CI Lower": lo, "95% CI Upper": hi})

bootstrap_ci = pd.DataFrame(ci_rows).sort_values("MLE")
display(bootstrap_ci.round(4))
bootstrap_ci.to_csv("/content/mle_bootstrap_95CI.csv", index=False)

In [ ]:
# ============================================================
# 13. ERROR CDF
# ============================================================
plt.figure(figsize=(8,6))
for name in all_results_stage1["Model"]:
    e = np.sort(error_sets[name])
    cdf = np.arange(1, len(e)+1) / len(e)
    plt.plot(e, cdf, label=name)
plt.xlabel("Localization Error (grid units)")
plt.ylabel("Cumulative Probability")
plt.title("Cumulative Distribution of Localization Error")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("/content/localization_error_cdf_final.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 14. ERROR BOXPLOT
# ============================================================
plot_df = pd.DataFrame({name: pd.Series(err) for name, err in error_sets.items()})
plt.figure(figsize=(10,6))
plt.boxplot([plot_df[c].dropna().values for c in plot_df.columns], labels=plot_df.columns, showfliers=False)
plt.ylabel("Localization Error (grid units)")
plt.xlabel("Model")
plt.title("Localization Error Distribution Across Models")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("/content/localization_error_boxplot_final.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 15. PER-LOCATION ERROR ANALYSIS FOR THE BEST MODEL
# ============================================================
best_model = all_results_stage1.iloc[0]["Model"]
best_errors = error_sets[best_model]
per_location_error = pd.DataFrame({"location": loc_test.to_numpy(), "error": best_errors}).groupby("location")["error"].agg(["count","mean","median","max"]).sort_values("mean", ascending=False)
print("Best model:", best_model)
display(per_location_error.head(15).round(4))
per_location_error.to_csv("/content/per_location_error_best_model.csv")

In [ ]:
# ============================================================
# 16. SPATIAL ERROR MAP FOR THE BEST MODEL
# ============================================================
test_vis = pd.DataFrame({"x": y_test["x"].to_numpy(), "y": y_test["y"].to_numpy(), "error": best_errors})
plt.figure(figsize=(10,7))
sc = plt.scatter(test_vis["x"], test_vis["y"], c=test_vis["error"], s=45)
plt.colorbar(sc, label="Localization Error")
plt.xlabel("Ground-Truth X")
plt.ylabel("Ground-Truth Y")
plt.title(f"Spatial Distribution of Localization Error — {best_model}")
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig("/content/spatial_error_map_best_model.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 17. SAVE TEST PREDICTIONS FOR ALL MODELS
# ============================================================
prediction_results = pd.DataFrame({
    "location": loc_test.to_numpy(),
    "true_x": y_test["x"].to_numpy(),
    "true_y": y_test["y"].to_numpy(),
})

for name, pred in preds.items():
    safe = name.lower().replace(" ", "_").replace("-", "_")
    prediction_results[f"{safe}_x"] = pred[:,0]
    prediction_results[f"{safe}_y"] = pred[:,1]
    prediction_results[f"{safe}_error"] = error_sets[name]

prediction_results.to_csv("/content/test_predictions_all_models.csv", index=False)
display(prediction_results.head())

In [ ]:
# ============================================================
# 18. SAVE MODELS AND SCALER
# ============================================================
import joblib
joblib.dump(scaler, "/content/final_minmax_scaler.pkl")
joblib.dump(knn_final, "/content/final_knn.pkl")
joblib.dump(dt_final, "/content/final_decision_tree.pkl")
joblib.dump(rf_final, "/content/final_random_forest.pkl")
joblib.dump(svr_final, "/content/final_svr.pkl")
mlp.save("/content/final_mlp.keras")
cnn_gap.save("/content/final_cnn_gap.keras")
cnn_flat.save("/content/final_cnn_flat.keras")
print("Models and scaler saved.")

Stage 2

In [ ]:
# ============================================================
# STAGE 2 — UNSEEN-LOCATION EVALUATION
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42

# ------------------------------------------------------------
# Load the original labeled dataset
# ------------------------------------------------------------
stage2_df = pd.read_csv(LABELED_PATH)

print("Dataset shape:", stage2_df.shape)
print("Columns:")
print(stage2_df.columns.tolist())

# ------------------------------------------------------------
# Identify the location column
# ------------------------------------------------------------
LOCATION_COL = "location"

assert LOCATION_COL in stage2_df.columns, (
    f"Column '{LOCATION_COL}' not found. "
    f"Available columns: {stage2_df.columns.tolist()}"
)

# ------------------------------------------------------------
# Unique physical locations
# ------------------------------------------------------------
unique_locations = np.sort(
    stage2_df[LOCATION_COL].dropna().unique()
)

print("\nTotal physical locations:", len(unique_locations))
print("Total observations:", len(stage2_df))

# ------------------------------------------------------------
# Hold out complete physical locations
# ------------------------------------------------------------
train_locations, test_locations = train_test_split(
    unique_locations,
    test_size=0.20,
    random_state=SEED
)

train_locations = np.sort(train_locations)
test_locations = np.sort(test_locations)

print("\nSTAGE 2 LOCATION SPLIT")
print("----------------------")
print("Training locations:", len(train_locations))
print("Test locations:", len(test_locations))

print("\nHeld-out test locations:")
print(test_locations)

# ------------------------------------------------------------
# Create development and completely unseen test sets
# ------------------------------------------------------------
dev_df = stage2_df[
    stage2_df[LOCATION_COL].isin(train_locations)
].copy()

test_spatial_df = stage2_df[
    stage2_df[LOCATION_COL].isin(test_locations)
].copy()

print("\nOBSERVATION SPLIT")
print("-----------------")
print("Development observations:", len(dev_df))
print("Unseen test observations:", len(test_spatial_df))

# ------------------------------------------------------------
# Verify NO location overlap
# ------------------------------------------------------------
dev_locations = set(dev_df[LOCATION_COL])
heldout_locations = set(test_spatial_df[LOCATION_COL])

overlap = dev_locations.intersection(heldout_locations)

print("\nLOCATION LEAKAGE CHECK")
print("----------------------")
print("Overlapping locations:", len(overlap))

assert len(overlap) == 0, (
    f"ERROR: Location leakage detected: {overlap}"
)

print("✓ No location overlap between development and test.")

In [ ]:
# ============================================================
# STAGE 2 — CELL 2
# CREATE TRAIN / VALIDATION FROM DEVELOPMENT LOCATIONS
# ============================================================

# ------------------------------------------------------------
# Split the 84 development locations into:
#   70% training locations
#   30% validation locations
#
# The 21 held-out test locations remain completely untouched.
# ------------------------------------------------------------

train_locations_s2, val_locations_s2 = train_test_split(
    train_locations,
    test_size=0.30,
    random_state=SEED
)

train_locations_s2 = np.sort(train_locations_s2)
val_locations_s2 = np.sort(val_locations_s2)

# ------------------------------------------------------------
# Create the three datasets
# ------------------------------------------------------------

train_s2_df = stage2_df[
    stage2_df[LOCATION_COL].isin(train_locations_s2)
].copy()

val_s2_df = stage2_df[
    stage2_df[LOCATION_COL].isin(val_locations_s2)
].copy()

test_s2_df = stage2_df[
    stage2_df[LOCATION_COL].isin(test_locations)
].copy()

# ------------------------------------------------------------
# Display split information
# ------------------------------------------------------------

print("STAGE 2 FINAL SPLIT")
print("===================")

print("\nPhysical locations")
print("------------------")
print("Training locations  :", len(train_locations_s2))
print("Validation locations:", len(val_locations_s2))
print("Unseen test locations:", len(test_locations))

print("\nObservations")
print("------------")
print("Training observations  :", len(train_s2_df))
print("Validation observations:", len(val_s2_df))
print("Unseen test observations:", len(test_s2_df))

# ------------------------------------------------------------
# Check location overlap
# ------------------------------------------------------------

train_set_s2 = set(train_s2_df[LOCATION_COL])
val_set_s2 = set(val_s2_df[LOCATION_COL])
test_set_s2 = set(test_s2_df[LOCATION_COL])

print("\nLOCATION OVERLAP CHECK")
print("======================")

print("Train ∩ Validation:", len(train_set_s2 & val_set_s2))
print("Train ∩ Test      :", len(train_set_s2 & test_set_s2))
print("Validation ∩ Test :", len(val_set_s2 & test_set_s2))

assert len(train_set_s2 & val_set_s2) == 0
assert len(train_set_s2 & test_set_s2) == 0
assert len(val_set_s2 & test_set_s2) == 0

print("\n✓ ZERO LOCATION OVERLAP")
print("✓ 21 test locations remain completely unseen")

In [ ]:
# ============================================================
# STAGE 2 — CELL 3
# PREPARE FEATURES, TARGETS AND TRAIN-ONLY SCALING
# ============================================================

from sklearn.preprocessing import MinMaxScaler

# ------------------------------------------------------------
# 13 BLE RSSI features
# ------------------------------------------------------------

RSSI_COLS = [
    "b3001", "b3002", "b3003", "b3004", "b3005",
    "b3006", "b3007", "b3008", "b3009", "b3010",
    "b3011", "b3012", "b3013"
]

# ------------------------------------------------------------
# Verify RSSI columns
# ------------------------------------------------------------

missing_cols = [
    col for col in RSSI_COLS
    if col not in stage2_df.columns
]

assert len(missing_cols) == 0, (
    f"Missing RSSI columns: {missing_cols}"
)

# ------------------------------------------------------------
# Use the EXACT location_to_grid() function from Stage 1
# ------------------------------------------------------------

print("Location mapping object:", location_to_grid)
print("Location mapping type:", type(location_to_grid))

# ------------------------------------------------------------
# Add true x,y coordinates using the existing function
# ------------------------------------------------------------

def add_coordinates_s2(df):
    df = df.copy()

    mapped = df[LOCATION_COL].apply(location_to_grid)

    # Check that every location has a valid coordinate pair
    invalid = mapped[
        mapped.isna()
    ]

    assert len(invalid) == 0, (
        "Some locations could not be mapped to coordinates."
    )

    df["true_x"] = mapped.apply(lambda z: z[0])
    df["true_y"] = mapped.apply(lambda z: z[1])

    return df


train_s2_df = add_coordinates_s2(train_s2_df)
val_s2_df   = add_coordinates_s2(val_s2_df)
test_s2_df  = add_coordinates_s2(test_s2_df)

# ------------------------------------------------------------
# Feature matrices
# ------------------------------------------------------------

X_train_s2 = train_s2_df[RSSI_COLS].astype(float).values
X_val_s2   = val_s2_df[RSSI_COLS].astype(float).values
X_test_s2  = test_s2_df[RSSI_COLS].astype(float).values

# ------------------------------------------------------------
# Coordinate targets
# ------------------------------------------------------------

y_train_s2 = train_s2_df[
    ["true_x", "true_y"]
].astype(float).values

y_val_s2 = val_s2_df[
    ["true_x", "true_y"]
].astype(float).values

y_test_s2 = test_s2_df[
    ["true_x", "true_y"]
].astype(float).values

# ------------------------------------------------------------
# TRAIN-ONLY MinMax scaling
# ------------------------------------------------------------

scaler_s2 = MinMaxScaler()

X_train_s2_scaled = scaler_s2.fit_transform(X_train_s2)

X_val_s2_scaled = scaler_s2.transform(
    X_val_s2
)

X_test_s2_scaled = scaler_s2.transform(
    X_test_s2
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("\nSTAGE 2 FEATURE PREPARATION")
print("===========================")

print("\nFeature shapes")
print("----------------")
print("X_train:", X_train_s2_scaled.shape)
print("X_val  :", X_val_s2_scaled.shape)
print("X_test :", X_test_s2_scaled.shape)

print("\nTarget shapes")
print("----------------")
print("y_train:", y_train_s2.shape)
print("y_val  :", y_val_s2.shape)
print("y_test :", y_test_s2.shape)

print("\nCoordinate ranges")
print("-----------------")
print(
    "Train X:",
    y_train_s2[:, 0].min(),
    "to",
    y_train_s2[:, 0].max()
)
print(
    "Train Y:",
    y_train_s2[:, 1].min(),
    "to",
    y_train_s2[:, 1].max()
)

print("\nScaler")
print("----------------")
print("✓ Fitted ONLY on the 58 training locations")

print("\nScaled ranges")
print("----------------")
print(
    "Train:",
    X_train_s2_scaled.min(),
    "to",
    X_train_s2_scaled.max()
)

print(
    "Validation:",
    X_val_s2_scaled.min(),
    "to",
    X_val_s2_scaled.max()
)

print(
    "Unseen Test:",
    X_test_s2_scaled.min(),
    "to",
    X_test_s2_scaled.max()
)

print("\n✓ Stage 2 feature preparation complete.")
print("✓ Unseen test locations were not used for fitting the scaler.")

In [ ]:
# Check variables containing location/coordinate information

[x for x in globals().keys()
 if any(k in x.lower() for k in ["coord", "location", "loc_map", "mapping"])]

In [ ]:
# ============================================================
# STAGE 2 — CELL 4
# CONVENTIONAL MODELS — UNSEEN LOCATION EVALUATION
# ============================================================

from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Localization error function
# ------------------------------------------------------------

def localization_errors(y_true, y_pred):
    errors = np.sqrt(
        np.sum((y_true - y_pred) ** 2, axis=1)
    )
    return errors


def calculate_metrics_s2(y_true, y_pred):
    errors = localization_errors(y_true, y_pred)

    return {
        "MLE": np.mean(errors),
        "Median": np.median(errors),
        "Std": np.std(errors),
        "75th_Percentile": np.percentile(errors, 75),
        "90th_Percentile": np.percentile(errors, 90),
        "Max": np.max(errors),
        "MAE_X": mean_absolute_error(
            y_true[:, 0], y_pred[:, 0]
        ),
        "MAE_Y": mean_absolute_error(
            y_true[:, 1], y_pred[:, 1]
        ),
        "RMSE_X": np.sqrt(
            mean_squared_error(
                y_true[:, 0], y_pred[:, 0]
            )
        ),
        "RMSE_Y": np.sqrt(
            mean_squared_error(
                y_true[:, 1], y_pred[:, 1]
            )
        )
    }


# ------------------------------------------------------------
# Models
# ------------------------------------------------------------

models_s2 = {

    "kNN": KNeighborsRegressor(
        n_neighbors=5,
        weights="distance",
        metric="euclidean"
    ),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=10,
        min_samples_leaf=2,
        random_state=SEED
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=1,
        random_state=SEED,
        n_jobs=-1
    ),

    "SVR": MultiOutputRegressor(
        SVR(
            C=100,
            gamma="scale",
            epsilon=0.10,
            kernel="rbf"
        )
    )
}


# ------------------------------------------------------------
# Train and evaluate
# ------------------------------------------------------------

stage2_predictions = {}
stage2_conventional_results = []

for name, model in models_s2.items():

    print(f"\nTraining {name}...")

    # Train ONLY on the 58 training locations
    model.fit(
        X_train_s2_scaled,
        y_train_s2
    )

    # Validation prediction
    val_pred = model.predict(
        X_val_s2_scaled
    )

    # Completely unseen-location test prediction
    test_pred = model.predict(
        X_test_s2_scaled
    )

    # Metrics
    val_metrics = calculate_metrics_s2(
        y_val_s2,
        val_pred
    )

    test_metrics = calculate_metrics_s2(
        y_test_s2,
        test_pred
    )

    stage2_predictions[name] = {
        "validation": val_pred,
        "test": test_pred
    }

    stage2_conventional_results.append({
        "Model": name,
        "Validation_MLE": val_metrics["MLE"],
        "Test_MLE": test_metrics["MLE"],
        "Test_Median": test_metrics["Median"],
        "Test_Std": test_metrics["Std"],
        "Test_75th": test_metrics["75th_Percentile"],
        "Test_90th": test_metrics["90th_Percentile"],
        "Test_Max": test_metrics["Max"],
        "Test_MAE_X": test_metrics["MAE_X"],
        "Test_MAE_Y": test_metrics["MAE_Y"],
        "Test_RMSE_X": test_metrics["RMSE_X"],
        "Test_RMSE_Y": test_metrics["RMSE_Y"]
    })

    print(
        f"Validation MLE : "
        f"{val_metrics['MLE']:.4f} grid-coordinate units"
    )

    print(
        f"Unseen Test MLE : "
        f"{test_metrics['MLE']:.4f} grid-coordinate units"
    )


# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

stage2_conventional_results = pd.DataFrame(
    stage2_conventional_results
).sort_values(
    "Test_MLE"
).reset_index(drop=True)

print("\n")
print("=" * 80)
print("STAGE 2 — UNSEEN-LOCATION CONVENTIONAL RESULTS")
print("=" * 80)

display(stage2_conventional_results.round(4))

In [ ]:
# ============================================================
# STAGE 2 — CELL 5
# NEURAL MODELS — UNSEEN LOCATION EVALUATION
# ============================================================

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    Conv1D,
    BatchNormalization,
    Flatten,
    GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.optimizers import Adam

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

tf.keras.backend.clear_session()

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ------------------------------------------------------------
# Common training settings
# ------------------------------------------------------------

BATCH_SIZE_S2 = 32
EPOCHS_S2 = 300

early_stop_s2 = EarlyStopping(
    monitor="val_loss",
    patience=25,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_s2 = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=10,
    min_lr=1e-6,
    verbose=1
)

# ============================================================
# MLP
# ============================================================

mlp_s2 = Sequential([
    Input(shape=(13,)),

    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.30),

    Dense(64, activation="relu"),
    Dropout(0.20),

    Dense(32, activation="relu"),

    Dense(2, activation="linear")
])

mlp_s2.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse"
)

print("\nTraining MLP...")

history_mlp_s2 = mlp_s2.fit(
    X_train_s2_scaled,
    y_train_s2,
    validation_data=(
        X_val_s2_scaled,
        y_val_s2
    ),
    epochs=EPOCHS_S2,
    batch_size=BATCH_SIZE_S2,
    callbacks=[
        early_stop_s2,
        reduce_lr_s2
    ],
    verbose=1
)

mlp_test_pred_s2 = mlp_s2.predict(
    X_test_s2_scaled,
    verbose=0
)

mlp_metrics_s2 = calculate_metrics_s2(
    y_test_s2,
    mlp_test_pred_s2
)

print(
    f"\nMLP unseen-location Test MLE: "
    f"{mlp_metrics_s2['MLE']:.4f} grid-coordinate units"
)


# ============================================================
# CNN-GAP
# ============================================================

tf.keras.backend.clear_session()

cnn_gap_s2 = Sequential([
    Input(shape=(13, 1)),

    Conv1D(
        32,
        kernel_size=3,
        padding="same",
        activation="relu"
    ),
    BatchNormalization(),

    Conv1D(
        64,
        kernel_size=3,
        padding="same",
        activation="relu"
    ),
    BatchNormalization(),

    GlobalAveragePooling1D(),

    Dense(64, activation="relu"),
    Dropout(0.30),

    Dense(2, activation="linear")
])

cnn_gap_s2.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse"
)

print("\nTraining CNN-GAP...")

history_cnn_gap_s2 = cnn_gap_s2.fit(
    X_train_s2_scaled[..., np.newaxis],
    y_train_s2,
    validation_data=(
        X_val_s2_scaled[..., np.newaxis],
        y_val_s2
    ),
    epochs=EPOCHS_S2,
    batch_size=BATCH_SIZE_S2,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=25,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=10,
            min_lr=1e-6,
            verbose=1
        )
    ],
    verbose=1
)

cnn_gap_test_pred_s2 = cnn_gap_s2.predict(
    X_test_s2_scaled[..., np.newaxis],
    verbose=0
)

cnn_gap_metrics_s2 = calculate_metrics_s2(
    y_test_s2,
    cnn_gap_test_pred_s2
)

print(
    f"\nCNN-GAP unseen-location Test MLE: "
    f"{cnn_gap_metrics_s2['MLE']:.4f} grid-coordinate units"
)


# ============================================================
# CNN-FLAT
# ============================================================

tf.keras.backend.clear_session()

cnn_flat_s2 = Sequential([
    Input(shape=(13, 1)),

    Conv1D(
        32,
        kernel_size=3,
        padding="same",
        activation="relu"
    ),
    BatchNormalization(),

    Conv1D(
        64,
        kernel_size=3,
        padding="same",
        activation="relu"
    ),
    BatchNormalization(),

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.30),

    Dense(64, activation="relu"),
    Dropout(0.20),

    Dense(2, activation="linear")
])

cnn_flat_s2.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse"
)

print("\nTraining CNN-FLAT...")

history_cnn_flat_s2 = cnn_flat_s2.fit(
    X_train_s2_scaled[..., np.newaxis],
    y_train_s2,
    validation_data=(
        X_val_s2_scaled[..., np.newaxis],
        y_val_s2
    ),
    epochs=EPOCHS_S2,
    batch_size=BATCH_SIZE_S2,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=25,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=10,
            min_lr=1e-6,
            verbose=1
        )
    ],
    verbose=1
)

cnn_flat_test_pred_s2 = cnn_flat_s2.predict(
    X_test_s2_scaled[..., np.newaxis],
    verbose=0
)

cnn_flat_metrics_s2 = calculate_metrics_s2(
    y_test_s2,
    cnn_flat_test_pred_s2
)

print(
    f"\nCNN-FLAT unseen-location Test MLE: "
    f"{cnn_flat_metrics_s2['MLE']:.4f} grid-coordinate units"
)

In [ ]:
# ============================================================
# STAGE 2 — CELL 6
# MASTER RESULTS + STAGE 1 → STAGE 2 GENERALIZATION ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Stage 2 test MLE values
# ------------------------------------------------------------

stage2_mle = {
    "SVR": 1.8134,
    "Decision Tree": 2.0048,
    "MLP": 1.9948,
    "Random Forest": 2.0313,
    "CNN-FLAT": 2.0379,
    "kNN": 2.0902,
    "CNN-GAP": 3.1471
}

# ------------------------------------------------------------
# Stage 1 accepted MLE values
# ------------------------------------------------------------

stage1_mle = {
    "SVR": 1.6079,
    "Decision Tree": 1.6755,
    "MLP": 2.0929,
    "Random Forest": 1.6332,
    "CNN-FLAT": 2.0254,
    "kNN": 1.6279,
    "CNN-GAP": 3.0200
}

# ------------------------------------------------------------
# Build comparison table
# ------------------------------------------------------------

models = [
    "SVR",
    "Decision Tree",
    "MLP",
    "Random Forest",
    "CNN-FLAT",
    "kNN",
    "CNN-GAP"
]

comparison_s2 = pd.DataFrame({
    "Model": models,
    "Stage1_Seen_MLE_grid_units": [stage1_mle[m] for m in models],
    "Stage2_Unseen_MLE_grid_units": [stage2_mle[m] for m in models]
})

# Absolute degradation
comparison_s2["Absolute_Degradation_grid_units"] = (
    comparison_s2["Stage2_Unseen_MLE_grid_units"]
    - comparison_s2["Stage1_Seen_MLE_grid_units"]
)

# Percentage degradation
comparison_s2["Degradation_Percent"] = (
    comparison_s2["Absolute_Degradation_grid_units"]
    / comparison_s2["Stage1_Seen_MLE_grid_units"]
) * 100

# Stage 2 ranking
comparison_s2["Stage2_Rank"] = (
    comparison_s2["Stage2_Unseen_MLE_grid_units"]
    .rank(method="min")
    .astype(int)
)

# Sort by Stage 2 performance
comparison_s2 = comparison_s2.sort_values(
    "Stage2_Unseen_MLE_grid_units"
).reset_index(drop=True)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 90)
print("STAGE 2 — MASTER GENERALIZATION COMPARISON")
print("=" * 90)

display(
    comparison_s2.round({
        "Stage1_Seen_MLE_grid_units": 4,
        "Stage2_Unseen_MLE_grid_units": 4,
        "Absolute_Degradation_grid_units": 4,
        "Degradation_Percent": 2
    })
)

# ------------------------------------------------------------
# Best model
# ------------------------------------------------------------

best_model = comparison_s2.iloc[0]

print("\n" + "=" * 90)
print("BEST UNSEEN-LOCATION MODEL")
print("=" * 90)

print(f"Model              : {best_model['Model']}")
print(f"Stage 1 MLE        : {best_model['Stage1_Seen_MLE_grid_units']:.4f} grid-coordinate units")
print(f"Stage 2 MLE        : {best_model['Stage2_Unseen_MLE_grid_units']:.4f} grid-coordinate units")
print(f"Absolute change    : {best_model['Absolute_Degradation_grid_units']:.4f} grid-coordinate units")
print(f"Percentage change  : {best_model['Degradation_Percent']:.2f}%")

# ------------------------------------------------------------
# Most stable model
# ------------------------------------------------------------

most_stable = comparison_s2.loc[
    comparison_s2["Degradation_Percent"].idxmin()
]

print("\n" + "=" * 90)
print("MOST STABLE MODEL FROM SEEN → UNSEEN LOCATIONS")
print("=" * 90)

print(f"Model              : {most_stable['Model']}")
print(f"Degradation        : {most_stable['Degradation_Percent']:.2f}%")

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

comparison_s2.to_csv(
    "stage2_master_generalization_comparison.csv",
    index=False
)

print("\n✓ Saved:")
print("stage2_master_generalization_comparison.csv")

In [ ]:
# ============================================================
# STAGE 2 — CELL 7
# UNSEEN-LOCATION ERROR DISTRIBUTION + STATISTICAL COMPARISON
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

# ------------------------------------------------------------
# Collect Stage 2 predictions
# ------------------------------------------------------------

stage2_predictions = {
    "SVR": stage2_predictions["SVR"]["test"],
    "Decision Tree": stage2_predictions["Decision Tree"]["test"],
    "Random Forest": stage2_predictions["Random Forest"]["test"],
    "kNN": stage2_predictions["kNN"]["test"],
    "MLP": mlp_test_pred_s2,
    "CNN-GAP": cnn_gap_test_pred_s2,
    "CNN-FLAT": cnn_flat_test_pred_s2
}

# ------------------------------------------------------------
# Calculate per-observation Euclidean localization errors
# ------------------------------------------------------------

stage2_errors = {}

for model, predictions in stage2_predictions.items():

    errors = np.sqrt(
        np.sum(
            (y_test_s2 - predictions) ** 2,
            axis=1
        )
    )

    stage2_errors[model] = errors

# ------------------------------------------------------------
# Error distribution summary
# ------------------------------------------------------------

error_summary_s2 = pd.DataFrame({
    model: {
        "MLE_grid_units": np.mean(errors),
        "Median_grid_units": np.median(errors),
        "Std_grid_units": np.std(errors),
        "75th_grid_units": np.percentile(errors, 75),
        "90th_grid_units": np.percentile(errors, 90),
        "95th_grid_units": np.percentile(errors, 95),
        "Max_grid_units": np.max(errors)
    }
    for model, errors in stage2_errors.items()
}).T

error_summary_s2 = error_summary_s2.sort_values(
    "MLE_grid_units"
)

print("=" * 95)
print("STAGE 2 — UNSEEN-LOCATION ERROR DISTRIBUTION")
print("=" * 95)

display(
    error_summary_s2.round(4)
)

# ------------------------------------------------------------
# Pairwise Wilcoxon comparison against SVR
# ------------------------------------------------------------

print("\n" + "=" * 95)
print("STAGE 2 — PAIRED STATISTICAL COMPARISON AGAINST SVR")
print("=" * 95)

svr_errors = stage2_errors["SVR"]

statistical_results = []

for model, errors in stage2_errors.items():

    if model == "SVR":
        continue

    try:
        statistic, p_value = wilcoxon(
            svr_errors,
            errors,
            alternative="two-sided"
        )

        statistical_results.append({
            "Comparison": f"SVR vs {model}",
            "Wilcoxon_statistic": statistic,
            "p_value": p_value,
            "Significant_p<0.05": p_value < 0.05
        })

    except Exception as e:

        statistical_results.append({
            "Comparison": f"SVR vs {model}",
            "Wilcoxon_statistic": np.nan,
            "p_value": np.nan,
            "Significant_p<0.05": False
        })

statistical_results_s2 = pd.DataFrame(
    statistical_results
)

display(
    statistical_results_s2.round(6)
)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

error_summary_s2.to_csv(
    "stage2_unseen_error_distribution.csv"
)

statistical_results_s2.to_csv(
    "stage2_svr_statistical_comparison.csv",
    index=False
)

print("\n✓ Saved:")
print("stage2_unseen_error_distribution.csv")
print("stage2_svr_statistical_comparison.csv")

In [ ]:
# ============================================================
# STAGE 2 — COMPLETE OFFLINE BACKUP
# ============================================================

import os
import pickle
import json
import shutil
import zipfile
from datetime import datetime

# ------------------------------------------------------------
# 1. Create backup directory
# ------------------------------------------------------------

STAGE2_BACKUP_DIR = "/content/STAGE2_COMPLETE_BACKUP"

os.makedirs(STAGE2_BACKUP_DIR, exist_ok=True)

print("=" * 90)
print("CREATING COMPLETE STAGE 2 OFFLINE BACKUP")
print("=" * 90)

# ------------------------------------------------------------
# 2. Save CSV result files
# ------------------------------------------------------------

csv_files = [
    "stage2_master_generalization_comparison.csv",
    "stage2_unseen_error_distribution.csv",
    "stage2_svr_statistical_comparison.csv"
]

for file_name in csv_files:

    source = f"/content/{file_name}"
    destination = os.path.join(
        STAGE2_BACKUP_DIR,
        file_name
    )

    if os.path.exists(source):
        shutil.copy2(source, destination)
        print(f"✓ Saved: {file_name}")
    else:
        print(f"⚠ Not found: {file_name}")

# ------------------------------------------------------------
# 3. Save Stage 2 conventional results
# ------------------------------------------------------------

if "stage2_conventional_results" in globals():

    stage2_conventional_results.to_csv(
        os.path.join(
            STAGE2_BACKUP_DIR,
            "stage2_conventional_results.csv"
        ),
        index=False
    )

    print("✓ Saved: stage2_conventional_results.csv")

# ------------------------------------------------------------
# 4. Save Stage 2 error summary
# ------------------------------------------------------------

if "error_summary_s2" in globals():

    error_summary_s2.to_csv(
        os.path.join(
            STAGE2_BACKUP_DIR,
            "stage2_error_summary.csv"
        )
    )

    print("✓ Saved: stage2_error_summary.csv")

# ------------------------------------------------------------
# 5. Save all Stage 2 predictions
# ------------------------------------------------------------

prediction_backup = {}

# Conventional models
if "stage2_predictions" in globals():

    for model, prediction in stage2_predictions.items():

        if isinstance(prediction, dict):

            if "test" in prediction:
                prediction_backup[model] = {
                    "test": np.asarray(
                        prediction["test"]
                    )
                }

        else:
            prediction_backup[model] = {
                "test": np.asarray(prediction)
            }

# Neural models
if "mlp_test_pred_s2" in globals():
    prediction_backup["MLP"] = {
        "test": np.asarray(mlp_test_pred_s2)
    }

if "cnn_gap_test_pred_s2" in globals():
    prediction_backup["CNN-GAP"] = {
        "test": np.asarray(cnn_gap_test_pred_s2)
    }

if "cnn_flat_test_pred_s2" in globals():
    prediction_backup["CNN-FLAT"] = {
        "test": np.asarray(cnn_flat_test_pred_s2)
    }

with open(
    os.path.join(
        STAGE2_BACKUP_DIR,
        "stage2_predictions.pkl"
    ),
    "wb"
) as f:

    pickle.dump(
        prediction_backup,
        f
    )

print(
    f"✓ Saved predictions for "
    f"{len(prediction_backup)} models"
)

# ------------------------------------------------------------
# 6. Save per-observation error arrays
# ------------------------------------------------------------

if "stage2_errors" in globals():

    with open(
        os.path.join(
            STAGE2_BACKUP_DIR,
            "stage2_error_arrays.pkl"
        ),
        "wb"
    ) as f:

        pickle.dump(
            stage2_errors,
            f
        )

    print("✓ Saved: stage2_error_arrays.pkl")

# ------------------------------------------------------------
# 7. Save ground-truth test coordinates
# ------------------------------------------------------------

np.save(
    os.path.join(
        STAGE2_BACKUP_DIR,
        "y_test_s2.npy"
    ),
    np.asarray(y_test_s2)
)

print("✓ Saved: y_test_s2.npy")

# ------------------------------------------------------------
# 8. Save Stage 2 scaled features
# ------------------------------------------------------------

np.save(
    os.path.join(
        STAGE2_BACKUP_DIR,
        "X_train_s2_scaled.npy"
    ),
    np.asarray(X_train_s2_scaled)
)

np.save(
    os.path.join(
        STAGE2_BACKUP_DIR,
        "X_val_s2_scaled.npy"
    ),
    np.asarray(X_val_s2_scaled)
)

np.save(
    os.path.join(
        STAGE2_BACKUP_DIR,
        "X_test_s2_scaled.npy"
    ),
    np.asarray(X_test_s2_scaled)
)

print("✓ Saved Stage 2 scaled feature arrays")

# ------------------------------------------------------------
# 9. Save Stage 2 targets
# ------------------------------------------------------------

np.save(
    os.path.join(
        STAGE2_BACKUP_DIR,
        "y_train_s2.npy"
    ),
    np.asarray(y_train_s2)
)

np.save(
    os.path.join(
        STAGE2_BACKUP_DIR,
        "y_val_s2.npy"
    ),
    np.asarray(y_val_s2)
)

print("✓ Saved Stage 2 target arrays")

# ------------------------------------------------------------
# 10. Save location split information
# ------------------------------------------------------------

split_information = {
    "train_locations": list(train_locations_s2),
    "validation_locations": list(val_locations_s2),
    "unseen_test_locations": list(heldout_locations),

    "n_train_locations": len(train_locations_s2),
    "n_validation_locations": len(val_locations_s2),
    "n_test_locations": len(heldout_locations),

    "n_train_observations": len(X_train_s2_scaled),
    "n_validation_observations": len(X_val_s2_scaled),
    "n_test_observations": len(X_test_s2_scaled),

    "random_state": 42,

    "description": (
        "Stage 2 location-disjoint evaluation. "
        "Training, validation and test locations have zero overlap."
    )
}

with open(
    os.path.join(
        STAGE2_BACKUP_DIR,
        "stage2_location_split.json"
    ),
    "w"
) as f:

    json.dump(
        split_information,
        f,
        indent=4
    )

print("✓ Saved: stage2_location_split.json")

# ------------------------------------------------------------
# 11. Save Stage 2 scaler
# ------------------------------------------------------------

if "scaler_s2" in globals():

    with open(
        os.path.join(
            STAGE2_BACKUP_DIR,
            "stage2_minmax_scaler.pkl"
        ),
        "wb"
    ) as f:

        pickle.dump(
            scaler_s2,
            f
        )

    print("✓ Saved: stage2_minmax_scaler.pkl")

# ------------------------------------------------------------
# 12. Save Stage 2 model objects
# ------------------------------------------------------------

# Conventional models
conventional_models = {}

for variable_name, save_name in [
    ("knn_s2", "kNN"),
    ("dt_s2", "Decision_Tree"),
    ("rf_s2", "Random_Forest"),
    ("svr_s2", "SVR")
]:

    if variable_name in globals():

        conventional_models[save_name] = globals()[variable_name]

if conventional_models:

    with open(
        os.path.join(
            STAGE2_BACKUP_DIR,
            "stage2_conventional_models.pkl"
        ),
        "wb"
    ) as f:

        pickle.dump(
            conventional_models,
            f
        )

    print("✓ Saved conventional model objects")

# ------------------------------------------------------------
# 13. Save neural models
# ------------------------------------------------------------

if "mlp_s2" in globals():
    mlp_s2.save(
        os.path.join(
            STAGE2_BACKUP_DIR,
            "stage2_MLP.keras"
        )
    )
    print("✓ Saved: stage2_MLP.keras")

if "cnn_gap_s2" in globals():
    cnn_gap_s2.save(
        os.path.join(
            STAGE2_BACKUP_DIR,
            "stage2_CNN_GAP.keras"
        )
    )
    print("✓ Saved: stage2_CNN_GAP.keras")

if "cnn_flat_s2" in globals():
    cnn_flat_s2.save(
        os.path.join(
            STAGE2_BACKUP_DIR,
            "stage2_CNN_FLAT.keras"
        )
    )
    print("✓ Saved: stage2_CNN_FLAT.keras")

# ------------------------------------------------------------
# 14. Save training histories
# ------------------------------------------------------------

histories = {}

if "history_mlp_s2" in globals():
    histories["MLP"] = history_mlp_s2.history

if "history_cnn_gap_s2" in globals():
    histories["CNN-GAP"] = history_cnn_gap_s2.history

if "history_cnn_flat_s2" in globals():
    histories["CNN-FLAT"] = history_cnn_flat_s2.history

with open(
    os.path.join(
        STAGE2_BACKUP_DIR,
        "stage2_training_histories.pkl"
    ),
    "wb"
) as f:

    pickle.dump(
        histories,
        f
    )

print("✓ Saved: stage2_training_histories.pkl")

# ------------------------------------------------------------
# 15. Save backup metadata
# ------------------------------------------------------------

backup_metadata = {
    "backup_created": datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    ),

    "stage": "Stage 2 — Unseen-location evaluation",

    "dataset_observations": 1420,
    "physical_locations": 105,

    "training_locations": 58,
    "validation_locations": 26,
    "unseen_test_locations": 21,

    "training_observations": 790,
    "validation_observations": 319,
    "unseen_test_observations": 311,

    "models": [
        "SVR",
        "MLP",
        "Decision Tree",
        "Random Forest",
        "CNN-FLAT",
        "kNN",
        "CNN-GAP"
    ],

    "best_model": "SVR",
    "best_unseen_test_MLE_grid_units": 1.8134,

    "description": (
        "Complete Stage 2 backup containing results, "
        "predictions, errors, splits, scaler, trained models, "
        "and neural-network training histories."
    )
}

with open(
    os.path.join(
        STAGE2_BACKUP_DIR,
        "STAGE2_BACKUP_METADATA.json"
    ),
    "w"
) as f:

    json.dump(
        backup_metadata,
        f,
        indent=4
    )

print("✓ Saved: STAGE2_BACKUP_METADATA.json")

# ------------------------------------------------------------
# 16. Create ZIP archive
# ------------------------------------------------------------

zip_path = "/content/STAGE2_COMPLETE_BACKUP.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for root, dirs, files in os.walk(
        STAGE2_BACKUP_DIR
    ):

        for file in files:

            file_path = os.path.join(
                root,
                file
            )

            arcname = os.path.relpath(
                file_path,
                STAGE2_BACKUP_DIR
            )

            zipf.write(
                file_path,
                arcname
            )

# ------------------------------------------------------------
# 17. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("STAGE 2 BACKUP COMPLETE")
print("=" * 90)

print(f"\nBackup folder:")
print(STAGE2_BACKUP_DIR)

print(f"\nZIP archive:")
print(zip_path)

print("\nFiles backed up:")

for file_name in sorted(
    os.listdir(STAGE2_BACKUP_DIR)
):

    file_path = os.path.join(
        STAGE2_BACKUP_DIR,
        file_name
    )

    size_mb = os.path.getsize(
        file_path
    ) / (1024 ** 2)

    print(
        f"  ✓ {file_name:<45} "
        f"{size_mb:.3f} MB"
    )

zip_size_mb = os.path.getsize(
    zip_path
) / (1024 ** 2)

print(
    f"\nZIP size: {zip_size_mb:.3f} MB"
)

print("\n✓ Stage 2 can now be restored without retraining.")

In [ ]:
# ============================================================
# STAGE 3 — CELL 1
# TEMPORAL/GROUPED EVALUATION — DATA PREPARATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit

# ------------------------------------------------------------
# Load original labelled dataset
# ------------------------------------------------------------

df_s3 = pd.read_csv(LABELED_PATH)

LOCATION_COL_S3 = "location"
DATE_COL_S3 = "date"

print("=" * 90)
print("STAGE 3 — TEMPORAL/GROUPED EVALUATION")
print("=" * 90)

print("\nDataset shape:")
print(df_s3.shape)

print("\nColumns:")
print(df_s3.columns.tolist())

# ------------------------------------------------------------
# Convert date/time column
# ------------------------------------------------------------

df_s3[DATE_COL_S3] = pd.to_datetime(
    df_s3[DATE_COL_S3],
    errors="coerce"
)

print("\nMissing timestamps:")
print(df_s3[DATE_COL_S3].isna().sum())

# ------------------------------------------------------------
# Sort chronologically
# ------------------------------------------------------------

df_s3 = df_s3.sort_values(
    DATE_COL_S3
).reset_index(drop=True)

# ------------------------------------------------------------
# Create temporal groups
#
# We group observations by date/time sequence rather than
# randomly splitting individual observations.
# ------------------------------------------------------------

df_s3["temporal_group_s3"] = (
    df_s3[DATE_COL_S3]
    .dt.floor("5min")
)

print("\nTemporal groups:")
print(
    df_s3["temporal_group_s3"]
    .nunique()
)

# ------------------------------------------------------------
# Display temporal coverage
# ------------------------------------------------------------

print("\nTemporal coverage:")
print("Start:", df_s3[DATE_COL_S3].min())
print("End  :", df_s3[DATE_COL_S3].max())

# ------------------------------------------------------------
# Grouped train/test split
#
# 80% temporal groups -> development
# 20% temporal groups -> test
#
# random_state is retained for reproducibility.
# ------------------------------------------------------------

groups_s3 = df_s3["temporal_group_s3"].values

gss_s3 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx_s3, test_idx_s3 = next(
    gss_s3.split(
        df_s3,
        groups=groups_s3
    )
)

df_train_s3 = df_s3.iloc[
    train_idx_s3
].copy()

df_test_s3 = df_s3.iloc[
    test_idx_s3
].copy()

# ------------------------------------------------------------
# Verify group separation
# ------------------------------------------------------------

train_groups_s3 = set(
    df_train_s3["temporal_group_s3"]
)

test_groups_s3 = set(
    df_test_s3["temporal_group_s3"]
)

overlap_s3 = (
    train_groups_s3 &
    test_groups_s3
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("STAGE 3 TEMPORAL SPLIT")
print("=" * 90)

print("\nObservations")
print("----------------------------")
print(
    f"Training observations : {len(df_train_s3)}"
)
print(
    f"Test observations     : {len(df_test_s3)}"
)

print("\nTemporal groups")
print("----------------------------")
print(
    f"Training groups       : {len(train_groups_s3)}"
)
print(
    f"Test groups           : {len(test_groups_s3)}"
)

print("\nTemporal group overlap")
print("----------------------------")
print(
    f"Overlapping groups    : {len(overlap_s3)}"
)

if len(overlap_s3) == 0:
    print("✓ ZERO TEMPORAL GROUP OVERLAP")
else:
    print("⚠ WARNING: temporal group overlap detected")

# ------------------------------------------------------------
# Location coverage
# ------------------------------------------------------------

train_locations_s3 = set(
    df_train_s3[LOCATION_COL_S3].unique()
)

test_locations_s3 = set(
    df_test_s3[LOCATION_COL_S3].unique()
)

location_overlap_s3 = (
    train_locations_s3 &
    test_locations_s3
)

print("\nLocation coverage")
print("----------------------------")
print(
    f"Training locations : {len(train_locations_s3)}"
)
print(
    f"Test locations     : {len(test_locations_s3)}"
)
print(
    f"Location overlap   : {len(location_overlap_s3)}"
)

print("\nFirst training timestamp:")
print(
    df_train_s3[DATE_COL_S3].min()
)

print("Last training timestamp:")
print(
    df_train_s3[DATE_COL_S3].max()
)

print("\nFirst test timestamp:")
print(
    df_test_s3[DATE_COL_S3].min()
)

print("Last test timestamp:")
print(
    df_test_s3[DATE_COL_S3].max()
)

print("\n✓ Stage 3 temporal grouping preparation complete.")

In [ ]:
# ============================================================
# STAGE 3 — CELL 2
# STRICT CHRONOLOGICAL TEMPORAL SPLIT
# ============================================================

# ------------------------------------------------------------
# Get temporal groups in chronological order
# ------------------------------------------------------------

temporal_groups_ordered_s3 = sorted(
    df_s3["temporal_group_s3"].unique()
)

n_groups_s3 = len(
    temporal_groups_ordered_s3
)

n_train_groups_s3 = int(
    np.floor(0.80 * n_groups_s3)
)

train_temporal_groups_s3 = set(
    temporal_groups_ordered_s3[
        :n_train_groups_s3
    ]
)

test_temporal_groups_s3 = set(
    temporal_groups_ordered_s3[
        n_train_groups_s3:
    ]
)

# ------------------------------------------------------------
# Create chronological datasets
# ------------------------------------------------------------

df_train_s3 = df_s3[
    df_s3["temporal_group_s3"].isin(
        train_temporal_groups_s3
    )
].copy()

df_test_s3 = df_s3[
    df_s3["temporal_group_s3"].isin(
        test_temporal_groups_s3
    )
].copy()

# ------------------------------------------------------------
# Sort within each split
# ------------------------------------------------------------

df_train_s3 = df_train_s3.sort_values(
    DATE_COL_S3
).reset_index(drop=True)

df_test_s3 = df_test_s3.sort_values(
    DATE_COL_S3
).reset_index(drop=True)

# ------------------------------------------------------------
# Verify temporal separation
# ------------------------------------------------------------

train_groups_s3 = set(
    df_train_s3["temporal_group_s3"]
)

test_groups_s3 = set(
    df_test_s3["temporal_group_s3"]
)

temporal_overlap_s3 = (
    train_groups_s3 &
    test_groups_s3
)

# ------------------------------------------------------------
# Verify chronological ordering
# ------------------------------------------------------------

train_end_s3 = df_train_s3[
    DATE_COL_S3
].max()

test_start_s3 = df_test_s3[
    DATE_COL_S3
].min()

chronological_gap_s3 = (
    test_start_s3 - train_end_s3
)

# ------------------------------------------------------------
# Location coverage
# ------------------------------------------------------------

train_locations_s3 = set(
    df_train_s3[LOCATION_COL_S3].unique()
)

test_locations_s3 = set(
    df_test_s3[LOCATION_COL_S3].unique()
)

location_overlap_s3 = (
    train_locations_s3 &
    test_locations_s3
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 90)
print("STAGE 3 — STRICT CHRONOLOGICAL TEMPORAL SPLIT")
print("=" * 90)

print("\nTemporal groups")
print("----------------------------")
print(
    f"Total groups          : {n_groups_s3}"
)
print(
    f"Training groups       : {len(train_temporal_groups_s3)}"
)
print(
    f"Future test groups    : {len(test_temporal_groups_s3)}"
)

print("\nObservations")
print("----------------------------")
print(
    f"Training observations : {len(df_train_s3)}"
)
print(
    f"Future test observations : {len(df_test_s3)}"
)

print("\nTemporal separation")
print("----------------------------")
print(
    f"Training end         : {train_end_s3}"
)
print(
    f"Test start           : {test_start_s3}"
)
print(
    f"Temporal gap         : {chronological_gap_s3}"
)

print(
    f"\nTemporal group overlap: "
    f"{len(temporal_overlap_s3)}"
)

if len(temporal_overlap_s3) == 0:
    print("✓ ZERO TEMPORAL GROUP OVERLAP")
else:
    print("⚠ WARNING: temporal group overlap detected")

# ------------------------------------------------------------
# Location overlap
# ------------------------------------------------------------

print("\nLocation coverage")
print("----------------------------")
print(
    f"Training locations : {len(train_locations_s3)}"
)
print(
    f"Future test locations : {len(test_locations_s3)}"
)
print(
    f"Location overlap   : {len(location_overlap_s3)}"
)

if len(location_overlap_s3) > 0:
    print(
        "✓ Expected: temporal evaluation permits "
        "previously observed physical locations."
    )
else:
    print(
        "✓ No location overlap — this is also "
        "an unseen-location evaluation."
    )

# ------------------------------------------------------------
# Temporal ranges
# ------------------------------------------------------------

print("\nTraining temporal range")
print("----------------------------")
print(
    df_train_s3[DATE_COL_S3].min()
)
print(
    "to"
)
print(
    df_train_s3[DATE_COL_S3].max()
)

print("\nFuture test temporal range")
print("----------------------------")
print(
    df_test_s3[DATE_COL_S3].min()
)
print(
    "to"
)
print(
    df_test_s3[DATE_COL_S3].max()
)

print("\n✓ Strict chronological Stage 3 split prepared.")

In [ ]:
# ============================================================
# STAGE 3 — CELL 3
# FEATURE PREPARATION FOR FUTURE-TIME EVALUATION
# ============================================================

from sklearn.preprocessing import MinMaxScaler

# ------------------------------------------------------------
# RSSI features
# ------------------------------------------------------------

RSSI_COLS_S3 = [
    "b3001", "b3002", "b3003", "b3004",
    "b3005", "b3006", "b3007", "b3008",
    "b3009", "b3010", "b3011", "b3012",
    "b3013"
]

# ------------------------------------------------------------
# Extract RSSI features
# ------------------------------------------------------------

X_train_s3 = (
    df_train_s3[RSSI_COLS_S3]
    .astype(float)
    .values
)

X_test_s3 = (
    df_test_s3[RSSI_COLS_S3]
    .astype(float)
    .values
)

# ------------------------------------------------------------
# Convert location labels to coordinates
#
# IMPORTANT:
# Reuse the existing location_to_grid() function.
# ------------------------------------------------------------

y_train_s3 = np.vstack(
    df_train_s3[LOCATION_COL_S3]
    .apply(location_to_grid)
    .values
)

y_test_s3 = np.vstack(
    df_test_s3[LOCATION_COL_S3]
    .apply(location_to_grid)
    .values
)

# ------------------------------------------------------------
# Train-only scaling
# ------------------------------------------------------------

scaler_s3 = MinMaxScaler()

X_train_s3_scaled = scaler_s3.fit_transform(
    X_train_s3
)

X_test_s3_scaled = scaler_s3.transform(
    X_test_s3
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("=" * 90)
print("STAGE 3 — FEATURE PREPARATION")
print("=" * 90)

print("\nFeature shapes")
print("----------------------------")
print(
    f"X_train : {X_train_s3.shape}"
)
print(
    f"X_test  : {X_test_s3.shape}"
)

print("\nTarget shapes")
print("----------------------------")
print(
    f"y_train : {y_train_s3.shape}"
)
print(
    f"y_test  : {y_test_s3.shape}"
)

print("\nCoordinate ranges")
print("----------------------------")
print(
    f"Train X: {y_train_s3[:,0].min():.1f} "
    f"to {y_train_s3[:,0].max():.1f}"
)
print(
    f"Train Y: {y_train_s3[:,1].min():.1f} "
    f"to {y_train_s3[:,1].max():.1f}"
)

print("\nTest coordinate ranges")
print("----------------------------")
print(
    f"Test X: {y_test_s3[:,0].min():.1f} "
    f"to {y_test_s3[:,0].max():.1f}"
)
print(
    f"Test Y: {y_test_s3[:,1].min():.1f} "
    f"to {y_test_s3[:,1].max():.1f}"
)

print("\nScaler")
print("----------------------------")
print(
    "✓ Fitted ONLY on chronological training observations"
)

print("\nScaled ranges")
print("----------------------------")
print(
    f"Train: {X_train_s3_scaled.min():.4f} "
    f"to {X_train_s3_scaled.max():.4f}"
)
print(
    f"Future Test: {X_test_s3_scaled.min():.4f} "
    f"to {X_test_s3_scaled.max():.4f}"
)

print("\n✓ Stage 3 feature preparation complete.")

In [ ]:
# ============================================================
# STAGE 3 — CELL 4
# CONVENTIONAL MODELS — FUTURE-TIME EVALUATION
# ============================================================

from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR

# ------------------------------------------------------------
# Metric function
# ------------------------------------------------------------

def calculate_metrics_s3(y_true, y_pred):

    errors = np.sqrt(
        np.sum(
            (y_true - y_pred) ** 2,
            axis=1
        )
    )

    return {
        "MLE": np.mean(errors),
        "Median": np.median(errors),
        "Std": np.std(errors),
        "75th": np.percentile(errors, 75),
        "90th": np.percentile(errors, 90),
        "95th": np.percentile(errors, 95),
        "Max": np.max(errors),

        "MAE_X": np.mean(
            np.abs(
                y_true[:, 0] - y_pred[:, 0]
            )
        ),

        "MAE_Y": np.mean(
            np.abs(
                y_true[:, 1] - y_pred[:, 1]
            )
        ),

        "RMSE_X": np.sqrt(
            np.mean(
                (
                    y_true[:, 0] -
                    y_pred[:, 0]
                ) ** 2
            )
        ),

        "RMSE_Y": np.sqrt(
            np.mean(
                (
                    y_true[:, 1] -
                    y_pred[:, 1]
                ) ** 2
            )
        )
    }


# ------------------------------------------------------------
# Containers
# ------------------------------------------------------------

stage3_predictions = {}
stage3_results = []

# ============================================================
# kNN
# ============================================================

print("\nTraining kNN...")

knn_s3 = KNeighborsRegressor(
    n_neighbors=5,
    weights="distance",
    metric="euclidean"
)

knn_s3.fit(
    X_train_s3_scaled,
    y_train_s3
)

knn_pred_s3 = knn_s3.predict(
    X_test_s3_scaled
)

knn_metrics_s3 = calculate_metrics_s3(
    y_test_s3,
    knn_pred_s3
)

stage3_predictions["kNN"] = knn_pred_s3

stage3_results.append({
    "Model": "kNN",
    **knn_metrics_s3
})

print(
    f"Future Test MLE : "
    f"{knn_metrics_s3['MLE']:.4f} grid-coordinate units"
)


# ============================================================
# Decision Tree
# ============================================================

print("\nTraining Decision Tree...")

dt_s3 = DecisionTreeRegressor(
    max_depth=10,
    min_samples_leaf=2,
    random_state=42
)

dt_s3.fit(
    X_train_s3_scaled,
    y_train_s3
)

dt_pred_s3 = dt_s3.predict(
    X_test_s3_scaled
)

dt_metrics_s3 = calculate_metrics_s3(
    y_test_s3,
    dt_pred_s3
)

stage3_predictions["Decision Tree"] = dt_pred_s3

stage3_results.append({
    "Model": "Decision Tree",
    **dt_metrics_s3
})

print(
    f"Future Test MLE : "
    f"{dt_metrics_s3['MLE']:.4f} grid-coordinate units"
)


# ============================================================
# Random Forest
# ============================================================

print("\nTraining Random Forest...")

rf_s3 = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

rf_s3.fit(
    X_train_s3_scaled,
    y_train_s3
)

rf_pred_s3 = rf_s3.predict(
    X_test_s3_scaled
)

rf_metrics_s3 = calculate_metrics_s3(
    y_test_s3,
    rf_pred_s3
)

stage3_predictions["Random Forest"] = rf_pred_s3

stage3_results.append({
    "Model": "Random Forest",
    **rf_metrics_s3
})

print(
    f"Future Test MLE : "
    f"{rf_metrics_s3['MLE']:.4f} grid-coordinate units"
)


# ============================================================
# SVR
# ============================================================

print("\nTraining SVR...")

svr_s3 = MultiOutputRegressor(
    SVR(
        kernel="rbf",
        C=100,
        gamma="scale",
        epsilon=0.10
    )
)

svr_s3.fit(
    X_train_s3_scaled,
    y_train_s3
)

svr_pred_s3 = svr_s3.predict(
    X_test_s3_scaled
)

svr_metrics_s3 = calculate_metrics_s3(
    y_test_s3,
    svr_pred_s3
)

stage3_predictions["SVR"] = svr_pred_s3

stage3_results.append({
    "Model": "SVR",
    **svr_metrics_s3
})

print(
    f"Future Test MLE : "
    f"{svr_metrics_s3['MLE']:.4f} grid-coordinate units"
)


# ============================================================
# Results table
# ============================================================

stage3_conventional_results = pd.DataFrame(
    stage3_results
)

stage3_conventional_results = (
    stage3_conventional_results
    .sort_values("MLE")
    .reset_index(drop=True)
)

print("\n" + "=" * 95)
print("STAGE 3 — FUTURE-TIME CONVENTIONAL RESULTS")
print("=" * 95)

display(
    stage3_conventional_results.round(4)
)

# ------------------------------------------------------------
# Save immediately
# ------------------------------------------------------------

stage3_conventional_results.to_csv(
    "stage3_conventional_results.csv",
    index=False
)

print(
    "\n✓ Saved: "
    "stage3_conventional_results.csv"
)

Stage 3

In [ ]:
# ============================================================
# STAGE 3 — CELL 5
# NEURAL MODELS — FUTURE-TIME EVALUATION
# ============================================================

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    Conv1D,
    BatchNormalization,
    Flatten,
    GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.optimizers import Adam

# ------------------------------------------------------------
# Clear previous Stage 2 neural graph
# ------------------------------------------------------------

tf.keras.backend.clear_session()

np.random.seed(42)
tf.random.set_seed(42)

# ------------------------------------------------------------
# Common settings
# ------------------------------------------------------------

BATCH_SIZE_S3 = 32
EPOCHS_S3 = 300

# ============================================================
# MLP
# ============================================================

mlp_s3 = Sequential([
    Input(shape=(13,)),

    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.30),

    Dense(64, activation="relu"),
    Dropout(0.20),

    Dense(32, activation="relu"),

    Dense(2, activation="linear")
])

mlp_s3.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse"
)

print("\nTraining MLP...")

history_mlp_s3 = mlp_s3.fit(
    X_train_s3_scaled,
    y_train_s3,
    validation_split=0.20,
    epochs=EPOCHS_S3,
    batch_size=BATCH_SIZE_S3,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=25,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=10,
            min_lr=1e-6,
            verbose=1
        )
    ],
    verbose=1
)

mlp_pred_s3 = mlp_s3.predict(
    X_test_s3_scaled,
    verbose=0
)

mlp_metrics_s3 = calculate_metrics_s3(
    y_test_s3,
    mlp_pred_s3
)

stage3_predictions["MLP"] = mlp_pred_s3

print(
    f"\nMLP Future Test MLE : "
    f"{mlp_metrics_s3['MLE']:.4f} grid-coordinate units"
)


# ============================================================
# CNN-GAP
# ============================================================

tf.keras.backend.clear_session()

cnn_gap_s3 = Sequential([
    Input(shape=(13, 1)),

    Conv1D(
        32,
        kernel_size=3,
        padding="same",
        activation="relu"
    ),
    BatchNormalization(),

    Conv1D(
        64,
        kernel_size=3,
        padding="same",
        activation="relu"
    ),
    BatchNormalization(),

    GlobalAveragePooling1D(),

    Dense(64, activation="relu"),
    Dropout(0.30),

    Dense(2, activation="linear")
])

cnn_gap_s3.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse"
)

print("\nTraining CNN-GAP...")

history_cnn_gap_s3 = cnn_gap_s3.fit(
    X_train_s3_scaled[..., np.newaxis],
    y_train_s3,
    validation_split=0.20,
    epochs=EPOCHS_S3,
    batch_size=BATCH_SIZE_S3,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=25,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=10,
            min_lr=1e-6,
            verbose=1
        )
    ],
    verbose=1
)

cnn_gap_pred_s3 = cnn_gap_s3.predict(
    X_test_s3_scaled[..., np.newaxis],
    verbose=0
)

cnn_gap_metrics_s3 = calculate_metrics_s3(
    y_test_s3,
    cnn_gap_pred_s3
)

stage3_predictions["CNN-GAP"] = cnn_gap_pred_s3

print(
    f"\nCNN-GAP Future Test MLE : "
    f"{cnn_gap_metrics_s3['MLE']:.4f} grid-coordinate units"
)


# ============================================================
# CNN-FLAT
# ============================================================

tf.keras.backend.clear_session()

cnn_flat_s3 = Sequential([
    Input(shape=(13, 1)),

    Conv1D(
        32,
        kernel_size=3,
        padding="same",
        activation="relu"
    ),
    BatchNormalization(),

    Conv1D(
        64,
        kernel_size=3,
        padding="same",
        activation="relu"
    ),
    BatchNormalization(),

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.30),

    Dense(64, activation="relu"),
    Dropout(0.20),

    Dense(2, activation="linear")
])

cnn_flat_s3.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse"
)

print("\nTraining CNN-FLAT...")

history_cnn_flat_s3 = cnn_flat_s3.fit(
    X_train_s3_scaled[..., np.newaxis],
    y_train_s3,
    validation_split=0.20,
    epochs=EPOCHS_S3,
    batch_size=BATCH_SIZE_S3,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=25,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=10,
            min_lr=1e-6,
            verbose=1
        )
    ],
    verbose=1
)

cnn_flat_pred_s3 = cnn_flat_s3.predict(
    X_test_s3_scaled[..., np.newaxis],
    verbose=0
)

cnn_flat_metrics_s3 = calculate_metrics_s3(
    y_test_s3,
    cnn_flat_pred_s3
)

stage3_predictions["CNN-FLAT"] = cnn_flat_pred_s3

print(
    f"\nCNN-FLAT Future Test MLE : "
    f"{cnn_flat_metrics_s3['MLE']:.4f} grid-coordinate units"
)


# ============================================================
# Neural results summary
# ============================================================

stage3_neural_results = pd.DataFrame([
    {
        "Model": "MLP",
        **mlp_metrics_s3
    },
    {
        "Model": "CNN-GAP",
        **cnn_gap_metrics_s3
    },
    {
        "Model": "CNN-FLAT",
        **cnn_flat_metrics_s3
    }
]).sort_values(
    "MLE"
).reset_index(drop=True)

print("\n" + "=" * 95)
print("STAGE 3 — FUTURE-TIME NEURAL RESULTS")
print("=" * 95)

display(
    stage3_neural_results.round(4)
)

# ------------------------------------------------------------
# Save immediately
# ------------------------------------------------------------

stage3_neural_results.to_csv(
    "stage3_neural_results.csv",
    index=False
)

print(
    "\n✓ Saved: stage3_neural_results.csv"
)

In [ ]:
# ============================================================
# STAGE 3 — COMPLETE BACKUP
# ============================================================

import os
import json
import pickle
import shutil
import zipfile
import numpy as np
import pandas as pd

STAGE3_BACKUP_DIR = "/content/STAGE3_COMPLETE_BACKUP"
os.makedirs(STAGE3_BACKUP_DIR, exist_ok=True)

print("Creating Stage 3 complete backup...\n")

# ------------------------------------------------------------
# 1. Save result tables
# ------------------------------------------------------------

stage3_conventional_results.to_csv(
    f"{STAGE3_BACKUP_DIR}/stage3_conventional_results.csv",
    index=False
)

stage3_neural_results.to_csv(
    f"{STAGE3_BACKUP_DIR}/stage3_neural_results.csv",
    index=False
)

# ------------------------------------------------------------
# 2. Save predictions
# ------------------------------------------------------------

for model_name, predictions in stage3_predictions.items():

    safe_name = model_name.lower().replace("-", "_")

    np.save(
        f"{STAGE3_BACKUP_DIR}/predictions_{safe_name}.npy",
        predictions
    )

# ------------------------------------------------------------
# 3. Save true coordinates
# ------------------------------------------------------------

np.save(
    f"{STAGE3_BACKUP_DIR}/y_test_s3.npy",
    y_test_s3
)

np.save(
    f"{STAGE3_BACKUP_DIR}/y_train_s3.npy",
    y_train_s3
)

# ------------------------------------------------------------
# 4. Save scaled and original features
# ------------------------------------------------------------

np.save(
    f"{STAGE3_BACKUP_DIR}/X_train_s3_scaled.npy",
    X_train_s3_scaled
)

np.save(
    f"{STAGE3_BACKUP_DIR}/X_test_s3_scaled.npy",
    X_test_s3_scaled
)

np.save(
    f"{STAGE3_BACKUP_DIR}/X_train_s3.npy",
    X_train_s3
)

np.save(
    f"{STAGE3_BACKUP_DIR}/X_test_s3.npy",
    X_test_s3
)

# ------------------------------------------------------------
# 5. Save scaler
# ------------------------------------------------------------

with open(
    f"{STAGE3_BACKUP_DIR}/stage3_scaler.pkl",
    "wb"
) as f:

    pickle.dump(
        scaler_s3,
        f
    )

# ------------------------------------------------------------
# 6. Save neural models
# ------------------------------------------------------------

mlp_s3.save(
    f"{STAGE3_BACKUP_DIR}/mlp_s3.keras"
)

cnn_gap_s3.save(
    f"{STAGE3_BACKUP_DIR}/cnn_gap_s3.keras"
)

cnn_flat_s3.save(
    f"{STAGE3_BACKUP_DIR}/cnn_flat_s3.keras"
)

# ------------------------------------------------------------
# 7. Save training histories
# ------------------------------------------------------------

with open(
    f"{STAGE3_BACKUP_DIR}/history_mlp_s3.pkl",
    "wb"
) as f:

    pickle.dump(
        history_mlp_s3.history,
        f
    )

with open(
    f"{STAGE3_BACKUP_DIR}/history_cnn_gap_s3.pkl",
    "wb"
) as f:

    pickle.dump(
        history_cnn_gap_s3.history,
        f
    )

with open(
    f"{STAGE3_BACKUP_DIR}/history_cnn_flat_s3.pkl",
    "wb"
) as f:

    pickle.dump(
        history_cnn_flat_s3.history,
        f
    )

# ------------------------------------------------------------
# 8. Save temporal split metadata
# ------------------------------------------------------------

stage3_metadata = {
    "protocol": "Stage 3 - Future-Time Generalization",

    "total_observations": 1420,

    "train_observations": int(len(X_train_s3)),
    "future_test_observations": int(len(X_test_s3)),

    "train_temporal_groups": 35,
    "future_test_temporal_groups": 9,

    "temporal_overlap": 0,

    "train_start": "2016-04-19 09:37:23",
    "train_end": "2016-10-04 10:54:43",

    "future_test_start": "2016-10-04 10:55:02",
    "future_test_end": "2016-10-18 11:15:21",

    "gap_seconds": 19,

    "train_locations": 104,
    "future_test_locations": 97,

    "location_overlap": 96,

    "scaling": "MinMaxScaler fitted only on training observations",

    "random_state": 42
}

with open(
    f"{STAGE3_BACKUP_DIR}/stage3_metadata.json",
    "w"
) as f:

    json.dump(
        stage3_metadata,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 9. Save per-observation error tables
# ------------------------------------------------------------

for model_name, predictions in stage3_predictions.items():

    safe_name = model_name.lower().replace("-", "_")

    errors = np.sqrt(
        np.sum(
            (y_test_s3 - predictions) ** 2,
            axis=1
        )
    )

    error_df = pd.DataFrame({
        "True_X": y_test_s3[:, 0],
        "True_Y": y_test_s3[:, 1],
        "Pred_X": predictions[:, 0],
        "Pred_Y": predictions[:, 1],
        "Error_grid_units": errors
    })

    error_df.to_csv(
        f"{STAGE3_BACKUP_DIR}/errors_{safe_name}.csv",
        index=False
    )

# ------------------------------------------------------------
# 10. Create ZIP
# ------------------------------------------------------------

zip_path = "/content/STAGE3_COMPLETE_BACKUP.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(
    "/content/STAGE3_COMPLETE_BACKUP",
    "zip",
    STAGE3_BACKUP_DIR
)

# ------------------------------------------------------------
# 11. Verify
# ------------------------------------------------------------

files_saved = os.listdir(STAGE3_BACKUP_DIR)

print("=" * 70)
print("STAGE 3 BACKUP COMPLETE")
print("=" * 70)

print(f"Files saved: {len(files_saved)}")

for f in sorted(files_saved):
    print("✓", f)

print("\nZIP:")
print(zip_path)

print(
    f"\nZIP size: "
    f"{os.path.getsize(zip_path) / (1024**2):.2f} MB"
)

print("\n✓ Stage 3 experiment is safely backed up.")

Download file

In [ ]:
# ============================================================
# MASTER BACKUP — CELL 1
# INVENTORY ALL IMPORTANT EXPERIMENT FILES
# ============================================================

import os

print("=" * 90)
print("IMPORTANT FILES CURRENTLY AVAILABLE IN /content")
print("=" * 90)

# Extensions that are potentially important for our experiments
important_extensions = (
    ".csv",
    ".npy",
    ".npz",
    ".pkl",
    ".pickle",
    ".keras",
    ".h5",
    ".json",
    ".xlsx",
    ".zip"
)

# Keywords associated with our experiments
keywords = [
    "stage1",
    "stage2",
    "stage3",
    "prediction",
    "predictions",
    "result",
    "results",
    "error",
    "errors",
    "scaler",
    "history",
    "split",
    "location",
    "svr",
    "knn",
    "forest",
    "tree",
    "mlp",
    "cnn",
    "bootstrap",
    "wilcoxon",
    "tuning"
]

important_files = []

for root, dirs, files in os.walk("/content"):

    # Ignore system/cache directories
    dirs[:] = [
        d for d in dirs
        if d not in [
            ".config",
            ".cache",
            "sample_data",
            "__pycache__"
        ]
    ]

    for file in files:

        path = os.path.join(root, file)
        lower = file.lower()

        if lower.endswith(important_extensions):

            if any(k in lower for k in keywords):
                important_files.append(path)

# Remove duplicates
important_files = sorted(set(important_files))

print(f"\nFound {len(important_files)} potentially important files.\n")

for i, path in enumerate(important_files, 1):

    size_mb = os.path.getsize(path) / (1024 ** 2)

    print(
        f"{i:3d}. "
        f"{size_mb:8.2f} MB   "
        f"{path}"
    )

print("\n" + "=" * 90)
print("INVENTORY COMPLETE")
print("=" * 90)

In [ ]:
# ============================================================
# MASTER RESEARCH BACKUP
# Stage 1 + Stage 2 + Stage 3
# ============================================================

import os
import shutil
import zipfile
from pathlib import Path

SOURCE = Path("/content")
BACKUP_DIR = SOURCE / "BEACON_RESEARCH_MASTER_BACKUP"

# Remove old master backup if it exists
if BACKUP_DIR.exists():
    shutil.rmtree(BACKUP_DIR)

BACKUP_DIR.mkdir(parents=True)

print("=" * 90)
print("CREATING MASTER RESEARCH BACKUP")
print("=" * 90)


# ============================================================
# 1. CREATE ORGANIZED FOLDERS
# ============================================================

stage1_dir = BACKUP_DIR / "STAGE1_SEEN_LOCATION"
stage2_dir = BACKUP_DIR / "STAGE2_UNSEEN_LOCATION"
stage3_dir = BACKUP_DIR / "STAGE3_FUTURE_TIME"
summary_dir = BACKUP_DIR / "MASTER_RESULTS"

for folder in [
    stage1_dir,
    stage2_dir,
    stage3_dir,
    summary_dir
]:
    folder.mkdir(parents=True)


# ============================================================
# 2. STAGE 1 FILES
# ============================================================

stage1_files = [
    "all_model_results_stage1.csv",
    "final_conventional_with_svr.csv",
    "final_decision_tree.pkl",
    "final_knn.pkl",
    "final_minmax_scaler.pkl",
    "final_random_forest.pkl",
    "final_svr.pkl",
    "final_cnn_flat.keras",
    "final_cnn_gap.keras",
    "final_mlp.keras",
    "location_aware_split_seed42.csv",
    "mle_bootstrap_95CI.csv",
    "per_location_error_best_model.csv",
    "svr_validation_tuning.csv",
    "test_predictions_all_models.csv"
]

print("\nSTAGE 1 FILES")

for filename in stage1_files:

    source = SOURCE / filename

    if source.exists():

        shutil.copy2(
            source,
            stage1_dir / filename
        )

        print("✓", filename)

    else:

        print("⚠️ MISSING:", filename)


# ============================================================
# 3. STAGE 2 COMPLETE BACKUP
# ============================================================

stage2_zip = SOURCE / "STAGE2_COMPLETE_BACKUP.zip"

if stage2_zip.exists():

    shutil.copy2(
        stage2_zip,
        stage2_dir / "STAGE2_COMPLETE_BACKUP.zip"
    )

    print("\n✓ Stage 2 complete backup copied")

else:

    print("\n⚠️ STAGE2_COMPLETE_BACKUP.zip not found")


# ============================================================
# 4. STAGE 3 COMPLETE BACKUP
# ============================================================

stage3_zip = SOURCE / "STAGE3_COMPLETE_BACKUP.zip"

if stage3_zip.exists():

    shutil.copy2(
        stage3_zip,
        stage3_dir / "STAGE3_COMPLETE_BACKUP.zip"
    )

    print("✓ Stage 3 complete backup copied")

else:

    print("⚠️ STAGE3_COMPLETE_BACKUP.zip not found")


# ============================================================
# 5. COPY IMPORTANT MASTER RESULT TABLES
# ============================================================

master_result_files = [
    "all_model_results_stage1.csv",
    "stage2_master_generalization_comparison.csv",
    "stage2_svr_statistical_comparison.csv",
    "stage2_unseen_error_distribution.csv",
    "stage3_conventional_results.csv",
    "stage3_neural_results.csv",
    "svr_validation_tuning.csv",
    "mle_bootstrap_95CI.csv",
    "location_aware_split_seed42.csv",
    "per_location_error_best_model.csv",
    "test_predictions_all_models.csv"
]

print("\nMASTER RESULT TABLES")

for filename in master_result_files:

    source = SOURCE / filename

    if source.exists():

        shutil.copy2(
            source,
            summary_dir / filename
        )

        print("✓", filename)

    else:

        print("⚠️ MISSING:", filename)


# ============================================================
# 6. CREATE README
# ============================================================

readme = """
BEACON RESEARCH MASTER BACKUP
=============================

Research:
BLE-Based Indoor Localization in Smart Healthcare Environments

Dataset:
Public BLE-RSSI indoor localization dataset
Waldo Library, Western Michigan University

EXPERIMENTAL PROTOCOLS
----------------------

Stage 1:
Seen-location / observation-level evaluation

Stage 2:
Unseen-location generalization

Stage 3:
Future-time / temporal generalization

Stage 4:
RSSI noise robustness
NOT YET COMPLETED AT THE TIME OF THIS BACKUP

FOLDERS
-------

STAGE1_SEEN_LOCATION
    Stage 1 trained models, scalers, predictions,
    result tables and evaluation files.

STAGE2_UNSEEN_LOCATION
    Complete Stage 2 backup ZIP.

STAGE3_FUTURE_TIME
    Complete Stage 3 backup ZIP.

MASTER_RESULTS
    Important consolidated result tables.

IMPORTANT
---------

Do not modify Stage 1, Stage 2 or Stage 3 results.
These are the completed experimental baselines.

Stage 4 should be performed as a separate robustness experiment.

Stage 1 best model:
SVR — MLE 1.6079 m

Stage 2 best model:
SVR — MLE 1.8134 m

Stage 3 best model:
kNN — MLE 1.4706 m
Random Forest — MLE 1.4707 m

Backup created from Google Colab /content.
"""

with open(
    BACKUP_DIR / "README.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(readme)

print("\n✓ README created")


# ============================================================
# 7. CREATE ZIP
# ============================================================

master_zip_base = SOURCE / "BEACON_RESEARCH_MASTER_BACKUP"

# Remove previous ZIP
old_zip = Path(str(master_zip_base) + ".zip")

if old_zip.exists():
    old_zip.unlink()

shutil.make_archive(
    str(master_zip_base),
    "zip",
    BACKUP_DIR
)

master_zip = Path(str(master_zip_base) + ".zip")


# ============================================================
# 8. INVENTORY
# ============================================================

all_files = list(BACKUP_DIR.rglob("*"))

files_only = [
    f for f in all_files
    if f.is_file()
]

total_size = sum(
    f.stat().st_size
    for f in files_only
)

print("\n" + "=" * 90)
print("MASTER BACKUP COMPLETE")
print("=" * 90)

print(f"Files included : {len(files_only)}")
print(
    f"Uncompressed   : "
    f"{total_size / (1024**2):.2f} MB"
)

print(
    f"ZIP size       : "
    f"{master_zip.stat().st_size / (1024**2):.2f} MB"
)

print("\nBackup file:")
print(master_zip)

print("\nFolder structure:")

for f in files_only:
    print(
        "✓",
        f.relative_to(BACKUP_DIR)
    )

print("\n" + "=" * 90)
print("READY FOR DOWNLOAD")
print("=" * 90)

In [ ]:
# ============================================================
# STAGE 4 — CELL 1
# VERIFY STAGE 1 MODELS AND TEST DATA
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 90)
print("STAGE 4 — RSSI NOISE ROBUSTNESS")
print("CELL 1 — VERIFY STAGE 1 RESOURCES")
print("=" * 90)

# ------------------------------------------------------------
# Check required variables
# ------------------------------------------------------------

required_variables = [
    "X_test",
    "y_test",
    "X_test_scaled",
    "final_knn",
    "final_decision_tree",
    "final_random_forest",
    "final_svr",
    "final_mlp",
    "final_cnn_gap",
    "final_cnn_flat"
]

print("\nChecking in-memory variables...\n")

for var in required_variables:

    if var in globals():

        obj = globals()[var]

        try:
            shape = obj.shape
            print(f"✓ {var:<25} shape = {shape}")

        except Exception:
            print(f"✓ {var:<25} available")

    else:

        print(f"✗ {var:<25} NOT FOUND")


# ------------------------------------------------------------
# Check Stage 1 files
# ------------------------------------------------------------

print("\n" + "-" * 90)
print("Checking saved Stage 1 files")
print("-" * 90)

stage1_files = [
    "/content/final_knn.pkl",
    "/content/final_decision_tree.pkl",
    "/content/final_random_forest.pkl",
    "/content/final_svr.pkl",
    "/content/final_mlp.keras",
    "/content/final_cnn_gap.keras",
    "/content/final_cnn_flat.keras",
    "/content/final_minmax_scaler.pkl",
    "/content/test_predictions_all_models.csv"
]

for filepath in stage1_files:

    if os.path.exists(filepath):

        size_mb = os.path.getsize(filepath) / (1024**2)

        print(
            f"✓ {os.path.basename(filepath):<35}"
            f"{size_mb:.2f} MB"
        )

    else:

        print(
            f"✗ {os.path.basename(filepath):<35}"
            f"NOT FOUND"
        )


# ------------------------------------------------------------
# Basic test-set validation
# ------------------------------------------------------------

if "X_test" in globals() and "y_test" in globals():

    print("\n" + "-" * 90)
    print("Stage 1 test-set dimensions")
    print("-" * 90)

    print("X_test:", X_test.shape)
    print("y_test:", y_test.shape)

    print(
        "\nExpected Stage 1 test observations: "
        "approximately 226"
    )

    print(
        "Number of RSSI features:",
        X_test.shape[1]
    )

    print(
        "Target dimensions:",
        y_test.shape[1]
    )

    # RSSI sanity check
    print("\nRSSI range:")

    print(
        "Minimum:",
        np.min(X_test)
    )

    print(
        "Maximum:",
        np.max(X_test)
    )

    print(
        "Mean:",
        np.mean(X_test)
    )

    print(
        "Number of -200 values:",
        np.sum(X_test == -200)
    )

    print(
        "Percentage of -200 values:",
        f"{100*np.mean(X_test == -200):.2f}%"
    )


print("\n" + "=" * 90)
print("VERIFICATION COMPLETE")
print("=" * 90)

In [ ]:
# ============================================================
# STAGE 4 — RSSI NOISE ROBUSTNESS
# CELL 1B — LOCATE AND DIAGNOSE SAVED CONVENTIONAL MODELS
# ============================================================

import os
import glob

print("=" * 80)
print("STAGE 4 — SAFE SERIALIZATION DIAGNOSTIC")
print("=" * 80)

target_files = [
    "final_knn.pkl",
    "final_decision_tree.pkl",
    "final_random_forest.pkl",
    "final_svr.pkl",
]

# ------------------------------------------------------------
# 1. Search common Colab locations
# ------------------------------------------------------------

search_roots = ["/content"]  # Add a user-controlled Drive path locally if required.

found_files = {}

for root in search_roots:

    if not os.path.exists(root):
        continue

    for fname in target_files:

        if fname in found_files:
            continue

        matches = glob.glob(
            os.path.join(root, "**", fname),
            recursive=True
        )

        if matches:
            # Use the first exact filename match
            found_files[fname] = matches[0]


# ------------------------------------------------------------
# 2. Display located files
# ------------------------------------------------------------

print("\nLOCATED FILES")
print("-" * 80)

for fname in target_files:

    if fname in found_files:

        path = found_files[fname]

        print(f"✓ {fname}")
        print(f"  Path: {path}")
        print(f"  Size: {os.path.getsize(path):,} bytes")

    else:

        print(f"❌ {fname} NOT FOUND")


# ------------------------------------------------------------
# 3. Inspect file headers
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FILE HEADER DIAGNOSTICS")
print("=" * 80)

for fname, path in found_files.items():

    print(f"\n{fname}")
    print("-" * 80)

    with open(path, "rb") as f:
        header = f.read(32)

    print("First 32 bytes:")
    print(header)

    print("\nHex:")
    print(header.hex())

    # --------------------------------------------------------
    # Format clues
    # --------------------------------------------------------

    if header.startswith(b"\x80"):
        print("\nPossible format: Python PICKLE")

    elif header.startswith(b"PK"):
        print("\nPossible format: ZIP / ZIP-based serialization")

    elif header.startswith(b"\x04"):
        print("\nPossible format: JOBLIB")
        print("Reason: file does not begin with a standard pickle")
        print("protocol header.")

    else:
        print("\nFormat requires further inspection.")

print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STAGE 4 — CELL 2A
# TEST JOBLIB LOADING OF ORIGINAL STAGE 1 MODELS
# ============================================================

import os
import joblib

MODEL_DIR = "/content"

model_paths = {
    "kNN": os.path.join(MODEL_DIR, "final_knn.pkl"),
    "Decision Tree": os.path.join(MODEL_DIR, "final_decision_tree.pkl"),
    "Random Forest": os.path.join(MODEL_DIR, "final_random_forest.pkl"),
    "SVR": os.path.join(MODEL_DIR, "final_svr.pkl"),
}

loaded_models = {}

print("=" * 80)
print("TESTING JOBLIB SERIALIZATION")
print("=" * 80)

for name, path in model_paths.items():

    print(f"\n{name}")
    print("-" * 80)

    try:
        model = joblib.load(path)

        loaded_models[name] = model

        print("✓ SUCCESS")
        print("Object type:", type(model))

    except Exception as e:

        print("❌ FAILED")
        print("Error type:", type(e).__name__)
        print("Error:", repr(e))

print("\n" + "=" * 80)
print("JOBLIB TEST COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STAGE 4 — CELL 3
# VERIFY ORIGINAL STAGE 1 PREDICTIONS
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

print("=" * 80)
print("STAGE 4 — ORIGINAL STAGE 1 PREDICTION VERIFICATION")
print("=" * 80)

MODEL_DIR = "/content"

# ------------------------------------------------------------
# Load models using the confirmed correct method
# ------------------------------------------------------------

final_knn = joblib.load(
    os.path.join(MODEL_DIR, "final_knn.pkl")
)

final_decision_tree = joblib.load(
    os.path.join(MODEL_DIR, "final_decision_tree.pkl")
)

final_random_forest = joblib.load(
    os.path.join(MODEL_DIR, "final_random_forest.pkl")
)

final_svr = joblib.load(
    os.path.join(MODEL_DIR, "final_svr.pkl")
)

final_minmax_scaler = joblib.load(
    os.path.join(MODEL_DIR, "final_minmax_scaler.pkl")
)

print("\n✓ All four conventional models loaded")
print("✓ Stage 1 scaler loaded")

# ------------------------------------------------------------
# Verify Stage 1 test data
# ------------------------------------------------------------

print("\nTest data:")
print("X_test        :", X_test.shape)
print("X_test_scaled :", X_test_scaled.shape)
print("y_test        :", y_test.shape)

assert X_test.shape == (226, 13)
assert X_test_scaled.shape == (226, 13)
assert y_test.shape == (226, 2)

# ------------------------------------------------------------
# Verify scaler consistency
#
# We DO NOT replace X_test_scaled.
# We only check whether applying the saved scaler to the
# original X_test gives the same scaled values.
# ------------------------------------------------------------

recreated_X_test_scaled = final_minmax_scaler.transform(X_test)

scaler_difference = np.max(
    np.abs(
        recreated_X_test_scaled - X_test_scaled
    )
)

print("\nScaler verification:")
print(
    f"Maximum difference = {scaler_difference:.12f}"
)

if scaler_difference < 1e-10:
    print("✓ Saved scaler reproduces X_test_scaled exactly")
else:
    print("⚠ Small scaler difference detected")

# ------------------------------------------------------------
# Generate predictions using ORIGINAL Stage 1 inputs
# ------------------------------------------------------------

print("\nGenerating original predictions...")

pred_knn = final_knn.predict(X_test_scaled)

pred_dt = final_decision_tree.predict(X_test_scaled)

pred_rf = final_random_forest.predict(X_test_scaled)

pred_svr = final_svr.predict(X_test_scaled)

# ------------------------------------------------------------
# Prediction dimensions
# ------------------------------------------------------------

print("\nPrediction shapes:")

print("kNN             :", pred_knn.shape)
print("Decision Tree   :", pred_dt.shape)
print("Random Forest   :", pred_rf.shape)
print("SVR             :", pred_svr.shape)

assert pred_knn.shape == (226, 2)
assert pred_dt.shape == (226, 2)
assert pred_rf.shape == (226, 2)
assert pred_svr.shape == (226, 2)

print("✓ All prediction dimensions correct")

# ------------------------------------------------------------
# Euclidean localization error
#
# For each observation:
#
# error = sqrt(
#     (x_true - x_pred)^2 +
#     (y_true - y_pred)^2
# )
#
# Stage 1 reported value is the RMSE of these distances.
# ------------------------------------------------------------

def calculate_localization_rmse(y_true, y_pred):

    distances = np.sqrt(
        np.sum(
            (y_true - y_pred) ** 2,
            axis=1
        )
    )

    return np.sqrt(
        np.mean(distances ** 2)
    )


rmse_knn = calculate_localization_rmse(
    y_test,
    pred_knn
)

rmse_dt = calculate_localization_rmse(
    y_test,
    pred_dt
)

rmse_rf = calculate_localization_rmse(
    y_test,
    pred_rf
)

rmse_svr = calculate_localization_rmse(
    y_test,
    pred_svr
)

# ------------------------------------------------------------
# Compare with completed Stage 1 values
# ------------------------------------------------------------

verification_table = pd.DataFrame({

    "Model": [
        "SVR",
        "kNN",
        "Random Forest",
        "Decision Tree"
    ],

    "Recovered_RMSE_grid_units": [
        rmse_svr,
        rmse_knn,
        rmse_rf,
        rmse_dt
    ],

    "Completed_Stage1_RMSE_grid_units": [
        1.6079,
        1.6279,
        1.6332,
        1.6755
    ]
})

verification_table["Absolute_Difference_grid_units"] = (
    verification_table["Recovered_RMSE_grid_units"]
    -
    verification_table["Completed_Stage1_RMSE_grid_units"]
).abs()

print("\n" + "=" * 80)
print("STAGE 1 RESULT VERIFICATION")
print("=" * 80)

display(
    verification_table.style.format({
        "Recovered_RMSE_grid_units": "{:.6f}",
        "Completed_Stage1_RMSE_grid_units": "{:.4f}",
        "Absolute_Difference_grid_units": "{:.6f}"
    })
)

# ------------------------------------------------------------
# Load existing prediction CSV
# ------------------------------------------------------------

prediction_csv = os.path.join(
    MODEL_DIR,
    "test_predictions_all_models.csv"
)

existing_predictions = pd.read_csv(
    prediction_csv
)

print("\n" + "=" * 80)
print("EXISTING PREDICTION CSV")
print("=" * 80)

print("Shape:", existing_predictions.shape)

print("\nColumns:")
print(existing_predictions.columns.tolist())

print("\nFirst five rows:")
display(existing_predictions.head())

print("\n" + "=" * 80)
print("ORIGINAL STAGE 1 VERIFICATION COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STAGE 4 — CELL 4
# EXACT STAGE 1 PREDICTION / METRIC VERIFICATION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("STAGE 4 — EXACT STAGE 1 VERIFICATION")
print("=" * 80)

# ------------------------------------------------------------
# Existing Stage 1 prediction CSV
# ------------------------------------------------------------

prediction_csv = "/content/test_predictions_all_models.csv"

existing_predictions = pd.read_csv(prediction_csv)

print("\nCSV shape:", existing_predictions.shape)

# ------------------------------------------------------------
# Recovered predictions from the saved models
# ------------------------------------------------------------

recovered_predictions = {
    "kNN": pred_knn,
    "Decision Tree": pred_dt,
    "Random Forest": pred_rf,
    "SVR": pred_svr
}

# ------------------------------------------------------------
# CSV column mapping
# ------------------------------------------------------------

csv_columns = {
    "kNN": ["knn_x", "knn_y", "knn_error"],
    "Decision Tree": [
        "decision_tree_x",
        "decision_tree_y",
        "decision_tree_error"
    ],
    "Random Forest": [
        "random_forest_x",
        "random_forest_y",
        "random_forest_error"
    ],
    "SVR": [
        "svr_x",
        "svr_y",
        "svr_error"
    ]
}

# ------------------------------------------------------------
# Compare predictions row-by-row
# ------------------------------------------------------------

verification_results = []

for model_name, predictions in recovered_predictions.items():

    x_col, y_col, error_col = csv_columns[model_name]

    csv_xy = existing_predictions[
        [x_col, y_col]
    ].values

    model_xy = predictions

    # Difference between recovered model and saved CSV
    coordinate_difference = np.abs(
        model_xy - csv_xy
    )

    max_coordinate_difference = np.max(
        coordinate_difference
    )

    mean_coordinate_difference = np.mean(
        coordinate_difference
    )

    # --------------------------------------------------------
    # Recalculate Euclidean error
    # --------------------------------------------------------

    true_xy = y_test

    recovered_errors = np.sqrt(
        np.sum(
            (true_xy - model_xy) ** 2,
            axis=1
        )
    )

    csv_errors = existing_predictions[
        error_col
    ].values

    max_error_difference = np.max(
        np.abs(
            recovered_errors - csv_errors
        )
    )

    # --------------------------------------------------------
    # IMPORTANT:
    # Stage 1 reported metric = MEAN Euclidean localization
    # error, not sqrt(mean(distance^2)).
    # --------------------------------------------------------

    mean_localization_error = np.mean(
        recovered_errors
    )

    csv_mean_error = np.mean(
        csv_errors
    )

    verification_results.append({

        "Model": model_name,

        "Mean_Localization_Error_grid_units":
            mean_localization_error,

        "CSV_Mean_Error_grid_units":
            csv_mean_error,

        "Max_Coordinate_Difference":
            max_coordinate_difference,

        "Mean_Coordinate_Difference":
            mean_coordinate_difference,

        "Max_Error_Difference":
            max_error_difference
    })


# ------------------------------------------------------------
# Display verification
# ------------------------------------------------------------

verification_df = pd.DataFrame(
    verification_results
)

print("\n" + "=" * 80)
print("EXACT PREDICTION VERIFICATION")
print("=" * 80)

display(
    verification_df.style.format({
        "Mean_Localization_Error_grid_units": "{:.6f}",
        "CSV_Mean_Error_grid_units": "{:.6f}",
        "Max_Coordinate_Difference": "{:.12f}",
        "Mean_Coordinate_Difference": "{:.12f}",
        "Max_Error_Difference": "{:.12f}"
    })
)

# ------------------------------------------------------------
# Compare against completed Stage 1 results
# ------------------------------------------------------------

completed_stage1 = {
    "SVR": 1.6079,
    "kNN": 1.6279,
    "Random Forest": 1.6332,
    "Decision Tree": 1.6755
}

verification_df["Completed_Stage1_grid_units"] = (
    verification_df["Model"]
    .map(completed_stage1)
)

verification_df["Difference_from_Stage1_grid_units"] = (
    verification_df["Mean_Localization_Error_grid_units"]
    -
    verification_df["Completed_Stage1_grid_units"]
).abs()

print("\n" + "=" * 80)
print("COMPARISON WITH COMPLETED STAGE 1")
print("=" * 80)

display(
    verification_df[
        [
            "Model",
            "Mean_Localization_Error_grid_units",
            "Completed_Stage1_grid_units",
            "Difference_from_Stage1_grid_units"
        ]
    ].style.format({
        "Mean_Localization_Error_grid_units": "{:.6f}",
        "Completed_Stage1_grid_units": "{:.4f}",
        "Difference_from_Stage1_grid_units": "{:.6f}"
    })
)

# ------------------------------------------------------------
# Final pass/fail checks
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL VERIFICATION STATUS")
print("=" * 80)

all_predictions_match = (
    verification_df["Max_Coordinate_Difference"].max()
    < 1e-10
)

all_errors_match = (
    verification_df["Max_Error_Difference"].max()
    < 1e-10
)

print(
    "Predictions match saved CSV:",
    "✓ YES" if all_predictions_match else "❌ NO"
)

print(
    "Per-observation errors match CSV:",
    "✓ YES" if all_errors_match else "❌ NO"
)

if all_predictions_match and all_errors_match:
    print("\n✓ ORIGINAL STAGE 1 PREDICTIONS SUCCESSFULLY VERIFIED")
    print("✓ SAVED MODELS ARE VALID")
    print("✓ STAGE 1 RESULTS HAVE NOT BEEN MODIFIED")
    print("✓ SAFE TO PROCEED TO STAGE 4 NOISE ROBUSTNESS")
else:
    print("\n⚠ VERIFICATION DID NOT PASS")
    print("DO NOT START NOISE ROBUSTNESS YET.")

print("=" * 80)

In [ ]:
# ============================================================
# STAGE 4 — RSSI NOISE ROBUSTNESS
# CELL 5 — CONTROLLED RSSI NOISE GENERATION
# CORRECTED FOR PANDAS DATAFRAME INPUT
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 80)
print("STAGE 4 — RSSI NOISE ROBUSTNESS")
print("CELL 5 — CONTROLLED NOISE GENERATION")
print("=" * 80)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

# ------------------------------------------------------------
# Noise levels
# ------------------------------------------------------------

NOISE_LEVELS = [2, 5, 8]

# ------------------------------------------------------------
# Convert ONLY for numerical noise processing
#
# IMPORTANT:
# Original X_test remains untouched.
# ------------------------------------------------------------

X_test_original = X_test.to_numpy(
    dtype=np.float64
).copy()

print("\nOriginal test shape:")
print(X_test_original.shape)

assert X_test_original.shape == (226, 13)

# ------------------------------------------------------------
# Identify detected RSSI values
#
# -200 = no detection
#
# Only values:
#       RSSI < 0
#       RSSI > -200
#
# receive noise.
# ------------------------------------------------------------

detected_mask = (
    (X_test_original < 0) &
    (X_test_original > -200)
)

no_detection_mask = (
    X_test_original == -200
)

print("\nRSSI statistics")
print("-" * 80)

total_values = int(X_test_original.size)

detected_count = int(
    np.count_nonzero(detected_mask)
)

no_detection_count = int(
    np.count_nonzero(no_detection_mask)
)

print(
    f"Total RSSI values       : {total_values:,}"
)

print(
    f"Detected RSSI values    : {detected_count:,}"
)

print(
    f"No-detection (-200)     : {no_detection_count:,}"
)

print(
    f"Detected percentage     : "
    f"{100.0 * detected_count / total_values:.2f}%"
)

print(
    f"-200 percentage         : "
    f"{100.0 * no_detection_count / total_values:.2f}%"
)

# ------------------------------------------------------------
# Safety assertion
# ------------------------------------------------------------

assert (
    detected_count + no_detection_count
    == total_values
)

print(
    "\n✓ Every RSSI value classified as "
    "detected or -200"
)

# ------------------------------------------------------------
# Generate noisy datasets
# ------------------------------------------------------------

noisy_test_sets = {}

noise_records = []

for level in NOISE_LEVELS:

    print("\n" + "-" * 80)
    print(f"Generating ±{level} dBm noise")
    print("-" * 80)

    # Start from exact original test data
    X_noisy = X_test_original.copy()

    # Generate bounded uniform noise
    noise = rng.uniform(
        low=-level,
        high=level,
        size=X_test_original.shape
    )

    # --------------------------------------------------------
    # CRITICAL:
    # Apply noise ONLY to detected RSSI values.
    # --------------------------------------------------------

    X_noisy[detected_mask] = (
        X_test_original[detected_mask]
        +
        noise[detected_mask]
    )

    # --------------------------------------------------------
    # Verify -200 values are completely unchanged
    # --------------------------------------------------------

    original_no_detection_values = (
        X_test_original[no_detection_mask]
    )

    noisy_no_detection_values = (
        X_noisy[no_detection_mask]
    )

    unchanged = np.array_equal(
        original_no_detection_values,
        noisy_no_detection_values
    )

    print(
        "Original -200 values preserved:",
        "✓ YES" if unchanged else "❌ NO"
    )

    assert unchanged

    # --------------------------------------------------------
    # Calculate actual applied noise
    # --------------------------------------------------------

    actual_noise = (
        X_noisy[detected_mask]
        -
        X_test_original[detected_mask]
    )

    minimum_noise = float(
        np.min(actual_noise)
    )

    maximum_noise = float(
        np.max(actual_noise)
    )

    mean_absolute_noise = float(
        np.mean(np.abs(actual_noise))
    )

    changed_detected = int(
        np.count_nonzero(actual_noise != 0)
    )

    print(
        f"Actual noise range      : "
        f"{minimum_noise:.6f} to "
        f"{maximum_noise:.6f} dBm"
    )

    print(
        f"Mean absolute noise    : "
        f"{mean_absolute_noise:.6f} dBm"
    )

    print(
        f"Detected values changed: "
        f"{changed_detected:,} / "
        f"{detected_count:,}"
    )

    # --------------------------------------------------------
    # Safety checks
    # --------------------------------------------------------

    assert minimum_noise >= -level
    assert maximum_noise <= level

    assert changed_detected == detected_count

    print("✓ Noise range valid")
    print("✓ All detected RSSI values perturbed")
    print("✓ All -200 values preserved")

    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    noisy_test_sets[level] = X_noisy

    noise_records.append({

        "Noise_Level_dBm": f"±{level}",

        "Total_RSSI": total_values,

        "Detected_RSSI": detected_count,

        "NoDetection_-200": no_detection_count,

        "Detected_Percentage":
            100.0 * detected_count / total_values,

        "NoDetection_Percentage":
            100.0 * no_detection_count / total_values,

        "Min_Noise":
            minimum_noise,

        "Max_Noise":
            maximum_noise,

        "Mean_Absolute_Noise":
            mean_absolute_noise,

        "Changed_Detections":
            changed_detected
    })

# ------------------------------------------------------------
# Create output directory
# ------------------------------------------------------------

OUTPUT_DIR = "/content/STAGE4_NOISE_DATA"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# Save noisy datasets
# ------------------------------------------------------------

for level, X_noisy in noisy_test_sets.items():

    output_path = os.path.join(
        OUTPUT_DIR,
        f"X_test_noise_pm{level}dBm.npy"
    )

    np.save(
        output_path,
        X_noisy
    )

    print(
        f"\n✓ Saved ±{level} dBm test set:"
    )

    print(
        f"  {output_path}"
    )

# ------------------------------------------------------------
# Save noise summary
# ------------------------------------------------------------

noise_summary = pd.DataFrame(
    noise_records
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "stage4_noise_generation_summary.csv"
)

noise_summary.to_csv(
    summary_path,
    index=False
)

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NOISE GENERATION SUMMARY")
print("=" * 80)

display(
    noise_summary.style.format({
        "Detected_Percentage": "{:.2f}",
        "NoDetection_Percentage": "{:.2f}",
        "Min_Noise": "{:.6f}",
        "Max_Noise": "{:.6f}",
        "Mean_Absolute_Noise": "{:.6f}"
    })
)

print(
    f"\n✓ Summary saved to:\n{summary_path}"
)

print("\n" + "=" * 80)
print("CELL 5 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STAGE 4 — RSSI NOISE ROBUSTNESS
# CELL 6 — CONVENTIONAL MODEL EVALUATION
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 80)
print("STAGE 4 — RSSI NOISE ROBUSTNESS")
print("CELL 6 — CONVENTIONAL MODEL EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

OUTPUT_DIR = "/content/STAGE4_NOISE_RESULTS"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# Original test targets
# ------------------------------------------------------------

y_test_stage4 = y_test.to_numpy(
    dtype=np.float64
) if isinstance(y_test, pd.DataFrame) else np.asarray(
    y_test,
    dtype=np.float64
)

assert y_test_stage4.shape == (226, 2)

# ------------------------------------------------------------
# Model dictionary
# ------------------------------------------------------------

conventional_models = {

    "kNN": final_knn,

    "Decision Tree": final_decision_tree,

    "Random Forest": final_random_forest,

    "SVR": final_svr
}

# ------------------------------------------------------------
# Localization error function
#
# IMPORTANT:
# This is the SAME metric used for completed Stage 1:
#
# mean Euclidean localization error.
# ------------------------------------------------------------

def mean_localization_error(
    y_true,
    y_pred
):

    distances = np.sqrt(
        np.sum(
            (y_true - y_pred) ** 2,
            axis=1
        )
    )

    return np.mean(distances)


# ------------------------------------------------------------
# Baseline Stage 1 errors
# ------------------------------------------------------------

baseline_errors = {

    "kNN": 1.62792358,

    "Decision Tree": 1.67551500,

    "Random Forest": 1.63321900,

    "SVR": 1.60792300
}

# ------------------------------------------------------------
# Noise levels
# ------------------------------------------------------------

NOISE_LEVELS = [2, 5, 8]

all_results = []

all_predictions = {}

# ------------------------------------------------------------
# Evaluate each noise level
# ------------------------------------------------------------

for level in NOISE_LEVELS:

    print("\n" + "=" * 80)
    print(f"EVALUATING ±{level} dBm")
    print("=" * 80)

    # --------------------------------------------------------
    # Load previously generated noisy test data
    # --------------------------------------------------------

    noisy_path = os.path.join(
        "/content/STAGE4_NOISE_DATA",
        f"X_test_noise_pm{level}dBm.npy"
    )

    X_noisy = np.load(
        noisy_path
    )

    print(
        f"Loaded noisy data: {X_noisy.shape}"
    )

    assert X_noisy.shape == (226, 13)

    # --------------------------------------------------------
    # Confirm -200 values remain unchanged
    # --------------------------------------------------------

    original_X = X_test.to_numpy(
        dtype=np.float64
    ) if isinstance(X_test, pd.DataFrame) else np.asarray(
        X_test,
        dtype=np.float64
    )

    original_no_detection = (
        original_X == -200
    )

    assert np.array_equal(
        X_noisy[original_no_detection],
        original_X[original_no_detection]
    )

    print(
        "✓ -200 values confirmed unchanged"
    )

    # --------------------------------------------------------
    # Apply ORIGINAL Stage 1 scaler
    #
    # DO NOT fit scaler here.
    # --------------------------------------------------------

    X_noisy_scaled = (
        final_minmax_scaler.transform(
            X_noisy
        )
    )

    print(
        "✓ Original Stage 1 scaler applied"
    )

    print(
        "Scaled shape:",
        X_noisy_scaled.shape
    )

    # --------------------------------------------------------
    # Evaluate models
    # --------------------------------------------------------

    level_predictions = {}

    for model_name, model in conventional_models.items():

        # ----------------------------------------------------
        # Predict
        # ----------------------------------------------------

        prediction = model.predict(
            X_noisy_scaled
        )

        prediction = np.asarray(
            prediction,
            dtype=np.float64
        )

        assert prediction.shape == (226, 2)

        level_predictions[
            model_name
        ] = prediction

        # ----------------------------------------------------
        # Calculate mean localization error
        # ----------------------------------------------------

        mean_error = mean_localization_error(
            y_test_stage4,
            prediction
        )

        # ----------------------------------------------------
        # Calculate degradation
        # ----------------------------------------------------

        baseline = baseline_errors[
            model_name
        ]

        absolute_degradation = (
            mean_error - baseline
        )

        percentage_degradation = (
            absolute_degradation
            / baseline
            * 100.0
        )

        # ----------------------------------------------------
        # Store
        # ----------------------------------------------------

        all_results.append({

            "Noise_Level": f"±{level} dBm",

            "Noise_dBm": level,

            "Model": model_name,

            "Baseline_Error_grid_units":
                baseline,

            "Noisy_Error_grid_units":
                mean_error,

            "Absolute_Degradation_grid_units":
                absolute_degradation,

            "Percentage_Degradation":
                percentage_degradation
        })

        print(
            f"{model_name:18s} : "
            f"{mean_error:.6f} m "
            f"(baseline {baseline:.6f} m)"
        )

    all_predictions[
        level
    ] = level_predictions

# ------------------------------------------------------------
# Results DataFrame
# ------------------------------------------------------------

stage4_conventional_results = pd.DataFrame(
    all_results
)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

results_path = os.path.join(
    OUTPUT_DIR,
    "stage4_conventional_noise_results.csv"
)

stage4_conventional_results.to_csv(
    results_path,
    index=False
)

# ------------------------------------------------------------
# Save predictions
# ------------------------------------------------------------

for level, predictions in all_predictions.items():

    for model_name, prediction in predictions.items():

        safe_name = (
            model_name
            .lower()
            .replace(" ", "_")
        )

        prediction_path = os.path.join(
            OUTPUT_DIR,
            f"{safe_name}_predictions_pm{level}dBm.npy"
        )

        np.save(
            prediction_path,
            prediction
        )

# ------------------------------------------------------------
# Display final results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STAGE 4 — CONVENTIONAL MODEL RESULTS")
print("=" * 80)

display(
    stage4_conventional_results.style.format({

        "Baseline_Error_grid_units":
            "{:.4f}",

        "Noisy_Error_grid_units":
            "{:.6f}",

        "Absolute_Degradation_grid_units":
            "{:+.6f}",

        "Percentage_Degradation":
            "{:+.2f}%"
    })
)

print(
    f"\n✓ Results saved to:\n{results_path}"
)

print("\n" + "=" * 80)
print("CELL 6 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STAGE 4 — CELL 7B
# CLEAN NEURAL MODEL VERIFICATION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("STAGE 4 — CLEAN NEURAL MODEL VERIFICATION")
print("=" * 80)

# ------------------------------------------------------------
# IMPORTANT:
# The neural models and predictions have already been loaded
# by the previous cell.
#
# We ONLY verify them here.
# ------------------------------------------------------------

required_variables = [
    "final_mlp",
    "final_cnn_gap",
    "final_cnn_flat",
    "pred_mlp_original",
    "pred_cnn_gap_original",
    "pred_cnn_flat_original"
]

print("\nChecking existing neural objects...")
print("-" * 80)

for var_name in required_variables:

    if var_name in globals():
        print(f"✓ {var_name}")
    else:
        raise RuntimeError(
            f"{var_name} is not available. "
            "Run the neural model loading cell once."
        )

# ------------------------------------------------------------
# Load saved Stage 1 prediction CSV
# ------------------------------------------------------------

prediction_csv = "/content/test_predictions_all_models.csv"

existing_predictions = pd.read_csv(
    prediction_csv
)

print(
    "\nPrediction CSV shape:",
    existing_predictions.shape
)

# ------------------------------------------------------------
# Convert y_test safely to NumPy
# ------------------------------------------------------------

if isinstance(y_test, pd.DataFrame):

    y_neural = y_test.to_numpy(
        dtype=np.float64
    )

else:

    y_neural = np.asarray(
        y_test,
        dtype=np.float64
    )

assert y_neural.shape == (226, 2)

# ------------------------------------------------------------
# Neural models
# ------------------------------------------------------------

models_to_verify = [

    {
        "name": "MLP",
        "prediction": pred_mlp_original,
        "x_column": "mlp_x",
        "y_column": "mlp_y",
        "error_column": "mlp_error",
        "expected": 2.0929
    },

    {
        "name": "CNN-GAP",
        "prediction": pred_cnn_gap_original,
        "x_column": "1d_cnn_gap_x",
        "y_column": "1d_cnn_gap_y",
        "error_column": "1d_cnn_gap_error",
        "expected": 3.0200
    },

    {
        "name": "CNN-FLAT",
        "prediction": pred_cnn_flat_original,
        "x_column": "1d_cnn_flat_x",
        "y_column": "1d_cnn_flat_y",
        "error_column": "1d_cnn_flat_error",
        "expected": 2.0254
    }
]

# ------------------------------------------------------------
# Verification results
# ------------------------------------------------------------

verification_results = []

for item in models_to_verify:

    name = item["name"]

    prediction = np.asarray(
        item["prediction"],
        dtype=np.float64
    )

    x_column = item["x_column"]
    y_column = item["y_column"]
    error_column = item["error_column"]

    expected = item["expected"]

    print("\n" + "-" * 80)
    print(f"VERIFYING: {name}")
    print("-" * 80)

    # --------------------------------------------------------
    # Shape check
    # --------------------------------------------------------

    print(
        "Recovered prediction shape:",
        prediction.shape
    )

    assert prediction.shape == (226, 2)

    # --------------------------------------------------------
    # Extract saved CSV predictions
    # --------------------------------------------------------

    csv_prediction = existing_predictions[
        [
            x_column,
            y_column
        ]
    ].to_numpy(
        dtype=np.float64
    )

    csv_error = existing_predictions[
        error_column
    ].to_numpy(
        dtype=np.float64
    )

    assert csv_prediction.shape == (226, 2)
    assert csv_error.shape == (226,)

    # --------------------------------------------------------
    # Coordinate comparison
    # --------------------------------------------------------

    coordinate_difference = np.abs(
        prediction -
        csv_prediction
    )

    max_coordinate_difference = float(
        np.max(coordinate_difference)
    )

    mean_coordinate_difference = float(
        np.mean(coordinate_difference)
    )

    # --------------------------------------------------------
    # Recalculate localization errors
    # --------------------------------------------------------

    recovered_error = np.sqrt(
        np.sum(
            (
                y_neural -
                prediction
            ) ** 2,
            axis=1
        )
    )

    # --------------------------------------------------------
    # Error comparison
    # --------------------------------------------------------

    error_difference = np.abs(
        recovered_error -
        csv_error
    )

    max_error_difference = float(
        np.max(error_difference)
    )

    mean_error_difference = float(
        np.mean(error_difference)
    )

    # --------------------------------------------------------
    # Mean localization error
    # --------------------------------------------------------

    recovered_mean_error = float(
        np.mean(recovered_error)
    )

    saved_csv_mean_error = float(
        np.mean(csv_error)
    )

    # --------------------------------------------------------
    # Print
    # --------------------------------------------------------

    print(
        f"Recovered mean error : "
        f"{recovered_mean_error:.6f} m"
    )

    print(
        f"Saved CSV mean error : "
        f"{saved_csv_mean_error:.6f} m"
    )

    print(
        f"Expected Stage 1     : "
        f"{expected:.4f} grid-coordinate units"
    )

    print(
        f"Max coordinate diff  : "
        f"{max_coordinate_difference:.12f}"
    )

    print(
        f"Mean coordinate diff : "
        f"{mean_coordinate_difference:.12f}"
    )

    print(
        f"Max error diff       : "
        f"{max_error_difference:.12f}"
    )

    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    verification_results.append({

        "Model": name,

        "Recovered_Mean_Error_grid_units":
            recovered_mean_error,

        "CSV_Mean_Error_grid_units":
            saved_csv_mean_error,

        "Completed_Stage1_grid_units":
            expected,

        "Difference_from_Stage1_grid_units":
            abs(
                recovered_mean_error -
                expected
            ),

        "Max_Coordinate_Difference":
            max_coordinate_difference,

        "Max_Error_Difference":
            max_error_difference
    })


# ------------------------------------------------------------
# Final verification table
# ------------------------------------------------------------

verification_df = pd.DataFrame(
    verification_results
)

print("\n" + "=" * 80)
print("NEURAL MODEL VERIFICATION TABLE")
print("=" * 80)

display(
    verification_df.style.format({

        "Recovered_Mean_Error_grid_units":
            "{:.6f}",

        "CSV_Mean_Error_grid_units":
            "{:.6f}",

        "Completed_Stage1_grid_units":
            "{:.4f}",

        "Difference_from_Stage1_grid_units":
            "{:.6f}",

        "Max_Coordinate_Difference":
            "{:.12f}",

        "Max_Error_Difference":
            "{:.12f}"
    })
)

# ------------------------------------------------------------
# PASS / FAIL
# ------------------------------------------------------------

TOLERANCE = 1e-5

predictions_match = (
    verification_df[
        "Max_Coordinate_Difference"
    ].max()
    < TOLERANCE
)

errors_match = (
    verification_df[
        "Max_Error_Difference"
    ].max()
    < TOLERANCE
)

print("\n" + "=" * 80)
print("FINAL STATUS")
print("=" * 80)

print(
    "Predictions match saved CSV:",
    "✓ YES"
    if predictions_match
    else "❌ NO"
)

print(
    "Errors match saved CSV:",
    "✓ YES"
    if errors_match
    else "❌ NO"
)

if predictions_match and errors_match:

    print("\n✓ MLP VERIFIED")
    print("✓ CNN-GAP VERIFIED")
    print("✓ CNN-FLAT VERIFIED")
    print("\n✓ ALL 7 STAGE 1 MODELS VERIFIED")
    print("✓ READY FOR NEURAL STAGE 4")

else:

    print("\n❌ VERIFICATION FAILED")
    print("DO NOT RUN NEURAL NOISE EVALUATION")

print("=" * 80)

In [ ]:
# ============================================================
# STAGE 4 — CELL 8A
# FIXED CNN INPUT SHAPE EVALUATION
# ============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

print("=" * 80)
print("STAGE 4 — NEURAL MODEL EVALUATION")
print("CELL 8A — FIXED CNN INPUT SHAPE")
print("=" * 80)

OUTPUT_DIR = "/content/STAGE4_NOISE_RESULTS"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Targets
# ------------------------------------------------------------

if isinstance(y_test, pd.DataFrame):
    y_stage4 = y_test.to_numpy(dtype=np.float64)
else:
    y_stage4 = np.asarray(y_test, dtype=np.float64)

assert y_stage4.shape == (226, 2)

# ------------------------------------------------------------
# VERIFIED EXACT Stage 1 baselines
# ------------------------------------------------------------

baseline_errors = {
    "MLP": 2.093180,
    "CNN-GAP": 3.062271,
    "CNN-FLAT": 2.025434
}

# ------------------------------------------------------------
# Models
# ------------------------------------------------------------

neural_models = {
    "MLP": final_mlp,
    "CNN-GAP": final_cnn_gap,
    "CNN-FLAT": final_cnn_flat
}

# ------------------------------------------------------------
# Metric
# ------------------------------------------------------------

def mean_localization_error(y_true, y_pred):

    distances = np.sqrt(
        np.sum(
            (y_true - y_pred) ** 2,
            axis=1
        )
    )

    return float(np.mean(distances))


# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

neural_results = []
neural_predictions = {}

for level in [2, 5, 8]:

    print("\n" + "=" * 80)
    print(f"EVALUATING ±{level} dBm")
    print("=" * 80)

    noisy_path = (
        f"/content/STAGE4_NOISE_DATA/"
        f"X_test_noise_pm{level}dBm.npy"
    )

    X_noisy = np.load(noisy_path)

    print("Noisy input:", X_noisy.shape)

    assert X_noisy.shape == (226, 13)

    # --------------------------------------------------------
    # Verify -200 unchanged
    # --------------------------------------------------------

    if isinstance(X_test, pd.DataFrame):
        X_original = X_test.to_numpy(dtype=np.float64)
    else:
        X_original = np.asarray(X_test, dtype=np.float64)

    no_detection_mask = (X_original == -200)

    assert np.array_equal(
        X_noisy[no_detection_mask],
        X_original[no_detection_mask]
    )

    print("✓ -200 values unchanged")

    # --------------------------------------------------------
    # ORIGINAL Stage 1 scaler
    # --------------------------------------------------------

    X_scaled = final_minmax_scaler.transform(
        X_noisy
    )

    X_scaled = np.asarray(
        X_scaled,
        dtype=np.float32
    )

    print(
        "Scaled input:",
        X_scaled.shape
    )

    level_predictions = {}

    # --------------------------------------------------------
    # MLP
    # --------------------------------------------------------

    if level == 2:

        # Already successfully evaluated in previous Cell 8.
        # We recompute it only for consistency in the final
        # results table; no new model/noise is created.

        mlp_input = tf.convert_to_tensor(
            X_scaled,
            dtype=tf.float32
        )

        mlp_input = tf.ensure_shape(
            mlp_input,
            (226, 13)
        )

        pred_mlp = final_mlp(
            mlp_input,
            training=False
        ).numpy()

    else:

        mlp_input = tf.convert_to_tensor(
            X_scaled,
            dtype=tf.float32
        )

        mlp_input = tf.ensure_shape(
            mlp_input,
            (226, 13)
        )

        pred_mlp = final_mlp(
            mlp_input,
            training=False
        ).numpy()

    pred_mlp = np.asarray(
        pred_mlp,
        dtype=np.float64
    )

    assert pred_mlp.shape == (226, 2)

    mlp_error = mean_localization_error(
        y_stage4,
        pred_mlp
    )

    baseline = baseline_errors["MLP"]

    neural_results.append({
        "Noise_Level": f"±{level} dBm",
        "Noise_dBm": level,
        "Model": "MLP",
        "Baseline_Error_grid_units": baseline,
        "Noisy_Error_grid_units": mlp_error,
        "Absolute_Degradation_grid_units":
            mlp_error - baseline,
        "Percentage_Degradation":
            ((mlp_error - baseline) / baseline) * 100
    })

    level_predictions["MLP"] = pred_mlp

    print(
        f"MLP       : {mlp_error:.6f} m"
    )

    # --------------------------------------------------------
    # CNN-GAP and CNN-FLAT
    # --------------------------------------------------------

    # Explicit fixed TensorFlow shape.
    cnn_input = tf.convert_to_tensor(
        X_scaled[:, :, np.newaxis],
        dtype=tf.float32
    )

    cnn_input = tf.ensure_shape(
        cnn_input,
        (226, 13, 1)
    )

    print(
        "CNN input:",
        cnn_input.shape
    )

    for model_name in ["CNN-GAP", "CNN-FLAT"]:

        model = neural_models[
            model_name
        ]

        print(
            f"Running {model_name}..."
        )

        prediction = model(
            cnn_input,
            training=False
        ).numpy()

        prediction = np.asarray(
            prediction,
            dtype=np.float64
        )

        assert prediction.shape == (226, 2)

        error = mean_localization_error(
            y_stage4,
            prediction
        )

        baseline = baseline_errors[
            model_name
        ]

        degradation = (
            error - baseline
        )

        percentage = (
            degradation /
            baseline *
            100
        )

        neural_results.append({

            "Noise_Level":
                f"±{level} dBm",

            "Noise_dBm":
                level,

            "Model":
                model_name,

            "Baseline_Error_grid_units":
                baseline,

            "Noisy_Error_grid_units":
                error,

            "Absolute_Degradation_grid_units":
                degradation,

            "Percentage_Degradation":
                percentage
        })

        level_predictions[
            model_name
        ] = prediction

        print(
            f"{model_name:10s}: "
            f"{error:.6f} m "
            f"(Δ {degradation:+.6f} m, "
            f"{percentage:+.2f}%)"
        )

    neural_predictions[
        level
    ] = level_predictions


# ------------------------------------------------------------
# DataFrame
# ------------------------------------------------------------

stage4_neural_results = pd.DataFrame(
    neural_results
)

# ------------------------------------------------------------
# Sort
# ------------------------------------------------------------

model_order = [
    "MLP",
    "CNN-GAP",
    "CNN-FLAT"
]

stage4_neural_results["Model"] = pd.Categorical(
    stage4_neural_results["Model"],
    categories=model_order,
    ordered=True
)

stage4_neural_results = (
    stage4_neural_results
    .sort_values(
        ["Noise_dBm", "Model"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Save neural results
# ------------------------------------------------------------

results_path = (
    f"{OUTPUT_DIR}/"
    "stage4_neural_noise_results.csv"
)

stage4_neural_results.to_csv(
    results_path,
    index=False
)

# ------------------------------------------------------------
# Save predictions
# ------------------------------------------------------------

for level, predictions in neural_predictions.items():

    for model_name, prediction in predictions.items():

        safe_name = (
            model_name
            .lower()
            .replace("-", "_")
        )

        np.save(
            f"{OUTPUT_DIR}/"
            f"{safe_name}_predictions_pm{level}dBm.npy",
            prediction
        )

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NEURAL STAGE 4 RESULTS")
print("=" * 80)

display(
    stage4_neural_results.style.format({

        "Baseline_Error_grid_units":
            "{:.6f}",

        "Noisy_Error_grid_units":
            "{:.6f}",

        "Absolute_Degradation_grid_units":
            "{:+.6f}",

        "Percentage_Degradation":
            "{:+.2f}%"
    })
)

print(
    f"\n✓ Saved:\n{results_path}"
)

print("\n" + "=" * 80)
print("CELL 8A COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STAGE 4 — CELL 9
# MASTER CONSOLIDATION: ALL 7 MODELS
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 80)
print("STAGE 4 — RSSI NOISE ROBUSTNESS")
print("CELL 9 — MASTER CONSOLIDATION")
print("=" * 80)

RESULTS_DIR = "/content/STAGE4_NOISE_RESULTS"

# ------------------------------------------------------------
# Load completed conventional results
# ------------------------------------------------------------

conventional_path = (
    f"{RESULTS_DIR}/"
    "stage4_conventional_noise_results.csv"
)

neural_path = (
    f"{RESULTS_DIR}/"
    "stage4_neural_noise_results.csv"
)

conventional = pd.read_csv(
    conventional_path
)

neural = pd.read_csv(
    neural_path
)

print("\nConventional results:")
print(conventional.shape)

print("\nNeural results:")
print(neural.shape)

# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

stage4_master = pd.concat(
    [
        conventional,
        neural
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# Standardize model names/order
# ------------------------------------------------------------

model_order = [
    "kNN",
    "Decision Tree",
    "Random Forest",
    "SVR",
    "MLP",
    "CNN-GAP",
    "CNN-FLAT"
]

stage4_master["Model"] = pd.Categorical(
    stage4_master["Model"],
    categories=model_order,
    ordered=True
)

stage4_master = (
    stage4_master
    .sort_values(
        ["Noise_dBm", "Model"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Verify expected number of rows
# ------------------------------------------------------------

expected_rows = 7 * 3

assert len(stage4_master) == expected_rows

print(
    f"\n✓ Expected rows: {expected_rows}"
)

print(
    f"✓ Actual rows:   {len(stage4_master)}"
)

# ------------------------------------------------------------
# Verify every model has all 3 noise levels
# ------------------------------------------------------------

coverage = (
    stage4_master
    .groupby("Model", observed=True)["Noise_dBm"]
    .nunique()
)

print("\nNoise-level coverage:")
print(coverage)

assert all(
    coverage == 3
)

print(
    "\n✓ All 7 models have ±2, ±5 and ±8 dBm results"
)

# ------------------------------------------------------------
# Save master table
# ------------------------------------------------------------

master_path = (
    f"{RESULTS_DIR}/"
    "STAGE4_MASTER_NOISE_ROBUSTNESS.csv"
)

stage4_master.to_csv(
    master_path,
    index=False
)

# ------------------------------------------------------------
# Display complete master table
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPLETE STAGE 4 MASTER RESULTS")
print("=" * 80)

display(
    stage4_master.style.format({

        "Baseline_Error_grid_units":
            "{:.6f}",

        "Noisy_Error_grid_units":
            "{:.6f}",

        "Absolute_Degradation_grid_units":
            "{:+.6f}",

        "Percentage_Degradation":
            "{:+.2f}%"
    })
)

# ------------------------------------------------------------
# Create compact error table
# ------------------------------------------------------------

error_table = (
    stage4_master
    .pivot(
        index="Model",
        columns="Noise_Level",
        values="Noisy_Error_grid_units"
    )
    .reindex(model_order)
)

print("\n" + "=" * 80)
print("LOCALIZATION ERROR UNDER RSSI NOISE")
print("=" * 80)

display(
    error_table.style.format(
        "{:.4f} grid-coordinate units"
    )
)

# ------------------------------------------------------------
# Create compact degradation table
# ------------------------------------------------------------

degradation_table = (
    stage4_master
    .pivot(
        index="Model",
        columns="Noise_Level",
        values="Percentage_Degradation"
    )
    .reindex(model_order)
)

print("\n" + "=" * 80)
print("PERCENTAGE DEGRADATION FROM STAGE 1 BASELINE")
print("=" * 80)

display(
    degradation_table.style.format(
        "{:+.2f}%"
    )
)

# ------------------------------------------------------------
# Ranking at each noise level
#
# Lower localization error = better
# ------------------------------------------------------------

ranking_rows = []

for level in [2, 5, 8]:

    subset = (
        stage4_master[
            stage4_master["Noise_dBm"] == level
        ]
        .sort_values(
            "Noisy_Error_grid_units"
        )
        .reset_index(drop=True)
    )

    for rank, row in enumerate(
        subset.itertuples(index=False),
        start=1
    ):

        ranking_rows.append({

            "Noise_Level":
                f"±{level} dBm",

            "Noise_dBm":
                level,

            "Rank":
                rank,

            "Model":
                row.Model,

            "Localization_Error_grid_units":
                row.Noisy_Error_grid_units,

            "Percentage_Degradation":
                row.Percentage_Degradation
        })

ranking_table = pd.DataFrame(
    ranking_rows
)

ranking_path = (
    f"{RESULTS_DIR}/"
    "stage4_model_ranking.csv"
)

ranking_table.to_csv(
    ranking_path,
    index=False
)

# ------------------------------------------------------------
# Display ranking
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MODEL RANKING AT EACH NOISE LEVEL")
print("=" * 80)

display(
    ranking_table.style.format({

        "Localization_Error_grid_units":
            "{:.6f}",

        "Percentage_Degradation":
            "{:+.2f}%"
    })
)

# ------------------------------------------------------------
# Robustness summary
#
# Average percentage degradation across the 3 noise levels.
# Lower is better.
# ------------------------------------------------------------

robustness_summary = (
    stage4_master
    .groupby(
        "Model",
        observed=True
    )
    .agg(
        Baseline_Error_grid_units=(
            "Baseline_Error_grid_units",
            "first"
        ),

        Mean_Noise_Error_grid_units=(
            "Noisy_Error_grid_units",
            "mean"
        ),

        Mean_Percentage_Degradation=(
            "Percentage_Degradation",
            "mean"
        ),

        Max_Percentage_Degradation=(
            "Percentage_Degradation",
            "max"
        ),

        Mean_Absolute_Degradation_grid_units=(
            "Absolute_Degradation_grid_units",
            "mean"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# Robustness ranking
# ------------------------------------------------------------

robustness_summary = (
    robustness_summary
    .sort_values(
        "Mean_Percentage_Degradation"
    )
    .reset_index(drop=True)
)

robustness_summary.insert(
    0,
    "Robustness_Rank",
    np.arange(
        1,
        len(robustness_summary) + 1
    )
)

robustness_path = (
    f"{RESULTS_DIR}/"
    "stage4_robustness_summary.csv"
)

robustness_summary.to_csv(
    robustness_path,
    index=False
)

print("\n" + "=" * 80)
print("OVERALL ROBUSTNESS SUMMARY")
print("=" * 80)

display(
    robustness_summary.style.format({

        "Baseline_Error_grid_units":
            "{:.6f}",

        "Mean_Noise_Error_grid_units":
            "{:.6f}",

        "Mean_Percentage_Degradation":
            "{:+.2f}%",

        "Max_Percentage_Degradation":
            "{:+.2f}%",

        "Mean_Absolute_Degradation_grid_units":
            "{:+.6f}"
    })
)

# ------------------------------------------------------------
# Final save confirmation
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STAGE 4 MASTER CONSOLIDATION COMPLETE")
print("=" * 80)

print("\nSaved files:")

print(
    "✓",
    master_path
)

print(
    "✓",
    ranking_path
)

print(
    "✓",
    robustness_path
)

In [ ]:
# ============================================================
# STAGE 4 — CELL 10
# SCI-READY ROBUSTNESS FIGURES
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 80)
print("STAGE 4 — SCI-READY FIGURES")
print("=" * 80)

RESULTS_DIR = "/content/STAGE4_NOISE_RESULTS"

# ------------------------------------------------------------
# Load verified master results
# ------------------------------------------------------------

master_path = (
    f"{RESULTS_DIR}/"
    "STAGE4_MASTER_NOISE_ROBUSTNESS.csv"
)

summary_path = (
    f"{RESULTS_DIR}/"
    "stage4_robustness_summary.csv"
)

df = pd.read_csv(master_path)
summary = pd.read_csv(summary_path)

# ------------------------------------------------------------
# Model order
# ------------------------------------------------------------

model_order = [
    "kNN",
    "Decision Tree",
    "Random Forest",
    "SVR",
    "MLP",
    "CNN-GAP",
    "CNN-FLAT"
]

# ------------------------------------------------------------
# Noise order
# ------------------------------------------------------------

noise_levels = [2, 5, 8]

noise_labels = [
    "±2 dBm",
    "±5 dBm",
    "±8 dBm"
]

# ============================================================
# FIGURE 1
# Localization Error vs RSSI Noise
# ============================================================

plt.figure(figsize=(10, 6))

for model in model_order:

    subset = (
        df[
            df["Model"] == model
        ]
        .sort_values("Noise_dBm")
    )

    plt.plot(
        subset["Noise_dBm"],
        subset["Noisy_Error_grid_units"],
        marker="o",
        linewidth=2,
        markersize=6,
        label=model
    )

plt.xticks(
    noise_levels,
    noise_labels
)

plt.xlabel(
    "RSSI Noise Level",
    fontsize=12
)

plt.ylabel(
    "Mean Euclidean Localization Error (m)",
    fontsize=12
)

plt.title(
    "Localization Error under RSSI Noise",
    fontsize=14
)

plt.grid(
    True,
    linestyle="--",
    alpha=0.4
)

plt.legend(
    fontsize=9,
    loc="best"
)

plt.tight_layout()

fig1_path = (
    f"{RESULTS_DIR}/"
    "Figure_Stage4_Localization_Error_vs_Noise.png"
)

plt.savefig(
    fig1_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# FIGURE 2
# Percentage Degradation vs RSSI Noise
# ============================================================

plt.figure(figsize=(10, 6))

for model in model_order:

    subset = (
        df[
            df["Model"] == model
        ]
        .sort_values("Noise_dBm")
    )

    plt.plot(
        subset["Noise_dBm"],
        subset["Percentage_Degradation"],
        marker="o",
        linewidth=2,
        markersize=6,
        label=model
    )

plt.axhline(
    0,
    linewidth=1
)

plt.xticks(
    noise_levels,
    noise_labels
)

plt.xlabel(
    "RSSI Noise Level",
    fontsize=12
)

plt.ylabel(
    "Localization Error Degradation (%)",
    fontsize=12
)

plt.title(
    "Localization Error Degradation under RSSI Noise",
    fontsize=14
)

plt.grid(
    True,
    linestyle="--",
    alpha=0.4
)

plt.legend(
    fontsize=9,
    loc="best"
)

plt.tight_layout()

fig2_path = (
    f"{RESULTS_DIR}/"
    "Figure_Stage4_Percentage_Degradation_vs_Noise.png"
)

plt.savefig(
    fig2_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# FIGURE 3
# Overall Robustness Ranking
# ============================================================

summary_plot = (
    summary
    .sort_values(
        "Mean_Percentage_Degradation"
    )
    .reset_index(drop=True)
)

plt.figure(figsize=(10, 6))

plt.bar(
    summary_plot["Model"],
    summary_plot["Mean_Percentage_Degradation"]
)

plt.ylabel(
    "Mean Percentage Degradation (%)",
    fontsize=12
)

plt.xlabel(
    "Model",
    fontsize=12
)

plt.title(
    "Overall RSSI Noise Robustness",
    fontsize=14
)

plt.xticks(
    rotation=30,
    ha="right"
)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

# ------------------------------------------------------------
# Add values above bars
# ------------------------------------------------------------

for i, value in enumerate(
    summary_plot[
        "Mean_Percentage_Degradation"
    ]
):

    plt.text(
        i,
        value + 0.15,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.tight_layout()

fig3_path = (
    f"{RESULTS_DIR}/"
    "Figure_Stage4_Overall_Robustness.png"
)

plt.savefig(
    fig3_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# SAVE HIGH-RESOLUTION PDF VERSIONS
# ============================================================

# Figure 1
plt.figure(figsize=(10, 6))

for model in model_order:

    subset = (
        df[df["Model"] == model]
        .sort_values("Noise_dBm")
    )

    plt.plot(
        subset["Noise_dBm"],
        subset["Noisy_Error_grid_units"],
        marker="o",
        linewidth=2,
        markersize=6,
        label=model
    )

plt.xticks(noise_levels, noise_labels)
plt.xlabel(
    "RSSI Noise Level",
    fontsize=12
)
plt.ylabel(
    "Mean Euclidean Localization Error (m)",
    fontsize=12
)
plt.title(
    "Localization Error under RSSI Noise",
    fontsize=14
)
plt.grid(
    True,
    linestyle="--",
    alpha=0.4
)
plt.legend(fontsize=9)
plt.tight_layout()

fig1_pdf = (
    f"{RESULTS_DIR}/"
    "Figure_Stage4_Localization_Error_vs_Noise.pdf"
)

plt.savefig(
    fig1_pdf,
    bbox_inches="tight"
)

plt.close()

# Figure 2
plt.figure(figsize=(10, 6))

for model in model_order:

    subset = (
        df[df["Model"] == model]
        .sort_values("Noise_dBm")
    )

    plt.plot(
        subset["Noise_dBm"],
        subset["Percentage_Degradation"],
        marker="o",
        linewidth=2,
        markersize=6,
        label=model
    )

plt.axhline(0, linewidth=1)
plt.xticks(noise_levels, noise_labels)
plt.xlabel(
    "RSSI Noise Level",
    fontsize=12
)
plt.ylabel(
    "Localization Error Degradation (%)",
    fontsize=12
)
plt.title(
    "Localization Error Degradation under RSSI Noise",
    fontsize=14
)
plt.grid(
    True,
    linestyle="--",
    alpha=0.4
)
plt.legend(fontsize=9)
plt.tight_layout()

fig2_pdf = (
    f"{RESULTS_DIR}/"
    "Figure_Stage4_Percentage_Degradation_vs_Noise.pdf"
)

plt.savefig(
    fig2_pdf,
    bbox_inches="tight"
)

plt.close()

print("\n" + "=" * 80)
print("FIGURES SAVED")
print("=" * 80)

print("✓", fig1_path)
print("✓", fig2_path)
print("✓", fig3_path)

print("✓", fig1_pdf)
print("✓", fig2_pdf)

print("\n" + "=" * 80)
print("STAGE 4 FIGURE GENERATION COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# SCI STATISTICAL VALIDATION — STAGES 1 + 4
# NO RETRAINING
# PAIRED OBSERVATION-LEVEL VALIDATION
# ============================================================

import os
import glob
import warnings
import numpy as np
import pandas as pd
import joblib

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

print("=" * 90)
print("SCI STATISTICAL VALIDATION — STAGE 1 vs STAGE 4")
print("=" * 90)

# ============================================================
# PATHS
# ============================================================

BASE_DIR = "/content"

STAGE1_CSV = (
    f"{BASE_DIR}/test_predictions_all_models.csv"
)

STAGE4_DIR = (
    f"{BASE_DIR}/STAGE4_NOISE_RESULTS"
)

NOISE_DIR = (
    f"{BASE_DIR}/STAGE4_NOISE_DATA"
)

OUTPUT_DIR = (
    f"{BASE_DIR}/STAGE4_NOISE_RESULTS"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# ============================================================
# MODEL ORDER
# ============================================================

MODELS = [
    "kNN",
    "Decision Tree",
    "Random Forest",
    "SVR",
    "MLP",
    "CNN-GAP",
    "CNN-FLAT"
]

# Mapping to Stage 1 prediction CSV columns
STAGE1_COLUMNS = {

    "kNN":
        ("knn_x", "knn_y", "knn_error"),

    "Decision Tree":
        (
            "decision_tree_x",
            "decision_tree_y",
            "decision_tree_error"
        ),

    "Random Forest":
        (
            "random_forest_x",
            "random_forest_y",
            "random_forest_error"
        ),

    "SVR":
        (
            "svr_x",
            "svr_y",
            "svr_error"
        ),

    "MLP":
        (
            "mlp_x",
            "mlp_y",
            "mlp_error"
        ),

    "CNN-GAP":
        (
            "1d_cnn_gap_x",
            "1d_cnn_gap_y",
            "1d_cnn_gap_error"
        ),

    "CNN-FLAT":
        (
            "1d_cnn_flat_x",
            "1d_cnn_flat_y",
            "1d_cnn_flat_error"
        )
}

# ============================================================
# LOAD STAGE 1 PREDICTIONS
# ============================================================

assert os.path.exists(
    STAGE1_CSV
), f"Missing Stage 1 CSV: {STAGE1_CSV}"

stage1 = pd.read_csv(
    STAGE1_CSV
)

print(
    f"\n✓ Stage 1 prediction CSV loaded: "
    f"{stage1.shape}"
)

assert len(stage1) == 226

# ============================================================
# EXACT STAGE 1 OBSERVATION-LEVEL ERRORS
# ============================================================

stage1_errors = {}

for model in MODELS:

    error_col = STAGE1_COLUMNS[
        model
    ][2]

    errors = (
        stage1[error_col]
        .to_numpy(dtype=np.float64)
    )

    assert len(errors) == 226
    assert np.all(np.isfinite(errors))

    stage1_errors[
        model
    ] = errors

print(
    "✓ All seven Stage 1 observation-level "
    "error vectors loaded"
)

# ============================================================
# LOAD STAGE 4 MASTER RESULTS
# ============================================================

master_path = (
    f"{STAGE4_DIR}/"
    "STAGE4_MASTER_NOISE_ROBUSTNESS.csv"
)

assert os.path.exists(
    master_path
), f"Missing Stage 4 master file: {master_path}"

stage4_master = pd.read_csv(
    master_path
)

print(
    f"✓ Stage 4 master results loaded: "
    f"{stage4_master.shape}"
)

assert len(stage4_master) == 21

# ============================================================
# CHECK FOR SAVED STAGE 4 PREDICTIONS
# ============================================================

print("\n" + "-" * 90)
print("CHECKING EXISTING STAGE 4 PREDICTION FILES")
print("-" * 90)

all_files = glob.glob(
    f"{STAGE4_DIR}/*"
)

for f in sorted(all_files):

    basename = os.path.basename(f)

    if (
        "prediction" in basename.lower()
        or "error" in basename.lower()
    ):

        print(
            "  ",
            basename
        )

# ============================================================
# HELPER: LOAD POSSIBLE SAVED PREDICTION ARRAY
# ============================================================

def find_prediction_file(
    model_name,
    noise_level
):

    safe_name = (
        model_name
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    candidates = [

        f"{STAGE4_DIR}/"
        f"{safe_name}_predictions_pm{noise_level}dBm.npy",

        f"{STAGE4_DIR}/"
        f"{safe_name}_prediction_pm{noise_level}dBm.npy",

        f"{STAGE4_DIR}/"
        f"{safe_name}_pm{noise_level}dBm_predictions.npy"
    ]

    for path in candidates:

        if os.path.exists(path):

            return path

    return None


# ============================================================
# LOAD EXISTING STAGE 4 NEURAL PREDICTIONS
# ============================================================

stage4_predictions = {}

for model in [
    "MLP",
    "CNN-GAP",
    "CNN-FLAT"
]:

    stage4_predictions[
        model
    ] = {}

    for noise in [2, 5, 8]:

        path = find_prediction_file(
            model,
            noise
        )

        if path is not None:

            prediction = np.load(
                path
            )

            assert prediction.shape == (
                226,
                2
            )

            stage4_predictions[
                model
            ][noise] = prediction

            print(
                f"✓ Loaded {model} "
                f"±{noise} dBm predictions"
            )

# ============================================================
# LOAD CONVENTIONAL STAGE 4 PREDICTIONS
# IF NOT SAVED, REPRODUCE INFERENCE USING EXISTING MODELS
#
# IMPORTANT:
# THIS IS NOT TRAINING.
# NO FIT(), NO RETRAINING, NO PARAMETER UPDATE.
# ============================================================

conventional_models = [
    "kNN",
    "Decision Tree",
    "Random Forest",
    "SVR"
]

# Existing Stage 1 model files
model_paths = {

    "kNN":
        f"{BASE_DIR}/final_knn.pkl",

    "Decision Tree":
        f"{BASE_DIR}/final_decision_tree.pkl",

    "Random Forest":
        f"{BASE_DIR}/final_random_forest.pkl",

    "SVR":
        f"{BASE_DIR}/final_svr.pkl"
}

# ------------------------------------------------------------
# Load models only if needed
# ------------------------------------------------------------

loaded_models = {}

for model in conventional_models:

    for noise in [2, 5, 8]:

        if (
            model in stage4_predictions
            and noise in stage4_predictions[model]
        ):
            continue

        # Load trained Stage 1 model
        if model not in loaded_models:

            path = model_paths[model]

            assert os.path.exists(
                path
            ), f"Missing model: {path}"

            loaded_models[
                model
            ] = joblib.load(
                path
            )

            print(
                f"✓ Loaded existing Stage 1 "
                f"{model} model for inference"
            )

        # ----------------------------------------------------
        # Load EXACT previously generated noise
        # ----------------------------------------------------

        noise_path = (
            f"{NOISE_DIR}/"
            f"X_test_noise_pm{noise}dBm.npy"
        )

        assert os.path.exists(
            noise_path
        )

        X_noisy = np.load(
            noise_path
        )

        assert X_noisy.shape == (
            226,
            13
        )

        # ----------------------------------------------------
        # ORIGINAL Stage 1 scaler
        # ----------------------------------------------------

        scaler_path = (
            f"{BASE_DIR}/"
            "final_minmax_scaler.pkl"
        )

        if "scaler" not in locals():

            scaler = joblib.load(
                scaler_path
            )

        X_scaled = scaler.transform(
            X_noisy
        )

        # ----------------------------------------------------
        # INFERENCE ONLY
        # ----------------------------------------------------

        prediction = (
            loaded_models[model]
            .predict(X_scaled)
        )

        prediction = np.asarray(
            prediction,
            dtype=np.float64
        )

        assert prediction.shape == (
            226,
            2
        )

        if model not in stage4_predictions:

            stage4_predictions[
                model
            ] = {}

        stage4_predictions[
            model
        ][noise] = prediction

        print(
            f"✓ Recovered {model} "
            f"±{noise} dBm predictions "
            f"using inference only"
        )

# ============================================================
# LOAD NEURAL MODELS IF A SAVED PREDICTION IS MISSING
# ============================================================

neural_model_paths = {

    "MLP":
        f"{BASE_DIR}/final_mlp.keras",

    "CNN-GAP":
        f"{BASE_DIR}/final_cnn_gap.keras",

    "CNN-FLAT":
        f"{BASE_DIR}/final_cnn_flat.keras"
}

for model in [
    "MLP",
    "CNN-GAP",
    "CNN-FLAT"
]:

    for noise in [2, 5, 8]:

        if noise in stage4_predictions.get(
            model,
            {}
        ):

            continue

        import tensorflow as tf

        model_path = (
            neural_model_paths[
                model
            ]
        )

        assert os.path.exists(
            model_path
        )

        trained_model = (
            tf.keras.models.load_model(
                model_path
            )
        )

        noise_path = (
            f"{NOISE_DIR}/"
            f"X_test_noise_pm{noise}dBm.npy"
        )

        X_noisy = np.load(
            noise_path
        )

        X_scaled = scaler.transform(
            X_noisy
        )

        X_scaled = np.asarray(
            X_scaled,
            dtype=np.float32
        )

        if model == "MLP":

            model_input = (
                tf.convert_to_tensor(
                    X_scaled,
                    dtype=tf.float32
                )
            )

            model_input = tf.ensure_shape(
                model_input,
                (226, 13)
            )

        else:

            model_input = (
                tf.convert_to_tensor(
                    X_scaled[:, :, np.newaxis],
                    dtype=tf.float32
                )
            )

            model_input = tf.ensure_shape(
                model_input,
                (226, 13, 1)
            )

        prediction = (
            trained_model(
                model_input,
                training=False
            ).numpy()
        )

        prediction = np.asarray(
            prediction,
            dtype=np.float64
        )

        if model not in stage4_predictions:

            stage4_predictions[
                model
            ] = {}

        stage4_predictions[
            model
        ][noise] = prediction

# ============================================================
# RECONSTRUCT STAGE 4 OBSERVATION-LEVEL ERRORS
# ============================================================

# True coordinates
true_x = stage1[
    "true_x"
].to_numpy(dtype=np.float64)

true_y = stage1[
    "true_y"
].to_numpy(dtype=np.float64)

y_true = np.column_stack(
    [
        true_x,
        true_y
    ]
)

stage4_errors = {}

for model in MODELS:

    stage4_errors[
        model
    ] = {}

    for noise in [2, 5, 8]:

        prediction = (
            stage4_predictions[
                model
            ][noise]
        )

        errors = np.sqrt(
            np.sum(
                (
                    y_true -
                    prediction
                ) ** 2,
                axis=1
            )
        )

        assert len(errors) == 226
        assert np.all(
            np.isfinite(errors)
        )

        stage4_errors[
            model
        ][noise] = errors

print(
    "\n✓ Observation-level Stage 4 "
    "errors reconstructed for all 7 models"
)

# ============================================================
# STATISTICAL VALIDATION
# ============================================================

print("\n" + "=" * 90)
print("PAIRED STATISTICAL VALIDATION")
print("=" * 90)

# ------------------------------------------------------------
# Reproducible bootstrap
# ------------------------------------------------------------

rng = np.random.default_rng(
    20261008
)

N_BOOTSTRAP = 10000

statistical_rows = []

for model in MODELS:

    baseline = stage1_errors[
        model
    ]

    for noise in [2, 5, 8]:

        noisy = stage4_errors[
            model
        ][noise]

        # ----------------------------------------------------
        # Paired difference
        # Positive = degradation
        # Negative = improvement
        # ----------------------------------------------------

        difference = (
            noisy -
            baseline
        )

        mean_difference = (
            np.mean(difference)
        )

        median_difference = (
            np.median(difference)
        )

        std_difference = (
            np.std(
                difference,
                ddof=1
            )
        )

        # ----------------------------------------------------
        # Wilcoxon signed-rank
        # ----------------------------------------------------

        try:

            wilcoxon_result = (
                wilcoxon(
                    noisy,
                    baseline,
                    alternative="two-sided",
                    zero_method="wilcox",
                    method="auto"
                )
            )

            wilcoxon_stat = (
                float(
                    wilcoxon_result.statistic
                )
            )

            wilcoxon_p = (
                float(
                    wilcoxon_result.pvalue
                )
            )

        except Exception:

            wilcoxon_stat = np.nan
            wilcoxon_p = np.nan

        # ----------------------------------------------------
        # Paired Cohen's dz
        # ----------------------------------------------------

        if std_difference > 0:

            cohens_dz = (
                mean_difference /
                std_difference
            )

        else:

            cohens_dz = 0.0

        # ----------------------------------------------------
        # Bootstrap CI for mean difference
        # ----------------------------------------------------

        n = len(difference)

        bootstrap_means = np.empty(
            N_BOOTSTRAP,
            dtype=np.float64
        )

        for b in range(
            N_BOOTSTRAP
        ):

            sample = rng.choice(
                difference,
                size=n,
                replace=True
            )

            bootstrap_means[b] = (
                np.mean(sample)
            )

        ci_low, ci_high = (
            np.percentile(
                bootstrap_means,
                [2.5, 97.5]
            )
        )

        # ----------------------------------------------------
        # Relative degradation
        # ----------------------------------------------------

        baseline_mean = (
            np.mean(baseline)
        )

        relative_degradation = (
            mean_difference /
            baseline_mean *
            100.0
        )

        # ----------------------------------------------------
        # Stage 4 aggregate value
        # ----------------------------------------------------

        master_row = stage4_master[
            (
                stage4_master["Model"]
                == model
            )
            &
            (
                stage4_master["Noise_dBm"]
                == noise
            )
        ]

        if len(master_row) == 1:

            reported_degradation = float(
                master_row[
                    "Percentage_Degradation"
                ].iloc[0]
            )

        else:

            reported_degradation = np.nan

        statistical_rows.append({

            "Model":
                model,

            "Noise_dBm":
                noise,

            "Baseline_Mean_Error_grid_units":
                baseline_mean,

            "Stage4_Mean_Error_grid_units":
                np.mean(noisy),

            "Mean_Difference_grid_units":
                mean_difference,

            "Median_Difference_grid_units":
                median_difference,

            "95CI_Lower_grid_units":
                ci_low,

            "95CI_Upper_grid_units":
                ci_high,

            "Percentage_Degradation":
                relative_degradation,

            "Reported_Master_Degradation":
                reported_degradation,

            "Wilcoxon_Statistic":
                wilcoxon_stat,

            "Wilcoxon_p":
                wilcoxon_p,

            "Cohens_dz":
                cohens_dz,

            "N":
                n
        })

stats_df = pd.DataFrame(
    statistical_rows
)

# ============================================================
# MULTIPLE COMPARISON CORRECTION
# ============================================================

valid = (
    stats_df["Wilcoxon_p"]
    .notna()
)

reject, p_adjusted, _, _ = (
    multipletests(
        stats_df.loc[
            valid,
            "Wilcoxon_p"
        ],
        alpha=0.05,
        method="fdr_bh"
    )
)

stats_df.loc[
    valid,
    "Wilcoxon_p_FDR"
] = p_adjusted

stats_df.loc[
    valid,
    "Significant_FDR_0.05"
] = reject

# ============================================================
# STATISTICAL INTERPRETATION FLAGS
# ============================================================

stats_df[
    "CI_Contains_Zero"
] = (
    (
        stats_df["95CI_Lower_grid_units"]
        <= 0
    )
    &
    (
        stats_df["95CI_Upper_grid_units"]
        >= 0
    )
)

stats_df[
    "Significant_Raw_p_0.05"
] = (
    stats_df["Wilcoxon_p"] < 0.05
)

# ============================================================
# ROUNDING / SORTING
# ============================================================

stats_df["Model"] = pd.Categorical(
    stats_df["Model"],
    categories=MODELS,
    ordered=True
)

stats_df = (
    stats_df
    .sort_values(
        [
            "Noise_dBm",
            "Model"
        ]
    )
    .reset_index(drop=True)
)

# ============================================================
# SAVE
# ============================================================

stats_path = (
    f"{OUTPUT_DIR}/"
    "STAGE4_STATISTICAL_VALIDATION.csv"
)

stats_df.to_csv(
    stats_path,
    index=False
)

# ============================================================
# DISPLAY MAIN TABLE
# ============================================================

display_columns = [

    "Model",
    "Noise_dBm",
    "Baseline_Mean_Error_grid_units",
    "Stage4_Mean_Error_grid_units",
    "Mean_Difference_grid_units",
    "95CI_Lower_grid_units",
    "95CI_Upper_grid_units",
    "Percentage_Degradation",
    "Wilcoxon_p",
    "Wilcoxon_p_FDR",
    "Cohens_dz",
    "CI_Contains_Zero",
    "Significant_FDR_0.05"
]

print("\n" + "=" * 90)
print("STATISTICAL VALIDATION TABLE")
print("=" * 90)

display(
    stats_df[
        display_columns
    ].style.format({

        "Baseline_Mean_Error_grid_units":
            "{:.6f}",

        "Stage4_Mean_Error_grid_units":
            "{:.6f}",

        "Mean_Difference_grid_units":
            "{:+.6f}",

        "95CI_Lower_grid_units":
            "{:+.6f}",

        "95CI_Upper_grid_units":
            "{:+.6f}",

        "Percentage_Degradation":
            "{:+.2f}%",

        "Wilcoxon_p":
            "{:.6f}",

        "Wilcoxon_p_FDR":
            "{:.6f}",

        "Cohens_dz":
            "{:+.4f}"
    })
)

# ============================================================
# SIGNIFICANT COMPARISONS
# ============================================================

significant = stats_df[
    stats_df[
        "Significant_FDR_0.05"
    ] == True
]

print("\n" + "=" * 90)
print("FDR-SIGNIFICANT COMPARISONS")
print("=" * 90)

if len(significant) == 0:

    print(
        "No comparisons remain statistically "
        "significant after Benjamini–Hochberg "
        "FDR correction at α = 0.05."
    )

else:

    display(
        significant[
            display_columns
        ].style.format({

            "Baseline_Mean_Error_grid_units":
                "{:.6f}",

            "Stage4_Mean_Error_grid_units":
                "{:.6f}",

            "Mean_Difference_grid_units":
                "{:+.6f}",

            "95CI_Lower_grid_units":
                "{:+.6f}",

            "95CI_Upper_grid_units":
                "{:+.6f}",

            "Percentage_Degradation":
                "{:+.2f}%",

            "Wilcoxon_p":
                "{:.6f}",

            "Wilcoxon_p_FDR":
                "{:.6f}",

            "Cohens_dz":
                "{:+.4f}"
        })
    )

# ============================================================
# FINAL QUALITY CHECKS
# ============================================================

print("\n" + "=" * 90)
print("QUALITY CHECKS")
print("=" * 90)

# 21 comparisons
assert len(stats_df) == 21

# 226 paired observations
assert stats_df["N"].eq(226).all()

# No NaN in essential statistical fields
assert (
    stats_df[
        [
            "Mean_Difference_grid_units",
            "95CI_Lower_grid_units",
            "95CI_Upper_grid_units",
            "Wilcoxon_p",
            "Cohens_dz"
        ]
    ]
    .notna()
    .all()
    .all()
)

print(
    "✓ 21 model/noise comparisons"
)

print(
    "✓ 226 paired observations per comparison"
)

print(
    "✓ Wilcoxon signed-rank tests completed"
)

print(
    "✓ 10,000-bootstrap 95% CIs completed"
)

print(
    "✓ Cohen's dz calculated"
)

print(
    "✓ Benjamini–Hochberg FDR correction completed"
)

print(
    f"\n✓ Statistical validation saved to:\n"
    f"{stats_path}"
)

print("\n" + "=" * 90)
print("STATISTICAL VALIDATION COMPLETE")
print("=" * 90)

In [ ]:
# ============================================================
# STAGE 4 — CREATE SINGLE DOWNLOADABLE ZIP
# ============================================================

import os
import zipfile
from google.colab import files

BASE_DIR = "/content"

FOLDERS_TO_INCLUDE = [
    "STAGE4_NOISE_DATA",
    "STAGE4_NOISE_RESULTS"
]

ZIP_PATH = (
    f"{BASE_DIR}/STAGE4_COMPLETE_RESULTS.zip"
)

# ------------------------------------------------------------
# Create ZIP
# ------------------------------------------------------------

print("=" * 80)
print("CREATING STAGE 4 COMPLETE RESULTS ZIP")
print("=" * 80)

file_count = 0
total_size = 0

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    for folder in FOLDERS_TO_INCLUDE:

        folder_path = os.path.join(
            BASE_DIR,
            folder
        )

        if not os.path.exists(folder_path):

            print(
                f"⚠ Folder not found: {folder_path}"
            )

            continue

        for root, dirs, files_list in os.walk(
            folder_path
        ):

            for filename in files_list:

                full_path = os.path.join(
                    root,
                    filename
                )

                relative_path = os.path.relpath(
                    full_path,
                    BASE_DIR
                )

                zipf.write(
                    full_path,
                    arcname=relative_path
                )

                file_count += 1
                total_size += os.path.getsize(
                    full_path
                )

# ------------------------------------------------------------
# Verify ZIP
# ------------------------------------------------------------

zip_size = os.path.getsize(
    ZIP_PATH
)

print("\n" + "=" * 80)
print("ZIP CREATED SUCCESSFULLY")
print("=" * 80)

print(
    f"Files included : {file_count}"
)

print(
    f"Original size  : "
    f"{total_size / (1024**2):.2f} MB"
)

print(
    f"ZIP size       : "
    f"{zip_size / (1024**2):.2f} MB"
)

print(
    f"ZIP path       : {ZIP_PATH}"
)

# ------------------------------------------------------------
# List ZIP contents
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("ZIP CONTENTS")
print("-" * 80)

with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zipf:

    names = zipf.namelist()

    for name in names:

        print(
            "✓",
            name
        )

# ------------------------------------------------------------
# Integrity test
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("ZIP INTEGRITY CHECK")
print("-" * 80)

with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zipf:

    bad_file = zipf.testzip()

if bad_file is None:

    print(
        "✓ ZIP integrity check PASSED"
    )

else:

    print(
        "✗ Corrupted file:",
        bad_file
    )

# ------------------------------------------------------------
# Download
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("READY FOR DOWNLOAD")
print("=" * 80)

files.download(
    ZIP_PATH
)

In [ ]:
# ============================================================
# FINAL MASTER EXPERIMENTAL RESULTS
# STAGES 1–4
#
# READ-ONLY CONSOLIDATION
# NO RETRAINING
# NO NEW INFERENCE
# ============================================================

import os
import glob
import pandas as pd
import numpy as np

print("=" * 90)
print("FINAL MASTER EXPERIMENTAL RESULTS — STAGES 1–4")
print("=" * 90)

BASE_DIR = "/content"
OUTPUT_DIR = "/content/FINAL_EXPERIMENTAL_RESULTS"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

MODELS = [
    "kNN",
    "Decision Tree",
    "Random Forest",
    "SVR",
    "MLP",
    "CNN-GAP",
    "CNN-FLAT"
]

# ============================================================
# 1. STAGE 1
# ============================================================

stage1_path = (
    "/content/test_predictions_all_models.csv"
)

assert os.path.exists(
    stage1_path
), f"Stage 1 file not found: {stage1_path}"

stage1 = pd.read_csv(
    stage1_path
)

stage1_error_columns = {

    "kNN":
        "knn_error",

    "Decision Tree":
        "decision_tree_error",

    "Random Forest":
        "random_forest_error",

    "SVR":
        "svr_error",

    "MLP":
        "mlp_error",

    "CNN-GAP":
        "1d_cnn_gap_error",

    "CNN-FLAT":
        "1d_cnn_flat_error"
}

stage1_results = []

for model in MODELS:

    error_col = (
        stage1_error_columns[
            model
        ]
    )

    error = (
        stage1[error_col]
        .to_numpy(dtype=float)
    )

    stage1_results.append({

        "Model":
            model,

        "Stage1_Mean_Error_grid_units":
            np.mean(error)
    })

stage1_results = pd.DataFrame(
    stage1_results
)

print(
    f"\n✓ Stage 1 loaded: {stage1.shape}"
)

# ============================================================
# 2. STAGE 2
# ============================================================

print("\n" + "-" * 90)
print("SEARCHING FOR STAGE 2 RESULTS")
print("-" * 90)

stage2_candidates = []

for pattern in [
    "*STAGE2*.csv",
    "*stage2*.csv",
    "*Stage2*.csv"
]:

    stage2_candidates.extend(
        glob.glob(
            f"{BASE_DIR}/**/{pattern}",
            recursive=True
        )
    )

# Remove duplicates
stage2_candidates = sorted(
    list(set(stage2_candidates))
)

# Exclude unrelated backup files if any
stage2_candidates = [
    p for p in stage2_candidates
    if "STAGE4" not in p.upper()
]

for p in stage2_candidates:

    print(
        "  Candidate:",
        p
    )

# ------------------------------------------------------------
# Try to identify a result CSV containing model/error values
# ------------------------------------------------------------

stage2_path = None

for p in stage2_candidates:

    try:

        temp = pd.read_csv(p)

        columns_lower = [
            str(c).lower()
            for c in temp.columns
        ]

        text = " ".join(
            columns_lower
        )

        if (
            "model" in columns_lower
            and (
                "error" in text
                or "mean_error" in text
                or "localization" in text
            )
        ):

            stage2_path = p
            break

    except Exception:

        pass

# ------------------------------------------------------------
# If not automatically identified, use known Stage 2 values
# already completed and documented.
# ------------------------------------------------------------

if stage2_path is not None:

    stage2 = pd.read_csv(
        stage2_path
    )

    print(
        f"\n✓ Stage 2 result file identified:\n"
        f"  {stage2_path}"
    )

else:

    print(
        "\n⚠ No Stage 2 result CSV automatically identified."
    )

    print(
        "Using the verified completed Stage 2 results."
    )

    stage2 = pd.DataFrame({

        "Model": MODELS,

        "Stage2_Mean_Error_grid_units": [

            2.0902,   # kNN

            2.0048,   # Decision Tree

            2.0313,   # Random Forest

            1.8134,   # SVR

            1.9948,   # MLP

            3.1471,   # CNN-GAP

            2.0379    # CNN-FLAT
        ]
    })

# ------------------------------------------------------------
# Normalize Stage 2 if file was found
# ------------------------------------------------------------

if stage2_path is not None:

    # Find model column
    model_col = None

    for c in stage2.columns:

        if str(c).lower() == "model":

            model_col = c
            break

    assert model_col is not None

    # Find likely error column
    error_col = None

    priority_terms = [
        "mean_error",
        "localization_error",
        "error_grid_units"
    ]

    for term in priority_terms:

        for c in stage2.columns:

            if term in str(c).lower():

                error_col = c
                break

        if error_col is not None:
            break

    # Fallback: search any error column
    if error_col is None:

        for c in stage2.columns:

            if "error" in str(c).lower():

                error_col = c
                break

    assert error_col is not None

    stage2_results = pd.DataFrame({

        "Model":
            stage2[model_col],

        "Stage2_Mean_Error_grid_units":
            stage2[error_col]
    })

else:

    stage2_results = stage2.copy()

# ============================================================
# 3. STAGE 3
# ============================================================

print("\n" + "-" * 90)
print("SEARCHING FOR STAGE 3 RESULTS")
print("-" * 90)

stage3_candidates = []

for pattern in [
    "*STAGE3*.csv",
    "*stage3*.csv",
    "*Stage3*.csv"
]:

    stage3_candidates.extend(
        glob.glob(
            f"{BASE_DIR}/**/{pattern}",
            recursive=True
        )
    )

stage3_candidates = sorted(
    list(set(stage3_candidates))
)

stage3_candidates = [
    p for p in stage3_candidates
    if "STAGE4" not in p.upper()
]

for p in stage3_candidates:

    print(
        "  Candidate:",
        p
    )

stage3_path = None

for p in stage3_candidates:

    try:

        temp = pd.read_csv(p)

        columns_lower = [
            str(c).lower()
            for c in temp.columns
        ]

        text = " ".join(
            columns_lower
        )

        if (
            "model" in columns_lower
            and (
                "error" in text
                or "mean_error" in text
                or "localization" in text
            )
        ):

            stage3_path = p
            break

    except Exception:

        pass

# ------------------------------------------------------------
# If not automatically identified, use verified Stage 3 values
# ------------------------------------------------------------

if stage3_path is not None:

    stage3 = pd.read_csv(
        stage3_path
    )

    print(
        f"\n✓ Stage 3 result file identified:\n"
        f"  {stage3_path}"
    )

else:

    print(
        "\n⚠ No Stage 3 result CSV automatically identified."
    )

    print(
        "Using the verified completed Stage 3 results."
    )

    stage3 = pd.DataFrame({

        "Model": MODELS,

        "Stage3_Mean_Error_grid_units": [

            1.4706,   # kNN

            1.6900,   # Decision Tree

            1.4707,   # Random Forest

            1.5228,   # SVR

            1.8268,   # MLP

            3.3400,   # CNN-GAP

            1.7815    # CNN-FLAT
        ]
    })

# ------------------------------------------------------------
# Normalize Stage 3
# ------------------------------------------------------------

if stage3_path is not None:

    model_col = None

    for c in stage3.columns:

        if str(c).lower() == "model":

            model_col = c
            break

    assert model_col is not None

    error_col = None

    for term in [
        "mean_error",
        "localization_error",
        "error_grid_units"
    ]:

        for c in stage3.columns:

            if term in str(c).lower():

                error_col = c
                break

        if error_col is not None:
            break

    if error_col is None:

        for c in stage3.columns:

            if "error" in str(c).lower():

                error_col = c
                break

    assert error_col is not None

    stage3_results = pd.DataFrame({

        "Model":
            stage3[model_col],

        "Stage3_Mean_Error_grid_units":
            stage3[error_col]
    })

else:

    stage3_results = stage3.copy()

# ============================================================
# 4. STAGE 4
# ============================================================

stage4_path = (
    "/content/STAGE4_NOISE_RESULTS/"
    "STAGE4_MASTER_NOISE_ROBUSTNESS.csv"
)

assert os.path.exists(
    stage4_path
), f"Stage 4 master file not found: {stage4_path}"

stage4 = pd.read_csv(
    stage4_path
)

print(
    f"\n✓ Stage 4 loaded: {stage4.shape}"
)

# ------------------------------------------------------------
# Stage 4 mean error across all three noise levels
# ------------------------------------------------------------

stage4_mean = (
    stage4
    .groupby(
        "Model",
        as_index=False
    )
    .agg(

        Stage4_Mean_Error_grid_units=(
            "Noisy_Error_grid_units",
            "mean"
        ),

        Stage4_Mean_Degradation_Percent=(
            "Percentage_Degradation",
            "mean"
        ),

        Stage4_Max_Degradation_Percent=(
            "Percentage_Degradation",
            "max"
        )
    )
)

# ============================================================
# 5. NORMALIZE MODEL NAMES
# ============================================================

def normalize_model_name(x):

    x = str(x).strip()

    mapping = {

        "knn": "kNN",

        "KNN": "kNN",

        "k-NN": "kNN",

        "decision tree": "Decision Tree",

        "DecisionTree":
            "Decision Tree",

        "random forest": "Random Forest",

        "RandomForest":
            "Random Forest",

        "svr": "SVR",

        "mlp": "MLP",

        "cnn-gap": "CNN-GAP",

        "CNN GAP": "CNN-GAP",

        "cnn_gap": "CNN-GAP",

        "cnn-flat": "CNN-FLAT",

        "CNN FLAT": "CNN-FLAT",

        "cnn_flat": "CNN-FLAT"
    }

    return mapping.get(
        x,
        x
    )


for df in [
    stage1_results,
    stage2_results,
    stage3_results,
    stage4_mean
]:

    df["Model"] = (
        df["Model"]
        .map(normalize_model_name)
    )

# ============================================================
# 6. MERGE ALL STAGES
# ============================================================

master = (
    stage1_results
    .merge(
        stage2_results,
        on="Model",
        how="outer"
    )
    .merge(
        stage3_results,
        on="Model",
        how="outer"
    )
    .merge(
        stage4_mean,
        on="Model",
        how="outer"
    )
)

# ============================================================
# 7. FORCE MODEL ORDER
# ============================================================

master["Model"] = pd.Categorical(
    master["Model"],
    categories=MODELS,
    ordered=True
)

master = (
    master
    .sort_values("Model")
    .reset_index(drop=True)
)

# ============================================================
# 8. VERIFY COMPLETENESS
# ============================================================

assert len(master) == 7

required_columns = [

    "Model",

    "Stage1_Mean_Error_grid_units",

    "Stage2_Mean_Error_grid_units",

    "Stage3_Mean_Error_grid_units",

    "Stage4_Mean_Error_grid_units",

    "Stage4_Mean_Degradation_Percent",

    "Stage4_Max_Degradation_Percent"
]

for column in required_columns:

    assert column in master.columns, (
        f"Missing column: {column}"
    )

print(
    "\n✓ All seven models present"
)

print(
    "✓ Stages 1–4 successfully consolidated"
)

# ============================================================
# 9. CALCULATE CROSS-STAGE SUMMARY METRICS
# ============================================================

stage_columns = [

    "Stage1_Mean_Error_grid_units",
    "Stage2_Mean_Error_grid_units",
    "Stage3_Mean_Error_grid_units",
    "Stage4_Mean_Error_grid_units"
]

# Worst error observed across stages
master[
    "Worst_Error_grid_units"
] = master[
    stage_columns
].max(axis=1)

# Best error observed across stages
master[
    "Best_Error_grid_units"
] = master[
    stage_columns
].min(axis=1)

# Stage 1 → Stage 2 change
master[
    "Stage2_vs_Stage1_Percent"
] = (
    (
        master["Stage2_Mean_Error_grid_units"]
        -
        master["Stage1_Mean_Error_grid_units"]
    )
    /
    master["Stage1_Mean_Error_grid_units"]
    *
    100
)

# Stage 3 → Stage 1 change
master[
    "Stage3_vs_Stage1_Percent"
] = (
    (
        master["Stage3_Mean_Error_grid_units"]
        -
        master["Stage1_Mean_Error_grid_units"]
    )
    /
    master["Stage1_Mean_Error_grid_units"]
    *
    100
)

# Stage 4 mean degradation is already calculated
# from the three fixed noise levels.

# ============================================================
# 10. SAVE MASTER CSV
# ============================================================

master_path = (
    f"{OUTPUT_DIR}/"
    "FINAL_MASTER_EXPERIMENTAL_RESULTS.csv"
)

master.to_csv(
    master_path,
    index=False
)

# ============================================================
# 11. DISPLAY FINAL MASTER TABLE
# ============================================================

print("\n" + "=" * 90)
print("FINAL MASTER EXPERIMENTAL RESULTS")
print("=" * 90)

display(
    master.style.format({

        "Stage1_Mean_Error_grid_units":
            "{:.4f}",

        "Stage2_Mean_Error_grid_units":
            "{:.4f}",

        "Stage3_Mean_Error_grid_units":
            "{:.4f}",

        "Stage4_Mean_Error_grid_units":
            "{:.4f}",

        "Stage4_Mean_Degradation_Percent":
            "{:+.2f}%",

        "Stage4_Max_Degradation_Percent":
            "{:+.2f}%",

        "Worst_Error_grid_units":
            "{:.4f}",

        "Best_Error_grid_units":
            "{:.4f}",

        "Stage2_vs_Stage1_Percent":
            "{:+.2f}%",

        "Stage3_vs_Stage1_Percent":
            "{:+.2f}%"
    })
)

# ============================================================
# 12. CREATE MANUSCRIPT-FRIENDLY TABLE
# ============================================================

paper_table = master[
    [
        "Model",
        "Stage1_Mean_Error_grid_units",
        "Stage2_Mean_Error_grid_units",
        "Stage3_Mean_Error_grid_units",
        "Stage4_Mean_Error_grid_units",
        "Stage4_Mean_Degradation_Percent"
    ]
].copy()

paper_table_path = (
    f"{OUTPUT_DIR}/"
    "FINAL_MASTER_PAPER_TABLE.csv"
)

paper_table.to_csv(
    paper_table_path,
    index=False
)

print("\n" + "=" * 90)
print("MANUSCRIPT-FRIENDLY TABLE")
print("=" * 90)

display(
    paper_table.style.format({

        "Stage1_Mean_Error_grid_units":
            "{:.4f}",

        "Stage2_Mean_Error_grid_units":
            "{:.4f}",

        "Stage3_Mean_Error_grid_units":
            "{:.4f}",

        "Stage4_Mean_Error_grid_units":
            "{:.4f}",

        "Stage4_Mean_Degradation_Percent":
            "{:+.2f}%"
    })
)

# ============================================================
# 13. BEST MODEL BY EACH STAGE
# ============================================================

print("\n" + "=" * 90)
print("BEST MODEL BY EVALUATION STAGE")
print("=" * 90)

for stage_col in stage_columns:

    row = master.loc[
        master[stage_col].idxmin()
    ]

    print(
        f"{stage_col}: "
        f"{row['Model']} "
        f"({row[stage_col]:.4f} grid-coordinate units)"
    )

# ============================================================
# 14. OVERALL ROBUSTNESS
# ============================================================

robustness_row = master.loc[
    master[
        "Stage4_Mean_Degradation_Percent"
    ].idxmin()
]

print(
    "\nBest Stage 4 robustness:"
)

print(
    f"  {robustness_row['Model']} "
    f"({robustness_row['Stage4_Mean_Degradation_Percent']:+.2f}% "
    f"mean degradation)"
)

# ============================================================
# 15. FINAL FILES
# ============================================================

print("\n" + "=" * 90)
print("FINAL MASTER CONSOLIDATION COMPLETE")
print("=" * 90)

print(
    "\n✓ Saved:"
)

print(
    master_path
)

print(
    paper_table_path
)

print(
    "\nNo models were retrained."
)

print(
    "No new predictions were generated."
)

print(
    "Only existing experimental results were consolidated."
)

print("=" * 90)

In [ ]:
# ============================================================
# FINAL AUDIT — VERIFY SAVED STAGE 2 & STAGE 3 RESULTS
# No training / no prediction / no experiment modification
# ============================================================

import pandas as pd
import os

# ------------------------------------------------------------
# Expected verified values from the completed experiments
# ------------------------------------------------------------
expected_stage2 = {
    "kNN": 2.0902,
    "Decision Tree": 2.0048,
    "Random Forest": 2.0313,
    "SVR": 1.8134,
    "MLP": 1.9948,
    "CNN-GAP": 3.1471,
    "CNN-FLAT": 2.0379
}

expected_stage3 = {
    "kNN": 1.4706,
    "Decision Tree": 1.6900,
    "Random Forest": 1.4707,
    "SVR": 1.5228,
    "MLP": 1.8268,
    "CNN-GAP": 3.3400,
    "CNN-FLAT": 1.7815
}

# ------------------------------------------------------------
# Candidate saved files
# ------------------------------------------------------------
stage2_files = [
    "/content/STAGE2_COMPLETE_BACKUP/stage2_conventional_results.csv",
    "/content/STAGE2_COMPLETE_BACKUP/stage2_error_summary.csv",
    "/content/STAGE2_COMPLETE_BACKUP/stage2_master_generalization_comparison.csv",
    "/content/BEACON_RESEARCH_MASTER_BACKUP/MASTER_RESULTS/stage2_conventional_results.csv",
    "/content/BEACON_RESEARCH_MASTER_BACKUP/MASTER_RESULTS/stage2_master_generalization_comparison.csv",
    "/content/stage2_conventional_results.csv",
    "/content/stage2_error_summary.csv",
    "/content/stage2_master_generalization_comparison.csv"
]

stage3_files = [
    "/content/STAGE3_COMPLETE_BACKUP/stage3_conventional_results.csv",
    "/content/STAGE3_COMPLETE_BACKUP/stage3_neural_results.csv",
    "/content/BEACON_RESEARCH_MASTER_BACKUP/MASTER_RESULTS/stage3_conventional_results.csv",
    "/content/BEACON_RESEARCH_MASTER_BACKUP/MASTER_RESULTS/stage3_neural_results.csv",
    "/content/stage3_conventional_results.csv",
    "/content/stage3_neural_results.csv"
]

print("=" * 80)
print("FINAL EXPERIMENTAL AUDIT — STAGE 2 & STAGE 3")
print("=" * 80)

# ------------------------------------------------------------
# 1. Show available files
# ------------------------------------------------------------
print("\nAVAILABLE STAGE 2 FILES")
print("-" * 80)

for f in stage2_files:
    if os.path.exists(f):
        print("✓", f)

print("\nAVAILABLE STAGE 3 FILES")
print("-" * 80)

for f in stage3_files:
    if os.path.exists(f):
        print("✓", f)

# ------------------------------------------------------------
# 2. Inspect CSV contents
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("STAGE 2 CSV CONTENTS")
print("=" * 80)

for f in stage2_files:
    if os.path.exists(f):
        print("\nFILE:", f)
        try:
            df = pd.read_csv(f)
            print("Shape:", df.shape)
            print("Columns:", list(df.columns))
            print(df.head(10).to_string(index=False))
        except Exception as e:
            print("ERROR:", e)

print("\n" + "=" * 80)
print("STAGE 3 CSV CONTENTS")
print("=" * 80)

for f in stage3_files:
    if os.path.exists(f):
        print("\nFILE:", f)
        try:
            df = pd.read_csv(f)
            print("Shape:", df.shape)
            print("Columns:", list(df.columns))
            print(df.head(10).to_string(index=False))
        except Exception as e:
            print("ERROR:", e)

print("\n" + "=" * 80)
print("AUDIT COMPLETE")
print("=" * 80)

print("""
IMPORTANT:
No model was trained.
No prediction was generated.
No dataset was modified.
No experimental result was changed.

The purpose of this cell is ONLY to identify and inspect the
already-saved Stage 2 and Stage 3 result files.
""")

In [ ]:
# ============================================================
# FINAL SCI MASTER TABLE — FREEZE VERIFIED EXPERIMENTAL RESULTS
# ============================================================
# NO training
# NO prediction
# NO dataset modification
# ONLY consolidation of already verified results
# ============================================================

import pandas as pd
import os

OUT_DIR = "/content/FINAL_EXPERIMENTAL_RESULTS"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# AUTHORITATIVE STAGE 1 VALUES
# Verified previously against test_predictions_all_models.csv
# ------------------------------------------------------------
stage1 = {
    "kNN": 1.627924,
    "Decision Tree": 1.675515,
    "Random Forest": 1.633219,
    "SVR": 1.607923,
    "MLP": 2.093180,
    "CNN-GAP": 3.062271,
    "CNN-FLAT": 2.025434
}

# ------------------------------------------------------------
# AUTHORITATIVE STAGE 2 VALUES
# Taken from the audited saved Stage 2 result files
# ------------------------------------------------------------
stage2 = {
    "kNN": 2.090205,
    "Decision Tree": 2.004785,
    "Random Forest": 2.031278,
    "SVR": 1.813362,
    "MLP": 1.994793,
    "CNN-GAP": 3.147060,
    "CNN-FLAT": 2.037865
}

# ------------------------------------------------------------
# AUTHORITATIVE STAGE 3 VALUES
# Taken from the audited saved Stage 3 result files
# ------------------------------------------------------------
stage3 = {
    "kNN": 1.470590,
    "Decision Tree": 1.690008,
    "Random Forest": 1.470747,
    "SVR": 1.522835,
    "MLP": 1.826808,
    "CNN-GAP": 3.340041,
    "CNN-FLAT": 1.781505
}

# ------------------------------------------------------------
# AUTHORITATIVE STAGE 4 MEAN ERRORS
# Mean across ±2, ±5 and ±8 dBm
# ------------------------------------------------------------
stage4_mean = {
    "kNN": 1.655644,
    "Decision Tree": 1.872106,
    "Random Forest": 1.757122,
    "SVR": 1.640684,
    "MLP": 2.093377,
    "CNN-GAP": 3.129047,
    "CNN-FLAT": 2.029087
}

# ------------------------------------------------------------
# STAGE 4 ROBUSTNESS
# Mean and maximum degradation
# ------------------------------------------------------------
stage4_mean_deg = {
    "kNN": 1.70,
    "Decision Tree": 11.73,
    "Random Forest": 7.59,
    "SVR": 2.04,
    "MLP": 0.01,
    "CNN-GAP": 2.18,
    "CNN-FLAT": 0.18
}

stage4_max_deg = {
    "kNN": 3.64,
    "Decision Tree": 24.19,
    "Random Forest": 14.69,
    "SVR": 3.96,
    "MLP": 0.31,
    "CNN-GAP": 3.30,
    "CNN-FLAT": 0.45
}

models = [
    "kNN",
    "Decision Tree",
    "Random Forest",
    "SVR",
    "MLP",
    "CNN-GAP",
    "CNN-FLAT"
]

# ------------------------------------------------------------
# BUILD MASTER TABLE
# ------------------------------------------------------------
rows = []

for model in models:

    s1 = stage1[model]
    s2 = stage2[model]
    s3 = stage3[model]

    rows.append({
        "Model": model,
        "Stage1_Seen_MLE_grid_units": s1,
        "Stage2_Unseen_MLE_grid_units": s2,
        "Stage3_Future_MLE_grid_units": s3,
        "Stage4_Mean_MLE_grid_units": stage4_mean[model],

        "Stage2_vs_Stage1_%": ((s2 - s1) / s1) * 100,
        "Stage3_vs_Stage1_%": ((s3 - s1) / s1) * 100,

        "Stage4_Mean_Degradation_%": stage4_mean_deg[model],
        "Stage4_Max_Degradation_%": stage4_max_deg[model],

        "Best_Stage_MLE_grid_units": min(s1, s2, s3),
        "Worst_Stage_MLE_grid_units": max(s1, s2, s3)
    })

master = pd.DataFrame(rows)

# ------------------------------------------------------------
# RANK EACH STAGE
# ------------------------------------------------------------
master["Stage1_Rank"] = master["Stage1_Seen_MLE_grid_units"].rank(
    method="min", ascending=True
).astype(int)

master["Stage2_Rank"] = master["Stage2_Unseen_MLE_grid_units"].rank(
    method="min", ascending=True
).astype(int)

master["Stage3_Rank"] = master["Stage3_Future_MLE_grid_units"].rank(
    method="min", ascending=True
).astype(int)

master["Stage4_Robustness_Rank"] = master[
    "Stage4_Mean_Degradation_%"
].rank(method="min", ascending=True).astype(int)

# ------------------------------------------------------------
# ROUND FOR PAPER TABLE
# ------------------------------------------------------------
paper_table = master.copy()

for col in [
    "Stage1_Seen_MLE_grid_units",
    "Stage2_Unseen_MLE_grid_units",
    "Stage3_Future_MLE_grid_units",
    "Stage4_Mean_MLE_grid_units",
    "Best_Stage_MLE_grid_units",
    "Worst_Stage_MLE_grid_units"
]:
    paper_table[col] = paper_table[col].round(4)

for col in [
    "Stage2_vs_Stage1_%",
    "Stage3_vs_Stage1_%",
    "Stage4_Mean_Degradation_%",
    "Stage4_Max_Degradation_%"
]:
    paper_table[col] = paper_table[col].round(2)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
master_path = os.path.join(
    OUT_DIR,
    "FINAL_MASTER_EXPERIMENTAL_RESULTS_VERIFIED.csv"
)

paper_path = os.path.join(
    OUT_DIR,
    "FINAL_MASTER_PAPER_TABLE_VERIFIED.csv"
)

master.to_csv(master_path, index=False)
paper_table.to_csv(paper_path, index=False)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------
print("=" * 100)
print("FINAL VERIFIED SCI EXPERIMENTAL MASTER TABLE")
print("=" * 100)

print(
    paper_table[
        [
            "Model",
            "Stage1_Seen_MLE_grid_units",
            "Stage2_Unseen_MLE_grid_units",
            "Stage3_Future_MLE_grid_units",
            "Stage4_Mean_MLE_grid_units",
            "Stage2_vs_Stage1_%",
            "Stage3_vs_Stage1_%",
            "Stage4_Mean_Degradation_%",
            "Stage4_Max_Degradation_%"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 100)
print("BEST MODEL BY EVALUATION")
print("=" * 100)

print(
    "Stage 1 — Seen-location:      ",
    master.loc[master["Stage1_Seen_MLE_grid_units"].idxmin(), "Model"]
)

print(
    "Stage 2 — Unseen-location:    ",
    master.loc[master["Stage2_Unseen_MLE_grid_units"].idxmin(), "Model"]
)

print(
    "Stage 3 — Future-time:        ",
    master.loc[master["Stage3_Future_MLE_grid_units"].idxmin(), "Model"]
)

print(
    "Stage 4 — Noise robustness:   ",
    master.loc[master["Stage4_Mean_Degradation_%"].idxmin(), "Model"]
)

print("\n" + "=" * 100)
print("FILES SAVED")
print("=" * 100)

print(master_path)
print(paper_path)

print("\n✓ No training performed")
print("✓ No prediction performed")
print("✓ No dataset modified")
print("✓ Previous experiments remain unchanged")
print("✓ Stage 1 stale values in old Stage 2 master CSV corrected")
print("=" * 100)

In [ ]:

# ============================================================
# SCI-QUALITY BLE-RSSI NOISE ROBUSTNESS FIGURES
# Outputs: PNG 600 dpi, TIFF 600 dpi, PDF (vector)
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

OUTPUT_DIR = "/content/ble_sci_figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. GLOBAL PUBLICATION STYLE
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
    "axes.linewidth": 0.8,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.08,
})

def save_publication_figure(fig, filename):
    """Save a figure in high-resolution raster and vector formats."""
    base = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(base + ".png", dpi=600, facecolor="white")
    fig.savefig(
        base + ".tiff", dpi=600, facecolor="white",
        pil_kwargs={"compression": "tiff_lzw"}
    )
    fig.savefig(base + ".pdf", facecolor="white")
    plt.show()
    plt.close(fig)

# ------------------------------------------------------------
# 2. INPUT DATA
# IMPORTANT:
# The first two arrays are approximate transcriptions of the
# screenshots. Replace them with exact CSV-derived values.
# ------------------------------------------------------------
noise_levels = ["±2 dBm", "±5 dBm", "±8 dBm"]

error_data = {
    "kNN":            [1.62, 1.66, 1.68],
    "Decision Tree":  [1.63, 1.90, 2.08],
    "Random Forest":  [1.61, 1.78, 1.87],
    "SVR":            [1.62, 1.63, 1.67],
    "MLP":            [2.09, 2.10, 2.09],
    "CNN-GAP":        [3.08, 3.16, 3.14],
    "CNN-FLAT":       [2.03, 2.04, 2.03],
}

degradation_data = {
    "kNN":            [-0.7, 2.1, 3.6],
    "Decision Tree":  [-2.6, 13.6, 24.2],
    "Random Forest":  [-1.1, 9.2, 14.7],
    "SVR":            [0.7, 1.4, 3.9],
    "MLP":            [0.3, 0.1, -0.3],
    "CNN-GAP":        [0.7, 3.3, 2.5],
    "CNN-FLAT":       [0.1, 0.4, -0.1],
}

# Reported mean percentage degradation across the three
# noise conditions from the current experiment summary.
mean_degradation = {
    "MLP": 0.009443,
    "CNN-FLAT": 0.180367,
    "kNN": 1.702815,
    "SVR": 2.037462,
    "CNN-GAP": 2.180613,
    "Random Forest": 7.586453,
    "Decision Tree": 11.733143,
}

markers = ["o", "s", "^", "D", "v", "P", "X"]

# ------------------------------------------------------------
# FIGURE 1: LOCALIZATION ERROR UNDER RSSI NOISE
# Numeric values are displayed at every point.
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.5, 4.8))

for (model, values), marker in zip(error_data.items(), markers):
    ax.plot(
        noise_levels, values,
        marker=marker, markersize=4.5,
        linewidth=1.6, label=model
    )

    for x, y in enumerate(values):
        ax.annotate(
            f"{y:.2f}",
            (x, y),
            xytext=(0, 6),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=7.5,
            clip_on=False
        )

ax.set_title("Localization Error under RSSI Noise")
ax.set_xlabel("RSSI perturbation level")
ax.set_ylabel("Mean localization error (grid-coordinate units)")
ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.6)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, ncol=2, loc="upper left")
ax.margins(y=0.18)

save_publication_figure(fig, "Figure_3A_Localization_Error")

# ------------------------------------------------------------
# FIGURE 2: PERCENTAGE ERROR DEGRADATION
# Negative values are retained: they indicate improvement.
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.5, 4.8))

for (model, values), marker in zip(degradation_data.items(), markers):
    ax.plot(
        noise_levels, values,
        marker=marker, markersize=4.5,
        linewidth=1.6, label=model
    )

    for x, y in enumerate(values):
        offset = 6 if y >= 0 else -9
        ax.annotate(
            f"{y:.1f}%",
            (x, y),
            xytext=(0, offset),
            textcoords="offset points",
            ha="center",
            va="bottom" if y >= 0 else "top",
            fontsize=7.5,
            clip_on=False
        )

ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Localization Error Degradation under RSSI Noise")
ax.set_xlabel("RSSI perturbation level")
ax.set_ylabel("Error degradation relative to baseline (%)")
ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.6)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, ncol=2, loc="upper left")
ax.margins(y=0.15)

save_publication_figure(fig, "Figure_3B_Error_Degradation")

# ------------------------------------------------------------
# FIGURE 3: MEAN DEGRADATION BAR CHART
# Every bar has its exact reported numerical value on top.
# ------------------------------------------------------------
ordered = sorted(
    mean_degradation.items(),
    key=lambda item: item[1]
)

model_names = [item[0] for item in ordered]
values = [item[1] for item in ordered]

fig, ax = plt.subplots(figsize=(7.5, 4.8))

bars = ax.bar(
    model_names, values,
    width=0.68,
    edgecolor="black",
    linewidth=0.5
)

for bar, value in zip(bars, values):
    ax.annotate(
        f"{value:.2f}%",
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8.5
    )

ax.set_title("Mean RSSI Noise-Induced Error Degradation")
ax.set_xlabel("Localization model")
ax.set_ylabel("Mean degradation across noise levels (%)")
ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.6)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="x", labelrotation=28)
ax.margins(y=0.14)

save_publication_figure(fig, "Figure_3C_Mean_Noise_Robustness")

print(f"\nAll figures saved in: {OUTPUT_DIR}")

# Download all outputs as a ZIP file.
import shutil
zip_path = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print("ZIP archive:", zip_path)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = "/content/ble_sci_figures_final"
os.makedirs(OUT_DIR, exist_ok=True)

# Consistent order across figures
models = [
    "kNN", "Decision Tree", "Random Forest", "SVR",
    "MLP", "CNN-GAP", "CNN-FLAT"
]

# Mean localization error in grid-coordinate units
stage1 = [1.6279, 1.6755, 1.6332, 1.6079, 2.0932, 3.0623, 2.0254]
stage2 = [2.0902, 2.0048, 2.0313, 1.8134, 1.9948, 3.1471, 2.0379]
stage3 = [1.4706, 1.6900, 1.4707, 1.5228, 1.8268, 3.3400, 1.7815]

# Stage 1 error-distribution statistics
median_error = [1.3508, 1.4417, 1.3385, 1.2984, 1.8820, 2.4203, 1.8361]
p90_error = [3.1583, 3.6056, 3.3280, 3.2021, 3.5272, 5.8606, 3.5240]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "savefig.dpi": 600,
})

def save_figure(fig, filename):
    fig.savefig(
        os.path.join(OUT_DIR, filename + ".png"),
        dpi=600, bbox_inches="tight"
    )
    fig.savefig(
        os.path.join(OUT_DIR, filename + ".pdf"),
        bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)

In [ ]:
x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))

b1 = ax.bar(
    x - width, stage1, width,
    label="Stage 1: Observation-level"
)
b2 = ax.bar(
    x, stage2, width,
    label="Stage 2: Held-out locations"
)
b3 = ax.bar(
    x + width, stage3, width,
    label="Stage 3: Chronological"
)

ax.set_title("Mean Localization Error Across Evaluation Protocols")
ax.set_xlabel("Model")
ax.set_ylabel("Mean localization error (grid-coordinate units)")
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=25, ha="right")

# Keep values visible above every bar
for bars in (b1, b2, b3):
    ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=7)

ax.set_ylim(0, max(stage1 + stage2 + stage3) * 1.18)
ax.grid(axis="y", linestyle="--", alpha=0.45)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper left")

fig.tight_layout()
save_figure(fig, "Figure_2_Mean_Error_Stages_1_to_3")

In [ ]:
x = np.arange(len(models))
width = 0.36

fig, ax = plt.subplots(figsize=(10.5, 6))

b1 = ax.bar(
    x - width/2, median_error, width,
    label="Median error"
)
b2 = ax.bar(
    x + width/2, p90_error, width,
    label="90th-percentile error"
)

ax.set_title("Stage 1 Localization Error Distribution")
ax.set_xlabel("Model")
ax.set_ylabel("Localization error (grid-coordinate units)")
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=25, ha="right")

ax.bar_label(b1, fmt="%.2f", padding=2, fontsize=8)
ax.bar_label(b2, fmt="%.2f", padding=2, fontsize=8)

ax.set_ylim(0, max(p90_error) * 1.18)
ax.grid(axis="y", linestyle="--", alpha=0.45)
ax.set_axisbelow(True)
ax.legend(frameon=False)

fig.tight_layout()
save_figure(fig, "Figure_3_Stage1_Error_Distribution")

In [ ]:
from google.colab import files
import os, shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Upload results/STAGE4_MASTER_NOISE_ROBUSTNESS.csv from this repository.
uploaded = files.upload()
csv_candidates = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_candidates:
    raise ValueError("Upload STAGE4_MASTER_NOISE_ROBUSTNESS.csv from the repository results folder.")
csv_path = csv_candidates[0]
df = pd.read_csv(csv_path)
print("Loaded:", csv_path)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head())


In [ ]:
def find_col(data, candidates):
    lookup = {str(c).strip().lower().replace(" ", "_"): c for c in data.columns}
    for candidate in candidates:
        key = candidate.strip().lower().replace(" ", "_")
        if key in lookup: return lookup[key]
    raise KeyError(f"Missing one of {candidates}. Available: {data.columns.tolist()}")

model_col = find_col(df, ["Model", "model_name", "estimator"])
noise_col = find_col(df, ["Noise_Level", "noise_level", "noise", "perturbation_level"])
error_col = find_col(df, ["Noisy_Error_grid_units", "Noisy_Error", "noisy_error", "noisy_mle", "Noisy_MLE"])
degradation_col = find_col(df, ["Percentage_Degradation", "percentage_degradation", "degradation_pct", "degradation_percent"])

plot_df = df[[model_col, noise_col, error_col, degradation_col]].copy()
plot_df.columns = ["Model", "Noise_Level", "Noisy_Error", "Percentage_Degradation"]
plot_df["Model"] = plot_df["Model"].astype(str).str.strip()
plot_df["Noise_Level"] = plot_df["Noise_Level"].astype(str).str.strip()
for c in ["Noisy_Error", "Percentage_Degradation"]:
    plot_df[c] = pd.to_numeric(plot_df[c], errors="coerce")
plot_df = plot_df.dropna(subset=["Model", "Noise_Level", "Noisy_Error", "Percentage_Degradation"])

def normalize_noise(v):
    s = str(v).lower().replace(" ", "")
    if "2" in s: return "±2 dBm"
    if "5" in s: return "±5 dBm"
    if "8" in s: return "±8 dBm"
    return str(v)

plot_df["Noise_Level"] = plot_df["Noise_Level"].map(normalize_noise)
noise_order = ["±2 dBm", "±5 dBm", "±8 dBm"]
model_order = ["kNN", "Decision Tree", "Random Forest", "SVR", "MLP", "CNN-GAP", "CNN-FLAT"]
print("Rows:", len(plot_df), "| Models:", sorted(plot_df["Model"].unique()), "| Noise levels:", sorted(plot_df["Noise_Level"].unique()))
display(plot_df.sort_values(["Model", "Noise_Level"]))

In [ ]:
OUT_DIR = "/content/ble_sci_figures_4_to_6"
os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 12, "axes.labelsize": 10,
    "legend.fontsize": 8, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "pdf.fonttype": 42, "ps.fonttype": 42
})
ERROR_UNIT_LABEL = "grid-coordinate units"  # Change only if physical calibration is documented.

def export_figure(fig, stem):
    fig.savefig(os.path.join(OUT_DIR, stem + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(OUT_DIR, stem + ".pdf"), bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved PNG and PDF:", stem)

In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 5.4))
for model in model_order:
    sub = plot_df[plot_df["Model"].str.lower() == model.lower()].copy()
    if sub.empty: continue
    sub["Noise_Level"] = pd.Categorical(sub["Noise_Level"], categories=noise_order, ordered=True)
    sub = sub.sort_values("Noise_Level")
    xx = [noise_order.index(str(n)) for n in sub["Noise_Level"]]
    ax.plot(xx, sub["Noisy_Error"], marker="o", linewidth=1.8, markersize=4.5, label=model)
ax.set_xticks(range(len(noise_order))); ax.set_xticklabels(noise_order)
ax.set_title("Localization Error Under RSSI Perturbations")
ax.set_xlabel("RSSI perturbation level")
ax.set_ylabel(f"Mean localization error ({ERROR_UNIT_LABEL})")
ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.4); ax.set_axisbelow(True)
ax.legend(frameon=False, ncol=2, loc="best")
fig.tight_layout()
export_figure(fig, "Figure_4_Localization_Error_Under_RSSI_Perturbations")

In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 5.4))
for model in model_order:
    sub = plot_df[plot_df["Model"].str.lower() == model.lower()].copy()
    if sub.empty: continue
    sub["Noise_Level"] = pd.Categorical(sub["Noise_Level"], categories=noise_order, ordered=True)
    sub = sub.sort_values("Noise_Level")
    xx = [noise_order.index(str(n)) for n in sub["Noise_Level"]]
    ax.plot(xx, sub["Percentage_Degradation"], marker="o", linewidth=1.8, markersize=4.5, label=model)
ax.axhline(0, color="black", linewidth=0.9)
ax.set_xticks(range(len(noise_order))); ax.set_xticklabels(noise_order)
ax.set_title("Localization Error Degradation Under RSSI Perturbations")
ax.set_xlabel("RSSI perturbation level")
ax.set_ylabel("Error degradation relative to baseline (%)")
ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.4); ax.set_axisbelow(True)
ax.legend(frameon=False, ncol=2, loc="best")
fig.tight_layout()
export_figure(fig, "Figure_5_Percentage_Degradation_Under_RSSI_Perturbations")

In [ ]:
summary = (plot_df.groupby("Model", as_index=False)
           .agg(Mean_Degradation=("Percentage_Degradation", "mean"),
                Mean_Noisy_Error=("Noisy_Error", "mean"),
                Number_of_Levels=("Noise_Level", "nunique")))
summary = summary.sort_values("Mean_Degradation", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9.2, 5.4))
bars = ax.bar(summary["Model"], summary["Mean_Degradation"])
ax.bar_label(bars, labels=[f"{v:.2f}%" for v in summary["Mean_Degradation"]], padding=3, fontsize=8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Mean RSSI Noise-Induced Error Degradation")
ax.set_xlabel("Model")
ax.set_ylabel("Mean degradation across perturbation levels (%)")
ax.tick_params(axis="x", labelrotation=25)
for tick in ax.get_xticklabels(): tick.set_ha("right")
ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.4); ax.set_axisbelow(True)
fig.tight_layout()
export_figure(fig, "Figure_6_Mean_RSSI_Noise_Induced_Degradation")
display(summary.round({"Mean_Degradation": 4, "Mean_Noisy_Error": 4}))
summary.to_csv(os.path.join(OUT_DIR, "Stage4_Figure6_Summary_Table.csv"), index=False)

In [ ]:
zip_output = shutil.make_archive("/content/Stage4_Figures_4_to_6_600dpi", "zip", OUT_DIR)
files.download(zip_output)